# SupplementRx AI — Schema Design and Mapping

**Implementation update — 2026-11-14:** the bounded [Preprocessing_Pilot.ipynb](Preprocessing_Pilot.ipynb) now implements contract 0.1.0 on the selected records, with validated outputs and an identical replay. Earlier design-only statements describe their original stage; no full-dataset preprocessing, semantic merging or clinical approval has occurred. See the pilot notebook and its build manifest for current results.

**Status: Steps 1–2 documented; Steps 3–4 contain the conceptual schema and source-field mapping drafts. Step 5A documents identity/provenance drafts and scoped reviews; Steps 5B–5C define draft normalization and deduplication rules. Step 6 documents an integrated representative-case design review; Step 7 provides contract 0.1.0 and a bounded pilot handoff; implementation validation and broader schema finalization remain pending. No shared identities or clinical claims have been approved.**

This notebook will turn the audit findings and provisional POC field selections into a shared schema and mapping rules. Refer to `Data_Audit_and_Preprocessing.ipynb` for the source audits, examples and selected fields. The broader schema is not finalized; Step 7 defines contract 0.1.0 for a bounded implementation pilot.

## Broad steps planned

1. **Review concrete POC questions and selected fields.** Define representative questions, identify the data needed to answer them, and document coverage gaps and evidence limits.
2. **Review cross-source identities and terminology mappings.** Examine PubChem, RxNorm and MeSH together, linked to originating DSLD ingredient/form occurrences. Distinguish equivalent identities, specific forms, broader concepts and unresolved candidates while preserving separate source identifiers.
3. **Draft the shared conceptual schema.** Define entities and relationships for product labels, ingredients, terminology concepts, NatMed clinical-reference assertions, Reddit user reports and source evidence. Keep evidence roles and source scope explicit.
4. **Define source-to-schema field mappings.** Specify how selected fields map to the shared model, including field types, missing-value states, source context, timestamps and provenance.
5. **Specify identity, normalization and deduplication rules.** Document safe representation changes, candidate-review requirements, assertion-equivalence criteria, conflict handling and update/version policies. Rule design comes here; applying transformations comes later.
6. **Review the draft against representative examples.** Check that it can represent equivalent units, distinct ingredient forms, different serving bases, source conflicts, clinical context and inherited Reddit topics without losing information. Document open decisions.
7. **Finalize the design and prepare the implementation handoff.** Consolidate the agreed schema, mapping rules and acceptance checks. Plan a small normalization pilot, followed by validation, deduplication and eventual graph construction in a separate implementation notebook.

## Working boundaries

- Keep raw datasets unchanged and retain original values alongside any future derived representations.
- Treat PubChem, RxNorm and MeSH as a coordinated terminology layer with distinct identities and reviewable mappings; a shared query or name does not establish equivalence.
- Keep DSLD label information, NatMed clinical-reference assertions and Reddit experiences distinguishable. Source traceability and structural validity do not establish clinical correctness.
- Preserve unresolved identities, missing context and conflicting assertions rather than guessing corrections or forcing merges.
- Consider stable identifiers, explicit relationship types and provenance for later data additions, graph storage and possible GNN/RGCN work; do not implement those stages here.
- Do not update `read.md` with the design until the shared schema is finalized.

**Current scope:** review what the selected fields can support. The combined candidate review is documented in Step 2, with unresolved identities retained. No mappings have been approved and no normalization, deduplication or graph construction has been performed. The Step 3 conceptual draft and Step 4 source-field mappings follow below. Steps 5A–5C now contain identity/provenance, normalization and deduplication specifications with illustrative evidence reviews. No transformations or approvals have been applied; the shared schema is not finalized.


## Step 1 — Review concrete POC questions and selected fields

**Deliverable:** a proposed question set, source/field coverage matrix and acceptance criteria. These are evaluation scenarios, not generated medical answers or approved user requirements. The local audits and selected-field cells in `Data_Audit_and_Preprocessing.ipynb` are the reference. No new data collection or transformations are performed.

The questions cover the five user needs: purpose/effectiveness, safety/interactions, amount/usage context, product composition/comparison and evidence transparency. Evidence transparency applies to every question, not only Q10.

Coverage labels below mean:

- **Direct source retrieval:** the selected fields support an attributed answer about a saved source. It does not establish present-day product availability or clinical truth.
- **Rule-dependent:** needed fields exist, but answering comparably requires later normalization/context rules.
- **Review-dependent:** candidate evidence is present, but identity, clinical scope or source interpretation remains unresolved. The POC may show qualified source excerpts, not silently treat pending annotations as approved facts.
- **Unresolved:** the answer should expose the limitation instead of inventing a value or identity.

These labels describe the feasibility of a future pipeline, not a functioning retrieval system.


### Question-to-field coverage matrix

| ID / user need | Concrete POC question | Selected fields and sources | Current coverage / boundary |
|---|---|---|---|
| **Q1 — Amount / product composition** | “How much magnesium does the saved Jarrow Magnesium Optimizer label list per serving?” | DSLD `id`, `brandName`, `fullName`; ingredient ID/name/forms; amount/unit/operator and serving basis; `servingSizes`; source/market context | **Direct source retrieval.** Labels 7841 and 212304 list 100 mg against different tablet counts. Present the selected label/version explicitly; do not return an unqualified shared dose. |
| **Q2 — Product comparison** | “Do BPI Creatine Alkaline labels 25731 and 43261 list the same creatine amount per two capsules?” | DSLD quantity/unit/operator/basis and ingredient forms; label provenance; future normalization-rule metadata | **Rule-dependent.** 1500 mg and 1.5 Gram(s) are equivalent mass representations at the same stated basis. This does not establish that the entire labels or formulations are duplicates. |
| **Q3 — Composition / label precautions** | “Which other ingredients and precautions appear on each of those BPI labels?” | DSLD `otheringredients.text`, ingredient names/IDs/forms; selected `statements.type`/`notes`; source label ID | **Direct source retrieval.** The labels differ in other ingredients. Keep label-specific lists and warnings. Missing text does not prove absence of an ingredient, allergy risk or precaution. |
| **Q4 — Purpose / effectiveness** | “What uses and effectiveness ratings does the saved NatMed Melatonin monograph describe?” | NatMed `ingredient`, `claim_type=effectiveness`, `indication`, `effectiveness_rating`, `population`, `claim_text`, `source`, `field_support`, `review`; scoped references | **Review-dependent.** 104 effectiveness annotations have indications, but all 104 lack structured population values. Retrieve the original qualification text; do not generalize across populations or promote pending claims into recommendations. |
| **Q5 — Safety context** | “What safety statements and population qualifications does the saved Ashwagandha monograph contain?” | NatMed safety annotations: ingredient/formulation, population, safety rating, full excerpt and source spans; original warning/overview sections when needed; review flags | **Review-dependent.** Seven safety annotations exist; six have structured populations. An upstream extraction gap affects overview-embedded warnings, so claims alone cannot support a complete safety summary. |
| **Q6 — Interactions** | “Which interaction partners and qualifications does the Magnesium monograph report?” | NatMed `claim_type=interaction`, partner/rating/severity/occurrence, evidence grade, population, `claim_text`; source section heading, field supports and review state | **Review-dependent.** 24 interaction annotations exist. Use section context to distinguish partner types; absence from the extracted set does not mean no interaction. No personalized compatibility decision is supported. |
| **Q7 — Dose context** | “What dosing contexts and regimens are described in the saved Creatine monograph?” | NatMed full `dose`, ingredient/formulation, population, indication, original excerpt, source support and review status | **Review-dependent.** Two dose annotations exist with populations but no structured indications. Preserve regimen, bounds, frequency and source text. Do not describe all values as study-specific doses or recommendations without source support. |
| **Q8 — Ingredient identity / terminology** | “Does the DSLD name Creatine HCl have an equivalent entry in PubChem, RxNorm and MeSH?” | Original DSLD occurrence; PubChem CID/properties and name-query candidates; RxNorm RxCUI/name/TTY and search method; MeSH term–concept–record links; provenance and candidate review status | **Unresolved identity.** PubChem returned CID 134732; RxNorm normalized search returned creatine and creatine monohydrate; MeSH returned no candidate. Show the disagreement; do not join all results as one ingredient. Actual mapping review is Step 2. |
| **Q9 — User questions / experiences** | “What are people in the saved Reddit sample asking or reporting about melatonin?” | Reddit title/text, source URL, record/native ID, subreddit, dates, `supplements.matched_terms`, `mention_scope`, thread/parent IDs, `evidence_role`; passage bindings | **Direct retrieval with relevance review.** There are 759 topic-associated records: 135 `own_text`, 624 `thread_context`. Scope is record-level: even `own_text` does not guarantee melatonin is the particular directly matched topic. Verify topic relevance in the actual text; qualify anecdotes and missing context. |
| **Q10 — Provenance / conflicting values** | “Why do the BetterYou Iron Spray records show different serving volumes, and which original fields support those values?” | DSLD label IDs 228638/257287, original quantity/serving fields and notes, source paths/hashes, conflict flags; original evidence | **Conflict explainable; correction unresolved.** Label 257287 has numeric 52 mL versus 0.52 ml in its own note. Show the source discrepancy and flag the affected calculation. Do not claim a verified cause or silently choose the corrected value. |

Every future answer needs source-specific evidence links, applicable label/version or monograph context, and any material unresolved status. A concise answer may hide internal identifiers by default, but they must remain traceable.


### Coverage findings and selected-field review

**1. Product-label questions have the strongest direct coverage.** DSLD supports composition, stated quantities, directions and precautions for saved labels. Preserve other ingredients: Q2 can have equivalent active-ingredient quantities while Q3 differs. Product identity, package/formulation identity and assertion equivalence must remain separate decisions.

**2. NatMed supports contextual source retrieval, not automatic clinical approval.** Its selected fields cover purpose, safety, interactions and dose context. Unknown structured population/formulation/indication fields and extraction gaps make the full excerpt and source section essential. All annotations remain pending despite passing structural/source-span checks. A future answer should attribute what the reference states and expose relevant limits.

**3. The terminology sources must be reviewed together.** Q8 requires a side-by-side candidate view anchored to the same DSLD name/form occurrence. PubChem, RxNorm and MeSH have different coverage and concept scope; missing matches must not discard the source ingredient. Shared IDs, names, formulas or synonyms cannot replace mapping review.

**4. Reddit requires conversation context and a distinct evidence role.** Its selected fields support finding relevant user questions and experiences. They do not yet support structured efficacy, dose or interaction conclusions. Inherited topics, repeated text, passage overlap and context gaps must not inflate apparent independent evidence or imply direct use.

**5. Citations and source context cannot be optional internal fields.** Every source must retain identity, source URL/path, snapshot/hash and extraction context. NatMed reference linkage needs claim-specific support; all monograph references cannot be attached to every claim. Reddit URLs attribute a user report, while terminology API URLs attribute a vocabulary result—not clinical evidence.

**Field-selection outcome:** retain the current provisional selections. No additional clinical fields are invented, and no source fields are removed during this step. In particular, keep:

- ingredient forms, nested/blend context, amount operator and serving basis;
- other ingredients and original directions/precautions;
- full clinical excerpts, source section, population/indication and dose qualifiers even when unknown;
- topic mention scope and thread/parent references;
- mapping method, complete candidate sets, provenance, source-specific dates and review/conflict flags.

**Gaps that need later work, not guessed values:** accepted cross-source ingredient mappings; source-specific product/version identity; incomplete dose/population context; claim-specific citation support; source conflicts; and Reddit topic relevance. Detailed schema requirements and entity relationships are intentionally deferred to Steps 2–4.

No source supports a blanket answer such as “this product is currently available,” “this supplement is safe for you,” or “this dose is appropriate for your medications” merely because an ingredient name matches. These are outside the initial question set.


### Acceptance criteria for the later POC implementation

| Scenario | Expected behavior | Failure to prevent |
|---|---|---|
| Label amount lookup (Q1) | State the amount and exact serving basis with label/version context | Returning 100 mg without specifying one versus two tablets |
| Equivalent units (Q2) | Compare compatible mass values while preserving originals and subject context | Declaring complete products identical from a single matching quantity |
| Other ingredients / precautions (Q3) | Attribute differences to the correct label | Losing composition differences during deduplication or treating missing text as no risk |
| Clinical-reference retrieval (Q4–Q7) | Retain source text, qualifications, review status and scope | Inferring missing population/indication, converting a bound into an exact recommendation, or treating pending annotations as approved |
| Terminology disagreement (Q8) | Display source-specific candidates and unresolved identity | Treating creatine, monohydrate and hydrochloride as interchangeable |
| Reddit retrieval (Q9) | Verify the queried topic in context and label statements as user reports | Presenting inherited thread topics as direct experiences or clinical evidence |
| Source conflict (Q10) | Expose the conflicting fields, retain evidence and flag derived calculations | Silently correcting 52 mL or averaging conflicting values |
| Evidence across all questions | Trace each factual statement to its source/context | Counting duplicate labels or overlapping passages as independent efficacy evidence |

These are design acceptance criteria, **not executed end-to-end tests**. The counts used in this review were checked against the current local source files; no generated medical answers or clinical assessments were produced.

**Step 1 outcome:** a concrete, proposed question set and source/field coverage review are documented. Step 2 follows below, using Q8 and the label examples to guide scope. No normalization, merging or graph construction has been performed. `read.md` remains unchanged.


## Step 2 — Combined terminology and cross-source identity review

**Scope:** compare existing PubChem, RxNorm and MeSH candidates for all 739 DSLD query groups. Verify their source occurrence inventories align before comparing candidates. Perform closer scope review of 19 magnesium/creatine/botanical/probiotic/mixture queries. This step does not approve mappings, normalize identities or doses, merge records, or finalize schema entities.

### Complete comparison results

- All three sources have identical query groups, source-name variants and occurrence inventories. All **3,391 occurrences** were rechecked against the **400 saved DSLD labels**.
- **147 queries** have at least one candidate in all three sources; **236** have partial source coverage; **356** have no candidate in any source. These categories partition the 739 queries.
- **135 queries** have exactly one candidate in each source. This is candidate-count agreement only—not proof of common meaning or chemical identity.
- **43 queries** have multiple candidates in at least one source. **31** use RxNorm normalized search. Categories overlap.
- **178 query groups** contain more than one DSLD ingredient ID. Grouped names are not canonical ingredient identities; retain IDs, roles and paths.

### Review artifacts and field meaning

`query_comparison.csv` provides one row per query group with candidates, source methods, coverage and routing flags. `dsld_occurrences.csv` provides all original label/ingredient/form occurrences, joined by `query_key`, including source file hashes. `review.json` retains detailed candidate properties, MeSH resolution links and shared candidates across queries. `manifest.json` records source fingerprints and output checksums.

The query key is an existing collection grouping key, **not** an accepted ingredient ID. PubChem CIDs, RxNorm RxCUIs and MeSH record/concept/term URIs retain separate namespaces. Candidate count, exact search, shared name and shared identifiers across different queries do not automatically imply equivalence. Every query retains `identity_decision=unresolved_no_equivalence_asserted`.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

terminology_root = Path.cwd()
if not (terminology_root / 'data/raw').is_dir():
    terminology_root = terminology_root.parent
terminology_run = subprocess.run(
    [sys.executable, str(terminology_root / 'scripts/review_terminology_mappings.py')],
    cwd=terminology_root, check=True, capture_output=True, text=True,
)
terminology_review_dir = Path(terminology_run.stdout.splitlines()[0])
terminology_review = json.loads((terminology_review_dir / 'review.json').read_text())
terminology_by_query = {r['query_key']:r for r in terminology_review['queries']}
print('Outputs:', terminology_review_dir.relative_to(terminology_root))
print(json.dumps(terminology_review['summary'], indent=2))
print('Complete table: query_comparison.csv')
print('Source occurrences: dsld_occurrences.csv')


Outputs: data/processed/terminology/terminology-review-v1-34294732268f2352
{
  "query_groups": 739,
  "source_name_variants": 753,
  "source_labels": 400,
  "verified_occurrences": 3391,
  "coverage_counts": {
    "partial_source_coverage": 236,
    "none_returned": 356,
    "all_three_have_candidates": 147
  },
  "one_candidate_in_every_source": 135,
  "queries_ambiguous_in_any_source": 43,
  "flag_counts": {
    "mesh_no_candidate": 469,
    "pubchem_no_candidate": 511,
    "rxnorm_no_candidate": 426,
    "pubchem_multiple_candidates": 7,
    "multiple_dsld_ingredient_ids_in_name_group": 178,
    "rxnorm_normalized_search": 31,
    "rxnorm_multiple_candidates": 31,
    "mesh_multiple_candidates": 6
  },
  "queries_with_multiple_dsld_ids": 178,
  "focus_queries": 19,
  "approved_mappings": 0,
  "merged_entities": 0
}
Complete table: query_comparison.csv
Source occurrences: dsld_occurrences.csv


In [ ]:
focus_queries = [
    'magnesium', 'magnesium citrate', 'magnesium taurinate',
    'creatine', 'creatine monohydrate', 'creatine hcl', 'creatine hydrochloride', 'creatine citrate',
    'ashwagandha', 'ashwagandha root extract', 'withania somnifera', 'turmeric', 'curcumin',
    'probiotics', 'probiotic blend', 'lactobacillus acidophilus', 'lactobacillus acidophilus (ncfm)',
    'fish oil', 'omega-3',
]
for key in focus_queries:
    row = terminology_by_query[key]
    print('\nQUERY:', row['query'])
    print('DSLD IDs:', row['dsld_ingredient_ids'], '| occurrence roles:', row['occurrence_kinds'])
    for source, candidates in row['candidates'].items():
        compact = [{k:c[k] for k in ['id','name','term_type','record_types'] if k in c} for c in candidates]
        print(source, '| method:', row['methods'][source], '| status:', row['source_status'][source])
        print(json.dumps(compact, ensure_ascii=False, indent=2))
    print('Review flags:', row['review_flags'])
print('\nDETAILED MESH CONCEPT/PATH EXAMPLE: TURMERIC')
print(json.dumps(terminology_by_query['turmeric']['candidates']['mesh'], ensure_ascii=False, indent=2))



QUERY: Magnesium
DSLD IDs: [4339, 6520, 6886, 7131, 8705, 10882, 17040, 21678, 22441, 26859, 30233, 31042, 34045, 40137, 56013, 68297, 69771, 71212, 81199, 85508, 93141, 100415, 102631, 106537, 113704, 135100, 139969, 143312, 155755, 157181, 162635, 198179, 202834, 205076, 225338, 229864, 270667, 276734, 286056] | occurrence roles: ['ingredient']
pubchem | method: saved_name_lookup | status: ok
[
  {
    "id": "pubchem:CID:5462224",
    "name": "Magnesium"
  }
]
rxnorm | method: exact | status: matched
[
  {
    "id": "rxnorm:6574",
    "name": "magnesium",
    "term_type": "IN"
  }
]
mesh | method: exact_entry_term | status: matched
[
  {
    "id": "http://id.nlm.nih.gov/mesh/D008274",
    "name": [
      "Magnesium"
    ],
    "record_types": [
      "http://id.nlm.nih.gov/mesh/vocab#TopicalDescriptor"
    ]
  }
]
Review flags: ['multiple_dsld_ingredient_ids_in_name_group']

QUERY: Magnesium Citrate
DSLD IDs: [4045, 15023, 17081, 55109, 63832] | occurrence roles: ['form']
pubchem | 

### Focused interpretation — candidates retained, no accepted equivalences

| Query / group | Observed source candidates | Scope review / next rule needed |
|---|---|---|
| Magnesium | PubChem Magnesium; RxNorm magnesium; MeSH Magnesium | Similar labels support a focused identity review; they do not settle elemental/ionic or product-quantity interpretation. Keep source representations. |
| Magnesium Citrate | PubChem Trimagnesium dicitrate; RxNorm magnesium citrate; MeSH magnesium citrate | Inspect specific chemical form/stoichiometry versus vocabulary scope before attaching one canonical chemical identity. Label magnesium amount cannot automatically become salt mass. |
| Magnesium Taurinate | Four PubChem candidates; no RxNorm or MeSH candidate | Retain the candidate set; no arbitrary first-CID selection. |
| Creatine | All three return a creatine candidate | Useful pilot for an explicit identity review, not an automatically accepted mapping. |
| Creatine Monohydrate | PubChem and RxNorm return monohydrate candidates; MeSH has none | Preserve hydrate specificity and partial coverage. Do not infer that lack of a MeSH result means no source ingredient. |
| Creatine HCl / Creatine Hydrochloride | Both reach the same PubChem candidate; RxNorm normalized results are creatine and creatine monohydrate; MeSH has none | A possible alias relationship between the two source names needs review. The RxNorm results do not establish hydrochloride equivalence. No transitive merge through the shared candidates. |
| Creatine Citrate | PubChem citrate-salt candidate; RxNorm returns creatine and monohydrate through normalized search; MeSH has none | Keep form-specific source identity unresolved rather than replacing it with a normalized parent candidate. |
| Ashwagandha / Ashwagandha Root Extract | Ashwagandha has a MeSH candidate only; the root-extract query has RxNorm Withania somnifera root extract only | Whole-source naming and preparation-specific naming need separate context. Do not merge plant, root or extract solely through topic association. |
| Withania somnifera | PubChem returns Withanolide D; MeSH returns Withania; RxNorm has no candidate | High-priority scope mismatch for review. Do not assert the queried botanical name is identical to the PubChem named compound or the MeSH candidate's scope. No mapping approved. |
| Turmeric / Curcumin | Turmeric: RxNorm turmeric extract and MeSH Curcuma, no PubChem candidate. Curcumin: candidates in all three sources | Keep botanical/extract/constituent scopes distinct. Neither a related term nor a matching topic establishes whole-ingredient equivalence. |
| Probiotics / Probiotic Blend | Probiotics has a MeSH candidate; Probiotic Blend has no candidates | A broad category and an unspecified product blend need not have one chemical identity. Preserve the blend/components from DSLD. |
| Lactobacillus acidophilus / Lactobacillus acidophilus (NCFM) | Species-name query has RxNorm and MeSH candidates; strain-qualified query has none | Do not erase strain qualifiers to force a match. Species-level reference context cannot automatically be applied to every strain or blend. |
| Fish Oil / Omega-3 | Fish Oil has RxNorm fish oils and MeSH Fish Oils; Omega-3 has PubChem Omega-3 Fatty Acids and RxNorm omega-3 fatty acids | Preserve mixture/category scope. A CID response is not proof that the source ingredient is one pure compound; Fish Oil and the broader Omega-3 topic are not automatically equivalent. |

This is a source-candidate scope review based on the saved responses, not an external chemistry or medical verification. The full 739-row table has mechanical review flags; the 19 focus queries received the closer interpretation above. The remaining rows have not been individually adjudicated.

### How these findings constrain the next schema draft

- Keep **source ingredient occurrences**, any future accepted **ingredient identities**, **chemical compounds**, and **terminology concepts** distinguishable.
- Model candidate mappings explicitly with source method, original name/form, evidence, scope and review state. A query can have zero, one or multiple candidates independently in each source.
- Preserve namespaces and roles: a MeSH term may resolve via a concept to a descriptor/supplementary record; these are not interchangeable IDs.
- Allow preparation/form/strain/mixture context without requiring a chemical CID for every ingredient.
- Keep potential exact identity, broader/narrower scope, related form and unresolved candidate as different relationship decisions. These relationship assignments are still pending; no typed equivalence edges were created here.
- Source product labels, NatMed monograph identities and Reddit topic annotations must not be joined using topic names alone. In particular, NatMed Fish Oil and Lactobacillus acidophilus keep their exact source scope; inherited Reddit topics are not direct ingredient use.
- No-match queries remain eligible source ingredients and evidence records. Additional lookup or curated mapping can be considered later without losing the original outcome.

**Step 2 outcome:** complete comparison artifacts plus a focused scope review, with all identities unresolved pending explicit approval criteria. Source fingerprints and occurrence links are preserved. Next: **Step 3 — draft the shared conceptual schema**, using these distinctions and the Step 1 question requirements. No normalization, deduplication, schema finalization, graph construction or GitHub push is performed by this step. `read.md` is unchanged.


## Step 3 — Shared conceptual schema draft

> **Version-history refinement:** Step 5A sections 27–30 specify raw-byte integrity, proposed lossless JSON fingerprints, separate capture events and targeted re-review (`version-history-draft-v1`). Earlier snapshot conventions are refined there; existing manifests are not migrated.

**Status: proposed conceptual design, not finalized or implemented.** This step defines the meanings of entities and relationships needed by our POC questions. It does not create graph nodes, resolve ingredient identities, approve clinical claims, normalize quantities or merge records. Detailed source-field mappings belong to Step 4; executable identity/normalization/deduplication rules belong to Step 5.

### Design objective

Support concise answers while retaining enough context to explain the source, scope and uncertainty of each assertion. The four data roles stay distinguishable:

| Data role | Sources | What this layer supplies |
|---|---|---|
| Product-label information | DSLD | Composition, forms, serving definitions, directions, precautions and label history |
| Chemical/terminology representation | PubChem, RxNorm, MeSH | Source-defined entities and explicit mapping candidates; vocabulary relations |
| Clinical-reference information | NatMed | Contextual, source-attributed annotations, passages and references; currently all annotations pending |
| User reports and questions | Reddit | Original posts/comments, conversation context and topic associations; not clinical evidence |

**Central distinction:** the source occurrence of a name, an accepted shared identity, and an assertion about that identity are different things. A query name can exist without an accepted cross-source identity. A useful source assertion can exist even when its ingredient mapping or quantity is unresolved.

### Proposed modeling conventions

- A **SourceRecord** identifies an external record independently of capture time. A **SourceSnapshot** represents a particular saved version and has a content hash, capture metadata and source-specific dates/status. Label, monograph and Reddit snapshots are specializations of this common provenance pattern, not extra copies of the text.
- **SourceEvidence** identifies an exact span, JSON pointer or RDF statement in a snapshot. Clinical field-support anchors can differ from the main excerpt span, including an earlier effectiveness heading.
- **Assertions** carry their own scope and evidence. A source explicitly reporting something is different from our system accepting the underlying association or identity.
- Simple context values (amount/basis, population wording, regimen and ratings) are initially structured attributes, not automatically independent nodes. Conditions, populations or interaction partners can link to reviewed identities later, without discarding their source wording.
- Source-role, identity-review and clinical-review states are separate dimensions. Passing integrity checks does not change either approval state. All current candidate mappings remain unresolved and all NatMed claim reviews remain pending.


### A. Entity definitions and identity boundaries

> **Version-history refinement:** Step 5A sections 27–30 specify raw-byte integrity, proposed lossless JSON fingerprints, separate capture events and targeted re-review (`version-history-draft-v1`). Earlier snapshot conventions are refined there; existing manifests are not migrated.

> **Reference handling refinement:** Step 5A sections 24–26 specify `SourceReference`, unresolved-target states and graph-export constraints (`reference-resolution-draft-v1`). A known external key is not a collected target record.

> **Product boundary refinement:** Step 5A sections 21–23 define `product-boundaries-draft-v1`: product family is commercial-line identity; variant includes disclosed formulation/presentation and package. Earlier definitions are provisional and refined there.

> **Later design refinement:** Step 5A sections 15–17 define `mapping-semantics-draft-v2`, separating concept denotation, category membership and exact material matching. Use that refinement for new proposals; this table preserves the earlier draft.

| Proposed entity / specialization | Meaning and essential context | Identity boundary |
|---|---|---|
| **SourceRecord** | One source-defined label, monograph, Reddit post/comment or terminology record | Namespaced source ID; not inferred from its display name. Source-specific URLs are provenance, not universal identity keys. |
| **SourceSnapshot** | A captured representation of a SourceRecord; original payload/text, hashes, collection time, source dates/version metadata | Separate mutable source identity from captured version. Hash equality alone does not merge different source records. Exact version-key policy is deferred. |
| **SourceEvidence** | A JSON field occurrence, source-text span, section/word window or RDF statement tied to a snapshot | Snapshot + locator + locator kind. Preserve original values and any supporting field role. Distinct evidence can support one equivalent assertion later. |
| **Passage** | A retrieval chunk from a snapshot, with exact source window and inherited context | Source-bound chunk identity. Overlap does not create independent evidence; a claim need not be fully contained in one passage. |
| **IngredientOccurrence** | An ingredient/form name as captured in a DSLD row, nested component, other-ingredient list or NatMed monograph subject | Source-bound occurrence and role, with original name/ID/context. Not a canonical ingredient. Forms and components retain links to their parent occurrence. |
| **IngredientIdentity** | A locally accepted shared identity with explicit scope (e.g. substance, preparation, species, strain or mixture) | Allocated only through later identity rules. Names, shared CIDs or single results do not create it automatically. Unknown form/strain remains unknown. |
| **ChemicalCompound** | A PubChem CID-defined compound representation, linked to its saved source snapshot and properties | Namespaced CID with versioned source representation. Different CIDs remain distinct absent a separate reviewed decision. |
| **TerminologyConcept** | An RxNorm concept or MeSH record/concept/term, retaining source system and type | RxCUI or full MeSH URI; keep IN/PIN/MIN and descriptor/SCR/concept/term roles. These entities are not automatically IngredientIdentity nodes. |
| **MappingAssertion** | A source-backed proposed relationship from an occurrence or accepted identity to a source concept/compound | Origin, candidate target, query/method, source evidence, proposed relation/scope and identity-review state. An unresolved result is not an accepted same-identity edge. |
| **LookupAttempt** | One saved name lookup with its original query, variants, method, parameters and outcome | Preserves successful, ambiguous and no-match attempts. Zero candidates produces no invented target entity. Several occurrences can share one name query. |
| **ProductIdentity** *(optional until reviewed)* | A commercial product grouping, with brand/name and identity-review context | Not one node per repeated brand/name automatically. UPC/SKU and source label relationships are evidence, not sufficient identity rules. |
| **ProductVariant** *(optional until reviewed)* | A distinguished composition/package variant under an established product identity | Relevant composition/form/package differences remain explicit. Label snapshots can be used without assigning either product entity. |
| **ServingDefinition** | A source-label serving specification, including size range, unit, daily-serving bounds and notes | Initially local to a label snapshot. It does not overwrite the explicit basis recorded on an ingredient quantity. |
| **CompositionAssertion** | A label-backed ingredient presence/amount statement in a particular label or accepted variant context | Ingredient occurrence/identity, role, nested context, forms, operator, amount/unit, explicit serving basis and qualifications. Multiple quantities on one row remain distinct statements. Unknown amount permits a presence assertion. |
| **LabelStatement** | Verbatim label directions, precaution or storage text | Source context and statement type retained; not an independently validated clinical conclusion. |
| **ClinicalAssertion** | A NatMed annotation with claim type, exact subject scope, source text, population/indication, ratings, dose/interaction context and review state | Stable upstream claim ID plus its source version/evidence. Different populations/regimens may yield separate assertions from one excerpt. |
| **ReferenceDocument / StudyReference** | Captured bibliography material and identified cited publications, respectively | Preserve local citation labels separately from external PMID/DOI identities. Citation support is not assumed from a bibliography entry alone. |
| **RedditPost / RedditComment** | Specializations of SourceRecord with their snapshots; thread, parent and subreddit context | Native source IDs; identical text in distinct conversations is not sufficient to merge records. Missing parents remain unresolved references. |
| **TopicAssociation** | A source record's association with a search/topic label, with matched terms and direct/inherited scope | Preserve `own_text` versus `thread_context`; a topic association is not evidence of use, benefit, or adverse effect. |
| **Topic** | A project/search category such as Omega-3 or Probiotics | A broad retrieval category, not necessarily one ingredient, species, chemical or formulation. |

**Complexity boundary:** these are conceptual roles, not a mandate to implement 20 physical tables or node labels. SourceRecord specializations share one provenance pattern. Amount, population, review metadata and brand name can remain attributes initially. Product/variant identities are optional until accepted; raw source information remains usable without them.


### B. Relationship meanings and proposed cardinality

> **Reference handling refinement:** Step 5A sections 24–26 specify `SourceReference`, unresolved-target states and graph-export constraints (`reference-resolution-draft-v1`). A known external key is not a collected target record.

> **Product boundary refinement:** Step 5A sections 21–23 define `product-boundaries-draft-v1`: product family is commercial-line identity; variant includes disclosed formulation/presentation and package. Earlier definitions are provisional and refined there.

> **Later design refinement:** Step 5A sections 15–17 define `mapping-semantics-draft-v2`, separating concept denotation, category membership and exact material matching. Use that refinement for new proposals; this table preserves the earlier draft.

Cardinality below is a proposed validation constraint, not an assertion that the current data is already complete. A source target not captured locally can remain an unresolved external identifier; do not fabricate its content.

| Relationship | Meaning / cardinality | Constraint |
|---|---|---|
| `SourceRecord HAS_SNAPSHOT SourceSnapshot` | One source record has one or more saved snapshots; each snapshot belongs to one source record | Never replace the original evidence merely because a later capture exists. |
| `SourceEvidence LOCATED_IN SourceSnapshot` | Each evidence item points to one snapshot | Locator kind and bounds/path must be valid for that snapshot. A set of evidence items can support an assertion. |
| `Passage EXCERPT_OF SourceSnapshot` | Each passage belongs to one captured source window | Preserve text, section/window, hash and citation scope. |
| `IngredientOccurrence LOCATED_IN SourceSnapshot` | Each occurrence belongs to one source snapshot | Keep path/span and source name/ID; matching text is not identity. |
| `IngredientOccurrence HAS_COMPONENT / HAS_FORM IngredientOccurrence` | Zero or more source-scoped component/form occurrences | Parent quantity is not assigned to every child or form. Blend totals and component amounts must not be added blindly. |
| `LookupAttempt QUERIED_FOR IngredientOccurrence` | One lookup may serve several name-grouped occurrences | Preserve the complete occurrence list and original variants. |
| `MappingAssertion ORIGINATES_FROM IngredientOccurrence or IngredientIdentity` | Exactly one scoped origin per mapping assertion | Query-level candidates may be displayed together, but acceptance must retain the source occurrence/identity scope. |
| `MappingAssertion CANDIDATE_TARGET ChemicalCompound or TerminologyConcept` | Exactly one candidate target per mapping assertion; a lookup can yield zero to many assertions | Preserve the full candidate set. A later accepted link may identify equivalent, broader/narrower or related scope; those assignments are not made here. |
| `MappingAssertion FROM_LOOKUP LookupAttempt` | Zero or one saved lookup; other evidence-led/manual mappings must declare their method | Evidence and review state required; never infer transitive equivalence from shared targets. |
| `IngredientOccurrence RESOLVES_TO IngredientIdentity` *(accepted only)* | Zero or one accepted identity at the occurrence's declared scope | Multiple alternatives remain mapping candidates, not multiple accepted identities. Composite occurrences need separate components, not forced singular chemistry. |
| `ProductIdentity HAS_VARIANT ProductVariant` | One product may have several accepted variants | Product grouping remains optional. Source label relationships remain source assertions rather than automatic identity links. |
| `label snapshot DESCRIBES ProductVariant` *(accepted only)* | Initially unresolved; one applicable variant if later established | Package/composition differences and version context constrain this assignment. |
| `CompositionAssertion HAS_SUBJECT label snapshot or ProductVariant` | Exactly one subject context | Source-label scope is sufficient before product identity is resolved. Cross-label consolidation requires an established shared context. |
| `CompositionAssertion ABOUT_INGREDIENT IngredientOccurrence or IngredientIdentity` | Exactly one scoped ingredient referent; multiple assertions cover multiple components | Preserve original occurrence evidence even after an accepted identity link exists. |
| `CompositionAssertion USES_SERVING ServingDefinition` | Zero or one resolved source serving definition per assertion | Retain explicit amount basis separately, including missing/conflicting basis. |
| `LabelStatement ON_LABEL label snapshot` | One source label context per statement occurrence | Identical text may be stored once internally, but applicability/source occurrences remain distinct. |
| `ClinicalAssertion ABOUT_SUBJECT IngredientOccurrence or IngredientIdentity` | One monograph subject with exact preparation/species/strain scope | Broad project topics are not substitutes for the monograph subject. |
| `ClinicalAssertion HAS_REFERENCE StudyReference` *(only when source-linked)* | Zero or more explicit citation links, each with local reference label and source evidence | Neither link presence nor publication type proves the study supports the assertion. Unresolved references remain unresolved. |
| `ReferenceDocument BIBLIOGRAPHY_FOR monograph snapshot` | Captured reference page tied to the monograph capture | Preserve its own URL/text/hash/date; do not attach all bibliography entries to every assertion. |
| `RedditComment REPLY_TO RedditPost or RedditComment` | Zero or one resolved immediate parent; original missing/external parent ID retained | Reply context does not establish endorsement. |
| `RedditComment IN_THREAD RedditPost` | At most one resolved thread post | Do not invent missing thread content. |
| `TopicAssociation ON_SOURCE source snapshot` and `ABOUT_TOPIC Topic` | One source context and one topic per association | Inherited relevance is not direct mention, use or clinical evidence. |
| `assertion SUPPORTED_BY SourceEvidence` | One or more evidence items for source-backed assertions | Here “supported” means extraction/provenance support, not clinical validation. Field-support roles distinguish amount, population, rating and main excerpt. |

**Vocabulary relationships:** retain MeSH predicates such as preferredConcept, preferredTerm, broaderDescriptor, broaderConcept and mappedTo with their original source namespaces, direction and snapshot provenance. They are not aliases of ingredient identity or clinical relations. If exported as direct graph edges later, they still need source attribution; physical representation is a later decision.

**Clinical roles:** indication, population and interaction partner retain source text and field support. Later concept links must state their role (e.g. indication versus condition-associated caution). Do not turn a condition in a safety statement into a treatment indication. The entity typing and vocabulary choice for these roles remain open until field-mapping review.


### C. Conceptual diagrams

Mermaid diagrams are illustrative only; the tables above define the intended meanings. Notebook viewers without Mermaid support may show the diagram source as text. No graph data is generated.

```mermaid
flowchart LR
    SR[SourceRecord] -->|HAS_SNAPSHOT| SS[SourceSnapshot]
    EV[SourceEvidence] -->|LOCATED_IN| SS
    PA[Passage] -->|EXCERPT_OF| SS
    IO[IngredientOccurrence] -->|LOCATED_IN| SS
    IO -->|HAS_COMPONENT or HAS_FORM| IO
    LA[LookupAttempt] -->|QUERIED_FOR| IO
    MA[MappingAssertion] -->|FROM_LOOKUP| LA
    MA -->|ORIGINATES_FROM| IO
    MA -->|CANDIDATE_TARGET| PC[PubChem ChemicalCompound]
    MA -->|CANDIDATE_TARGET| TC[RxNorm or MeSH TerminologyConcept]
    MA -->|SUPPORTED_BY| EV
    IO -. accepted only: RESOLVES_TO .-> II[IngredientIdentity]
```

```mermaid
flowchart LR
    PI[Optional ProductIdentity] -->|HAS_VARIANT| PV[Optional ProductVariant]
    LS[Label snapshot] -. accepted only: DESCRIBES .-> PV
    CA[CompositionAssertion] -->|HAS_SUBJECT: one applicable context| LS
    CA -. alternative accepted subject .-> PV
    CA -->|ABOUT_INGREDIENT| I[Ingredient occurrence or accepted identity]
    CA -->|USES_SERVING| SD[ServingDefinition]
    ST[LabelStatement] -->|ON_LABEL| LS
    CL[ClinicalAssertion: pending] -->|ABOUT_SUBJECT| I
    CL -->|HAS_REFERENCE when explicit| REF[StudyReference]
    CA -->|SUPPORTED_BY| E[SourceEvidence]
    ST -->|SUPPORTED_BY| E
    CL -->|SUPPORTED_BY| E
    RC[RedditComment] -->|REPLY_TO when resolved| RP[Reddit post or comment]
    TA[TopicAssociation] -->|ON_SOURCE| RS[Reddit snapshot]
    TA -->|ABOUT_TOPIC| T[Project Topic]
```

A clinical assertion's relation to an ingredient is not evidence that a particular commercial product inherits the same effects. That requires separately reviewed identity, formulation, population and applicability context. Reddit reports do not acquire clinical-reference status through sharing an ingredient or topic.


### D. Context, conflict and review semantics

| Concern | Proposed representation | Implication for answers |
|---|---|---|
| Equivalent quantity representations | Original amount/unit retained in evidence; later normalized values and rule version on derived assertions | 1.5 g can be compared with 1500 mg, but equivalent numbers alone do not unify products or ingredient forms. No conversion is performed now. |
| Quantity scope | Structured operator, amount/range, unit and explicit serving basis on CompositionAssertion; source ServingDefinition remains accessible | A label amount per tablet differs from the same number per two tablets; a bound is not an exact dose. |
| Clinical dose context | Full source dose text plus amount/bounds, unit, regimen, frequency, duration, route, population and indication as available | Preserve text-only values and unknown fields; do not infer missing preparation or indications. |
| Conflict | Source-scoped assertions with a conflict/review flag and evidence linking the disagreeing fields | Both values remain inspectable. Withhold the affected unsupported calculation, not unrelated ingredient-presence information. |
| Unknowns | Preserve raw missing/null/empty/placeholder distinctions in source metadata; interpreted value can remain unresolved | NP/zero is not automatically zero content; absent warnings or annotations are not negative findings. |
| Review state | Separate capture/integrity status, identity-mapping review and clinical-annotation review | “Hash verified” and “schema valid” are not mapping approval or medical approval. |
| Assertion consolidation | One equivalent assertion may later have multiple evidence links, only within established subject and context | Repeated label captures or overlapping passages do not multiply independent clinical evidence. |
| Source versions and dates | Retain record identity plus snapshots; preserve source revision/entry/publication and collection dates separately | New captures do not silently invalidate traceability to an old answer. Capture dates are not proof of current availability. |
| Unresolved references | Keep external IDs/URIs and local resolution status without fabricated nodes containing inferred facts | A missing MeSH target, Reddit parent or citation is a coverage gap, not automatic proof the source link is wrong. |

For now, review metadata and conflicts are attributes/evidence links rather than a separate workflow ontology. Detailed status enums, temporal IDs, normalization units and merge keys remain Step 4–5 decisions.


### E. Walkthroughs against the audited cases

**1. BPI Creatine Alkaline (DSLD 25731 / 43261).** Create two conceptual source records/snapshots, ingredient occurrences and source-scoped composition assertions. Each original quantity retains its unit and two-capsule basis. A future normalized amount can identify 1500 mg = 1.5 g; that alone does not create shared product/formulation identity. Other-ingredient assertions remain label-specific, including differences on 43261. Only after establishing applicable common subject/context can equivalent composition assertions share an assertion with both evidence links. Source labels remain intact.

**2. Jarrow Magnesium Optimizer (7841 / 212304).** Keep 100 mg at one-tablet and two-tablet bases as distinct composition assertions linked to their respective serving definitions and directions. Neither name grouping nor equal displayed amounts makes them equivalent. This supports Q1 and prevents losing serving context in Q2-style comparisons.

**3. Creatine HCl terminology candidates.** A DSLD IngredientOccurrence remains valid even without IngredientIdentity resolution. The lookup attempts create separate mapping candidates: PubChem CID 134732; RxNorm 2907 and 1310467 from normalized search; MeSH no result. Preserve every candidate and method, but create no accepted same-identity link to those targets. MeSH's no-result attempt has no artificial candidate node. This supports Q8's unresolved answer.

**4. BetterYou Iron Spray (228638 / 257287).** The 257287 snapshot retains numeric 52 mL and source-note 0.52 ml in separate evidence locations. The composition assertion and serving definition expose the conflict; no correction is chosen. Iron presence and source-label information remain retrievable, while the affected normalized calculation remains unresolved. The discrepancy can be explained for Q10 without claiming its cause.

**5. NatMed claims and citation context.** A monograph snapshot owns its exact subject occurrence (e.g. Fish Oil, not the broad Omega-3 topic). ClinicalAssertion retains the upstream claim ID, type, scope, fields and pending review. Evidence links include the main excerpt and field-support anchors; an earlier effectiveness heading can be a separate evidence item. Several Vitamin B12 claims can point to one paragraph while retaining different populations/doses. StudyReference links require explicit source citation linkage; nearby bibliography entries are not enough. This supports Q4–Q7 without flattening clinical context.

**6. Reddit inherited topics and missing parents.** A comment snapshot retains its original text, native identity, thread/parent IDs and user-report role. TopicAssociation records `thread_context` separately from a direct text match. Missing parent content remains unresolved; a TopicAssociation is not a clinical assertion that the commenter used, benefited from or was harmed by the supplement. A passage remains bound to this comment, preserving the same scope. This supports Q9 without treating inherited topics as first-hand evidence.

**7. Botanical and probiotic scope.** Withania somnifera, Withanolide D, Withania, a root extract, Probiotics, Lactobacillus acidophilus and a strain-qualified occurrence cannot be collapsed into one generic “supplement” entity. Source-specific occurrences/concepts and candidate mappings retain the distinctions. Accepting any shared identity or broader/narrower relation requires a later review decision.


### F. Proposed invariants, open decisions and handoff

These are design-review checks, not executed graph validation tests:

1. Every assertion and mapping has reconstructible source evidence, including the source snapshot and locator.
2. Source identities and candidate targets remain namespaced; display names are never sufficient identity keys.
3. Quantities preserve operator, unit and basis; parent blend/form amounts are not propagated to children.
4. Source scope survives joins: product label, monograph subject, chemical/vocabulary target and Reddit topic are not interchangeable.
5. No candidate mapping is treated as accepted without an explicit review decision; no clinical approval is inferred from structural checks.
6. Multiple source captures/passages can support one assertion later without inflating independent evidence counts.
7. Missing or conflicting information does not become an invented value, a negative clinical finding or a fabricated source record.
8. Historical evidence and unresolved external references remain inspectable when new data is added.

| Open decision | Proposed starting point | Resolve when |
|---|---|---|
| Minimal physical graph representation | Shared provenance pattern; preserve conceptual distinctions but avoid separate nodes for every scalar | Steps 4–7; conceptual roles do not yet prescribe CSV files or Neo4j labels |
| Product versus formulation/package modeling | Label-first assertions; optional reviewed product/variant identity | Source mappings and identity rules, especially repeated-label examples |
| Identity scope and canonical ingredient IDs | Explicit substance/preparation/species/strain/mixture scope; local stable IDs only after resolution | Step 5; no automatic identity creation in this draft |
| Conditions, populations and interaction partners | Preserve typed source wording/support first; optional reviewed links to terminology concepts | Step 4 and subsequent terminology review |
| Mapping/review statuses and relation vocabulary | Preserve unresolved candidate state; keep identity review distinct from clinical review | Steps 4–5 |
| Citation granularity | Source reference labels plus bibliography provenance; external study identity only when established | Step 4 and source-support review |
| Temporal identity and new captures | Stable source record + versioned snapshots and evidence | Steps 4–5, including invalidation/re-review when source content changes |
| Graph storage/export and later GNN inputs | Conceptual design remains storage-independent; preserve typed relations and IDs | After schema review and transformation validation, not now |

**Step 3 outcome:** a reviewable shared conceptual draft with entity meanings, relationship direction/cardinality, source/evidence boundaries and concrete case walkthroughs. No claim is made that the graph has been constructed or that these invariants have passed against a generated graph.

**Next planned step:** Step 4 — map the selected source fields to this draft and examine where the proposed roles can share representations without losing context. Review the open decisions before finalizing the schema. Normalization, deduplication, graph construction and publishing remain later work. `read.md` and raw datasets are unchanged; no GitHub push is performed.


## Step 4 — Source-field mappings into the conceptual draft

**Status: mapping specification only.** These tables explain how the selected source fields would populate the proposed roles. They do not run extractors, normalize units, mint canonical ingredient/product IDs, merge assertions, change review states or build a graph. Paths were checked against the current source structures and selections; this is not an exhaustive executed transformation test.

### Mapping conventions and shared provenance

- `[]` means every element, not just the first. `{key}` means a source dictionary key. JSON paths below are source access notation; evidence must retain a precise per-record pointer/span when extraction is implemented.
- Target attribute names are **draft logical names**, not finalized CSV columns or a database schema. “Copy” preserves original value and type; normalized values are separate future fields.
- **Native:** copied from a source. **Derived:** constructed by our pipeline and labeled as such. **Conditional:** a link can be created only when its identity/support requirements are met. **Deferred:** retained in raw storage but not initially projected.
- Missing, null, empty, placeholder and unresolved interpretation remain distinguishable. No default of zero, empty population, approved mapping or “safe” is inferred.
- Stable source identifiers can identify source objects without resolving cross-source ingredient identity. SourceRecord and ChemicalCompound/TerminologyConcept can be logical views of the same source-defined entity; their snapshots/properties need not be duplicated physically.

| Source input | Proposed target | Mapping / constraints |
|---|---|---|
| Source-native ID and namespace | `SourceRecord.source_id`, `source_system` | Native identity; namespaced. Label IDs, RxCUIs, CIDs and MeSH term/concept/record URIs are not interchangeable. |
| Saved payload, collection metadata, manifest hash and source path | `SourceSnapshot.payload_ref`, `capture_metadata`, `source_version`, `file_sha256` | A file hash describes the file, not every object inside it. Preserve upstream content hashes separately with their stated scope. Exact snapshot-key policy is Step 5. |
| JSON pointer / text span / RDF statement and its source locator | `SourceEvidence.locator`, `locator_kind`, `original_value`; `LOCATED_IN` snapshot | Derived locator over native evidence. Retain the full path to a claim or nested row and the enclosing source file. |
| Audit status and output manifest | Separate `validation_status`, `audit_rule_version`, `audit_report_ref` | Derived, never substituted for source clinical-review or identity-review states. |
| Source-specific dates | Distinct publication/entry/revision/collection fields | No single generic “date” that erases meaning. Preserve original date text; typed-date conversion rules are later work. |

**Assertion context:** initial CompositionAssertions use label-snapshot subjects and source IngredientOccurrences. ProductVariant/IngredientIdentity targets are optional accepted links, not prerequisites for extraction. Retain occurrence evidence after identity resolution.


### A. DSLD — labels, composition, servings and statements

Input: `data/raw/dsld/labels/label_<id>.json` (verified equivalent collection entries remain available).

| Source field / path | Proposed target | Mapping / cardinality / qualification |
|---|---|---|
| `id`, `brandName`, `fullName` | Label SourceRecord ID; snapshot brand/name | One record per native label ID, captured separately from a proposed product identity. Brand/name grouping remains a candidate. |
| `entryDate`, `offMarket` | Snapshot `source_entry_date`, original market-status value | Retain original flag; interpreted status is derived. These do not establish current availability. |
| Recursive `ingredientRows[]`, including `nestedRows[]` | IngredientOccurrence; label-scoped CompositionAssertion(s) | One occurrence per row path; preserve `ingredientId`, `name`, `description`, `notes` and parent context. Do not flatten nested rows into unrelated ingredients. |
| Each row's `forms[]`: `ingredientId`, `name`, `prefix`, `percent`, `order` | Form-role IngredientOccurrence and parent `HAS_FORM` link | One form occurrence per source path. Prefix/percentage/order qualify the link or occurrence; parent amount is not a form amount. |
| Each row's `quantity[]`: `quantity`, `operator`, `unit` | CompositionAssertion `original_amount`, `operator`, `original_unit` | One quantity assertion per entry under the same ingredient occurrence; keep original numeric representation in evidence. If entries are absent/unusable, retain ingredient presence with an unresolved amount. |
| `quantity[].servingSizeOrder` | `USES_SERVING` via label-snapshot-local serving order | Join only within the same label capture; retain unresolved reference if missing or invalid. |
| `quantity[].servingSizeQuantity`, `servingSizeUnit` | Assertion `explicit_basis_quantity`, `explicit_basis_unit` | Preserve separately from the referenced definition. Do not impute a missing basis or silently overwrite a disagreement. |
| `servingSizes[]`: `order`, `minQuantity`, `maxQuantity`, `unit`, `minDailyServings`, `maxDailyServings`, `notes`, `inSFB` | ServingDefinition | One per label/order; keep ranges and missing bounds, notes and source flag. No inferred daily total. |
| `otheringredients.ingredients[]` and forms | Other-ingredient-role IngredientOccurrence and presence assertion | Preserve ingredient ID/name, forms and qualifiers. Do not invent a dosage or make these indistinguishable from listed ingredient rows. |
| `otheringredients.text` | Label-scoped supporting text/evidence | Preserve verbatim when present, independently of the structured list. Null list/text is not proof of absence. |
| Selected `statements[]`: `type`, `notes` | LabelStatement with `ON_LABEL`, evidence link | Include directions, all precaution types and storage, using the saved type values. Exclude general marketing text from the initial projection. |
| `upcSku`, `productVersionCode`, `netContents`, `physicalState`, `labelRelationships` | Internal product/variant identity-review evidence | Preserve multiplicity, source relation type/direction and external targets; do not automatically populate accepted ProductIdentity/ProductVariant links. |

**Deferred:** contacts/images/branding, general marketing statements, claim-classification codes and daily-value percentages remain in raw storage. `NP`/zero remains source evidence with unresolved normalized amount; safe quantity conversion and deduplication are not performed here.


### B. PubChem, RxNorm and MeSH — coordinated terminology layer

All three sources share the same original query/occurrence inventory but have different target meanings. Candidate linking must not create `RESOLVES_TO` or same-identity edges automatically.

| Shared source field | Proposed target | Mapping rule |
|---|---|---|
| `queries[{key}].query`, `variants` | LookupAttempt query text/variants | The grouping key joins audit tables; it is not a canonical ingredient ID. |
| `queries[{key}].occurrences[]`: `dsld_id`, `ingredient_id`, `name`, `kind`, `json_path` | `QUERIED_FOR` the exact DSLD IngredientOccurrence | Resolve label + source path; same string with another role/path remains a separate occurrence. Retain source hashes from `dsld_provenance`. |
| Occurrence `search_categories` | Retrieval-context metadata / optional TopicAssociation | Categories can include co-ingredients; not an indication or ingredient-equivalence link. |
| Search status, method, complete candidates, request trace | LookupAttempt + source-backed MappingAssertion per origin/target | One query can serve many occurrences. Keep the grouped candidate set; mapping acceptance is scoped to the origin. No-match creates an attempt without a target. |
| Manifest/source hashes, collection times and version metadata | SourceSnapshot / evidence lineage | Bind target representations and lookup results to saved source material; source method is not a confidence score. |

#### PubChem

| Native source path | Proposed target | Constraint |
|---|---|---|
| `records[{cid}].cid`, `properties.CID` | ChemicalCompound / source identity | Validate agreement with dictionary key; namespaced CID. |
| `properties.Title`, `IUPACName` | Source display/systematic-name attributes | Missing IUPACName remains missing; a title is not an identity key. |
| `MolecularFormula`, `MolecularWeight`, `ConnectivitySMILES`, `InChI`, `InChIKey` within properties | Versioned compound properties | Preserve values/types and structural distinctions; molecular weight is not a dose. Shared strings do not approve a merge. |
| `synonyms.CID`, `synonyms.Synonym[]` | Source-scoped aliases | Many names per CID; a name may occur under multiple CIDs. No automatic registry-code parsing or semantic equivalence. |
| `search_results[{key}].cids`, `status`, `response`, `requests[]` | LookupAttempt result and candidate MappingAssertions | Retain original response and sorted candidate-list distinction; no rank inferred from order. |
| `compound_responses[]`: `kind`, `requested_cids`, status, response, requests | Property/synonym SourceEvidence | Link each selected property to a batch response and pointer; retain raw payload rather than duplicating it as clinical evidence. |

#### RxNorm

| Native source path | Proposed target | Constraint |
|---|---|---|
| `records[{rxcui}].rxcui`; `properties.response.properties.rxcui`, `name`, `tty`, `language`, `suppress` | TerminologyConcept identity and versioned attributes | Keep source concept type and suppression separate from mapping/clinical status. |
| Basic `synonym`; `all_properties.response.propConceptGroup.propConcept[]` with category `NAMES` | Source-scoped concept names | Blank basic synonym does not erase names from allProperties. |
| Property entries with category `CODES`, names UNII_CODE/DRUGBANK/SNOMEDCT | Namespaced source cross-reference values | Preserve all values and supporting entries; these are not accepted cross-source identity links. |
| Property entries with category `SOURCES` | Vocabulary-origin metadata | Not clinical study citations. |
| `search_results[{key}]`: status, match_method, rxcuis, requires_identity_review; `searches[]` | LookupAttempt + MappingAssertion evidence/review inputs | Retain exact/normalized sequence and request parameters. A false source review flag on no-match does not mean approval. |
| `properties` / `all_properties` status, valid, requests and responses | Endpoint-validation/source evidence | Upstream `valid` indicates collection checks only. |
| `version_start`, `version_end`, `search_policy`, collection timestamps | Snapshot version/method metadata | Preserve vocabulary and API versions separately. |

Bulk SPL_SET_ID and other unselected properties stay raw. Blank `umlscui` is not a usable identifier; no UMLS CUI is synthesized.

#### MeSH RDF

| Native binding / predicate | Proposed target | Constraint |
|---|---|---|
| Resolution bindings `term`, `concept`, `record`, `type` | Distinct MeSH TerminologyConcept roles and source resolution links | Keep full URIs for term/concept/descriptor/SCR entities; do not collapse nodes sharing a label. |
| RDF subject with `rdf:type`, `meshv:identifier`, `rdfs:label` | Source identity/type/label attributes | Associate properties with their actual subject, not blindly with the batch's record key. |
| `meshv:concept`, preferredConcept, term, preferredTerm | Typed source vocabulary relationships | Preserve direction and provenance; these are not accepted ingredient-equivalence relations. |
| `meshv:prefLabel`, altLabel, scopeNote | Source names/scope descriptions | Keep on the relevant term/concept subject. Scope notes do not supply clinical approval. |
| `treeNumber`, broaderDescriptor, broaderConcept, narrowerConcept | Source hierarchy attributes/edges | Preserve multiple placements and unresolved external targets. Tree absence can be expected for an SCR. |
| `mappedTo`, preferredMappedTo, relatedConcept, seeAlso | Distinct source relationships | Do not translate into a generic same-as edge. |
| `active`, lastActiveYear, dateCreated/dateEstablished/dateIntroduced/dateRevised/lastUpdated | Source status/date attributes | Preserve missing predicates and native datatypes. |
| `search_results`, `candidate_record_index`, resolution/subgraph response batches | Lookup/candidate/statement evidence | Keep raw term match, complete candidate records, batch context and completeness/truncation flags. |
| RDF object `type`, `value`, `datatype`, language tag | Typed source value in evidence/properties | URI is not literal; language/datatype differences remain meaningful. |
| Start/end source metadata, current endpoint and collection dates | Snapshot provenance | Available-year metadata is not a verified immutable release identifier. |

Clinical ratings, dosing, population and interaction claims are **not supplied by these collections**; no mapping into ClinicalAssertion is proposed from terminology attributes.


### C. NatMed — monograph scope, annotations, source spans and references

| Native source path | Proposed target | Mapping / qualification |
|---|---|---|
| Monograph `id`, `source`, `record_type`, `title`, `source_url` | Monograph SourceRecord and snapshot identity/display | Preserve upstream ID and exact source title; source-title IngredientOccurrence is distinct from a broad Topic. |
| `text`, `sections[].heading/text`, `content_hash` | Snapshot content and evidence locations | Keep full text/sections accessible. Section offsets refer to section text, not automatically full-monograph offsets. |
| `collected_at`, `first_collected_at`, `source_updated_at`, `source_batch_id`, `schema_version` | Snapshot capture/source metadata | Distinguish page update, capture and source schema versions. Do not infer literature-review cutoff from update date. |
| `evidence_role`, `requires_clinical_review`, `capture_quality`, `normalization`, `supplements[]` | Source-role/quality and topic-context metadata | `clinical_reference` and an upstream exact-title mapping do not approve clinical facts or broader topic identity. |
| Claim `id`, `claim_type`, `claim_text` | ClinicalAssertion source identity/type/excerpt | Keep all five annotation types. Use source-version context when preserving later annotation changes; exact ID/version policy remains Step 5. |
| Claim `ingredient.name`, formulation, strain | Exact subject occurrence and context attributes | Link to a reviewed IngredientIdentity only conditionally. Missing formulation/strain remains unknown. |
| `population`, `indication` | Distinct source-role context values with support | No inferred condition mapping. A condition-associated caution is not a treatment indication; retain source section heading. |
| `effectiveness_rating`, `safety_rating`, `adverse_effect_category`, `evidence_grade` | Separate ClinicalAssertion attributes | Preserve publisher wording; no invented numeric rating or substitution between dimensions. |
| Full `dose` object: text, value/minimum/maximum, unit, qualifier, regimen, frequency, duration(value/unit/qualifier), route | Structured clinical dose context plus original text | Do not equate with DSLD label servings. Preserve composite units, alternatives, bounds and unknowns. |
| `interaction.partner`, rating, severity, occurrence | Separate interaction-context attributes | Partner remains source text until role-specific identity mapping is reviewed. No invented interaction partner type or probability. |
| Claim `source.record_id`, url, content_hash, section_index/heading, start/end, excerpt | `SUPPORTED_BY` SourceEvidence and monograph binding | Validate source version, exact excerpt and offset coordinate system. |
| `field_support` dotted paths and start/end offsets | Additional field-role SourceEvidence items | One evidence item per supported field as needed; an earlier effectiveness heading can lie outside the main excerpt. |
| `review.state`, reviewer, reviewed_at | Clinical-review attributes | Copy pending state; structural validation cannot approve it. |
| `reference_documents[]`: url, title, text, captured_at, content_hash | ReferenceDocument snapshot/evidence and `BIBLIOGRAPHY_FOR` link | Capture pages are not already verified StudyReference entities. Preserve separate hashes and URLs. |
| Monograph/passage `citation_links`, original `references` text and reference fingerprints | Source citation context/evidence | Local reference labels remain scoped to a source bibliography. `references` can include footer text; do not presume it is a clean study list. |
| Extraction report issues, manifest schema/input hashes and method | Annotation provenance and unresolved-issue metadata | Derived/imported review context, not new clinical facts. |

**Citation gap:** current clinical annotations have no dedicated structured citation list. Mapping an assertion to StudyReference is therefore **conditional/deferred** until a cited local label is explicitly located in its source span and resolved against the relevant bibliography. Do not attach all monograph references or all overlapping-passage references to a claim. PMID/DOI metadata may identify/enrich a study later but does not establish evidential support.

**Source-text fallback:** selected safety/warning/formulation context may exist in monograph sections without an extracted claim. Keep retrieval access to those sections and their actual headings. Do not force every passage into a clinical annotation or infer missing claims.


### D. Reddit — source reports, conversations and topic associations

| Native source path | Proposed target | Mapping / qualification |
|---|---|---|
| `id`, `native_id`, `record_type` | RedditPost/RedditComment SourceRecord specialization | Native ID distinguishes records even when text repeats. Validate t1/t3 role; no text-based identity merging. |
| `title`, `text`, `content_hash` | Snapshot original content/hash | Empty comment title is legitimate; preserve text as user report, not a clinical assertion. |
| `source`, `evidence_role`, `requires_clinical_review` | Source-role/context attributes | Retain `user_report`. The source boolean is not a NatMed approval workflow. |
| `source_url`, `source_created_at`, `source_updated_at`, `collected_at`, `first_collected_at`, `schema_version` | SourceSnapshot provenance/dates | Null update dates stay unknown. Preserve publication versus collection time. |
| `thread_id`, `parent_id` | Conditional `IN_THREAD` / `REPLY_TO` + original target IDs | Link only to observed records; retain missing/unresolved target identifiers. No fabricated parent content. |
| `subreddit` | Snapshot/community metadata | Retrieval context, not an evidence-quality rating. |
| `supplements[]`: supplement, matched_terms | TopicAssociation to project Topic | Preserve each source-associated topic and its term list. This does not create IngredientIdentity or a claim of use. |
| `mention_scope`, `normalization` | TopicAssociation source-scope/method metadata | Scope is record-level. Even `own_text` does not prove every attached topic was individually mentioned; retain that limitation. |
| Audit context gaps, repeated-text candidates and validation metadata | Derived flags/source evidence references | Preserve source IDs and conversation distinctions; no deduplication from repeated text alone. |

No ClinicalAssertion fields are populated from Reddit in this POC selection. If dose/benefit/interaction experience extraction is added later, it requires a separate annotation contract. Null scores and NatMed-specific empty metadata remain outside the initial projection. Do not add author/profile data; free text still requires appropriate handling before publication.

### Shared passage mapping (NatMed and Reddit)

| Passage fields | Proposed target | Constraint |
|---|---|---|
| `id`, `record_id`, `text`, `title`, `section` | Passage and `EXCERPT_OF` the source snapshot | Preserve upstream identity and exact window; a chunk is not an independent source record. |
| `source_content_hash`, `section_index`, `word_start`, `word_count`, `chunk_words` | Source binding and window metadata | Word-window coordinates differ from clinical character offsets. Do not interchange them. |
| Source/url/role/type, capture/update/publication dates as supplied | Passage provenance or inherited source links | Keep meanings source-specific. Missing fields are not filled from unrelated sources. |
| NatMed citation_scope, reference_scope, citation_links, reference_documents_sha256 | Passage citation context | These are not claim-to-study support approvals. |
| NatMed capture/review metadata and topic normalization | Passage scope/review context | Preserve source limits and pending status. |
| Reddit mention_scope, thread_id, parent_id, supplements, normalization | Inherited conversation/topic context | Do not promote an inherited thread topic to direct mention or use. |

A ClinicalAssertion can overlap multiple passages or draw a rating from an earlier heading. Its primary support remains source-span evidence, not an arbitrary passage ID. Overlapping passages must not inflate evidence counts.


### E. Fit review: schema gaps exposed by the field mappings

| Finding | Proposed clarification to the Step 3 draft | Status |
|---|---|---|
| A single row can have forms, nested components and multiple quantities | Keep occurrence identity row-based; give each quantity/basis its own assertion. Forms/components attach to the occurrence, with assertion access through that occurrence. Presence survives an unusable quantity. | Clarification proposed; extraction rules remain Step 5 |
| Canonical product/formulation/ingredient identities are not supplied uniformly | Extract source-scoped labels, occurrences and assertions first. Optional accepted identities cannot be required non-null targets. | Consistent with the conceptual draft; no identities allocated |
| “Sources” include both content records and API query results/batches | Keep LookupAttempt/SourceEvidence tied to the enclosing response snapshot/file. A collection manifest and a compound record have different hash scopes. | Snapshot/locator details to finalize in Step 5 |
| PubChem/RxNorm source records and domain nodes can otherwise duplicate information | Treat ChemicalCompound/TerminologyConcept as source-defined identity roles with snapshots; do not require duplicate payload nodes merely to satisfy naming conventions. | Physical storage design deferred |
| Conditions and partners are strings, not validated canonical entities | Preserve role-qualified text and field-support spans. Optional later concept links must distinguish indication, population/caution and interaction partner. | Unresolved vocabulary/identity choice; does not block faithful extraction |
| NatMed bibliography records are not explicit claim citations | Use source-local reference labels/evidence first. StudyReference links stay absent/unresolved until citation-specific support is established. | Deferred linking, not an inferred relation |
| Clinical character offsets and retrieval word offsets differ | Explicit locator kind and coordinate system are required on evidence. | Required schema clarification |
| Reddit scope applies to the record, not each topic match | TopicAssociation carries the recorded scope plus its record-level limitation; do not label every topic “directly mentioned.” | Required schema clarification |
| Source review flags have different meanings | Keep capture completeness, validation, identity review, clinical approval and user-report role separate. | Required schema clarification |
| MeSH hierarchy targets can lack local descriptions | Preserve external URI and resolution state; a reference can exist without invented labels/content. | Required schema clarification |

**Selection outcome:** the selected data fits the conceptual roles with the clarifications above. This does not prove that an implemented graph passes validation. No new clinical fields, source corrections or mapping approvals were introduced.

**Step 4 acceptance review:** all six source selections have mapping destinations or explicit deferrals; native values and identifiers retain provenance; multiplicity, unknowns, role distinctions and conflicts are documented. Proposed node/attribute names remain provisional, and missing source links are not silently manufactured.

**Next planned step:** Step 5 — specify precise identity, normalization and deduplication rules, including units, source-version identity, mapping statuses, evidence locators and assertion-equivalence keys. Review the proposed clarifications before finalizing the schema. No transformations, merging, graph creation, `read.md` update or GitHub push has occurred in Step 4.


## Step 5A — Identity rules (design specification only)

> **Version-history refinement:** Step 5A sections 27–30 specify raw-byte integrity, proposed lossless JSON fingerprints, separate capture events and targeted re-review (`version-history-draft-v1`). Earlier snapshot conventions are refined there; existing manifests are not migrated.

**Scope:** establish how objects are identified and how shared-identity decisions are recorded. These rules are proposed for implementation review. No IDs are minted, source mappings approved, entities merged, quantities normalized or raw files changed in this step. Normalization and deduplication policies remain separate Step 5 work.

### 1. Three identity layers

1. **Source identity:** a source-defined record or concept. Native namespace + identifier can identify it without proving that another source's object has the same meaning.
2. **Source-version and occurrence identity:** a particular payload version, field occurrence, passage or assertion extracted from it. Identical text elsewhere does not make it the same occurrence.
3. **Shared semantic identity:** our reviewed ingredient/product identity, with explicit scope. It is optional until evidence establishes the match. Source objects remain accessible regardless of this decision.

**Identity is not name equality, amount equality or relationship similarity.** “Same source object,” “same ingredient,” “same formulation,” “same package,” and “equivalent assertion” are different decisions. A missing qualifier is unknown, not a wildcard that matches every form or strain.

### 2. Proposed identity keys and namespaces

The tuples below are conceptual key inputs. Implementation should use typed, length-safe serialization (for example canonical JSON arrays), not ambiguous string concatenation. Hashes identify representations, not medical truth. Canonical serialization details and migrations require implementation tests before adoption.

| Object | Proposed stable key / allocation rule | Boundary |
|---|---|---|
| DSLD label source record | (`dsld`, `label`, native `id`) | Label identity, not commercial-product identity. Preserve the native value; do not use brand/name as the key. |
| NatMed monograph source record | (`natmed`, `monograph`, imported native record ID) | Preserve the imported ID and publisher monograph ID/URL as supporting metadata. Do not replace it with a broad supplement topic. |
| Reddit post/comment | (`reddit`, native full ID including `t3_` / `t1_`) | A post and comment remain different types. Same text in different threads remains separate. |
| PubChem compound | (`pubchem`, `CID`, native CID) | A source-defined compound identity, not automatic equivalence with every DSLD query returning it. |
| RxNorm concept | (`rxnorm`, native RxCUI) | Retain term type/status in the versioned representation. Renames or lifecycle changes must not silently redirect existing mappings. |
| MeSH entity | Exact source URI | Preserve descriptor, supplementary record, concept, term and tree-number roles. Do not merge distinct URIs because their labels match. URL/URI alias rules are not assumed here. |
| SourceSnapshot | (source-record key, fingerprint of the full retained source payload) | Use a hash with explicitly documented scope. Upstream text-only hashes alone do not identify changes to metadata, sections, references or review flags. |
| Capture event / lookup response snapshot | Stable collection/run ID plus response locator and response fingerprint | Keep capture time/version metadata even when content repeats. Repeated captures can point to the same content snapshot while remaining distinct collection events; no mandatory extra graph node is implied. |
| IngredientOccurrence | (snapshot key, precise JSON pointer or documented text locator, occurrence role) | A form and its parent are separate occurrences; the same source ingredient ID used on many labels does not collapse their contextual occurrences. |
| DSLD ingredient registry reference | (`dsld`, `ingredient`, `ingredientId`) when available | A reusable source identifier reference, distinct from both occurrence identity and accepted cross-source IngredientIdentity. Missing IDs do not prevent occurrence creation. |
| ServingDefinition | (label snapshot, source serving order) | Require uniqueness within that snapshot. Missing/duplicate order stays an issue; do not choose an arbitrary definition. |
| SourceEvidence | (snapshot, locator kind, locator coordinates, field/support role) | Distinguish JSON pointers, section character offsets, section word windows and RDF statements. Same numbers in different coordinate systems are not the same locator. |
| Source-backed assertion occurrence | (snapshot, source assertion ID or locator, assertion type, occurrence role) | Preserve upstream NatMed claim IDs. Separate quantity entries require separate locators; absence of source ID uses deterministic source-local identity, not text alone. |
| Passage | (source snapshot, section, word start/count, chunking-rule version) plus retained upstream passage ID | Rechunking or source changes must not overwrite old evidence bindings. Upstream IDs can remain lookup aliases when unchanged. |
| LookupAttempt | (source system, capture/run, query group, method/stage, response locator) | Retain exact and normalized stages, unsuccessful attempts, parameters and original query. Name grouping does not create IngredientIdentity. |
| MappingAssertion | (origin occurrence/identity, target source entity, lookup/evidence context, proposed relation) | Multiple alternatives and methods can coexist. Acceptance/rejection history is metadata on versioned decisions, not a reason to delete original candidates. |
| Topic | Locally allocated topic ID with explicit project scope | Omega-3 and Probiotics remain categories, not automatically chemical/species identities. |
| TopicAssociation | (source snapshot, topic ID, source association locator) | Preserve record-level mention scope and matched terms. Identical topic names do not unify source records or imply direct use. |
| ReferenceDocument | (publisher/source URL identity, captured document fingerprint) | Keep monograph/bibliography context; not every reference document is an individual study. |
| Local citation occurrence | (monograph snapshot, reference-document context, publisher reference label) | Reference number 123 in one source must not be assumed identical to 123 elsewhere. |
| StudyReference | Accepted external PMID/DOI identity with cross-reference review; otherwise a source-local citation identity | A PMID and DOI are not assumed equivalent without corroboration. Metadata enrichment is not proof of claim support. |
| IngredientIdentity / ProductIdentity / ProductVariant | Persisted local opaque ID allocated after scope review | Do not derive canonical IDs from preferred names, first candidate, current list order or a hash of mutable attributes. Rename without changing accepted identity; splits/merges retain lineage. |

**Snapshot clarification to Step 3:** retain both payload-version identity and capture-event provenance. Hashing only `text` is insufficient if source references, qualifiers or metadata change. The exact metadata subset for a content fingerprint will be specified with transformation/version rules; until then, retain full source-file hashes and locators rather than asserting a narrower equivalence.


### 3. Ingredient and product identity acceptance policy

> **Product boundary refinement:** Step 5A sections 21–23 define `product-boundaries-draft-v1`: product family is commercial-line identity; variant includes disclosed formulation/presentation and package. Earlier definitions are provisional and refined there.

#### Ingredient scope

Before accepting a shared identity, explicitly characterize the relevant scope: substance/chemical form, preparation/extract, biological species/strain, mixture/blend or broad category. Unknown qualifiers remain unknown. Not every ingredient needs a CID or an exact concept in all three vocabularies.

A shared-identity match requires a recorded review establishing all of the following:

1. The source occurrence and candidate target are correctly traced to their saved source versions.
2. Their definitions/representations refer to the same intended entity at the stated scope, including relevant chemical form, preparation, species/strain or mixture distinctions.
3. No unresolved source conflict contradicts that proposed identity. Matching names or shared codes alone are insufficient; inspect available definitions, structure/form qualifiers and source-supported mappings as relevant.
4. The relationship chosen is explicit: accepted identity match, broader/narrower scope, related entity/form, or unresolved. A broader match must not be represented as exact identity.
5. The reviewer, evidence, rationale, rule version and timestamp are recorded. This design step itself approves no candidates and does not establish a medical-review credential.

**Permitted automatic recognition:** identical native IDs within the same namespace can refer to one source identity, with separate snapshots/occurrences retained. This does not authorize automatic cross-source semantic identity.

**Not sufficient for ingredient acceptance:** one returned candidate, an API “exact” match, all three sources returning something, shared synonym/InChIKey/code alone, a common DSLD search category, equal amount, or an absent competing candidate. No-match is not a reason to delete or broaden the source ingredient.

#### Product and variant scope

- Initially retain label-scoped assertions; canonical product assignment is optional.
- Brand/product names, UPC/SKU, version codes, physical state, package contents, composition and source label relationships are evidence to review together—not a universal product key.
- A **ProductIdentity** can group an established commercial product family; a **ProductVariant** preserves meaningful composition/form/package differences. Shared product identity does not make labels or formulations identical.
- Do not create an accepted variant match while relevant composition/form/package differences are unresolved. Missing other-ingredient information does not prove equivalence.
- Image-only changes may support shared variant identity after source comparison; differences in quantity basis, active/other ingredients or formulation require separate context. Package-size distinctions remain explicit even if composition matches.
- Do not infer a chronology or replacement from increasing DSLD IDs or entry dates. Preserve source dates/events with their actual meaning.

Detailed product-family/variant boundaries remain a design decision to confirm during example review. No current repeated-name group is promoted to accepted product identity by this policy.


### 4. Mapping review states, relation scope and updates

> **Later design refinement:** Step 5A sections 15–17 define `mapping-semantics-draft-v2`, separating concept denotation, category membership and exact material matching. Use that refinement for new proposals; this table preserves the earlier draft.

| Dimension | Proposed values / meaning | Constraint |
|---|---|---|
| Lookup outcome | Source-native statuses retained; interpreted outcome distinguishes candidates returned, no candidate, request failure and incomplete/truncated response | No-match is not rejection; request failure is not no-match. |
| Identity review state | `unreviewed`, `needs_review`, `accepted`, `rejected` | Acceptance applies to the specifically proposed relation and origin/target scope. `needs_review` retains a concrete reason. Missing review is not approval. |
| Proposed mapping relation | `candidate_unspecified`, `identity_match`, `origin_broader_than_target`, `origin_narrower_than_target`, `related_entity` | Direction is explicit. A candidate becomes an accepted identity only through `identity_match` plus accepted review; an accepted broader relation is not identity. |
| Review reason | Structured code plus explanation/evidence, e.g. ambiguous candidates, form mismatch, organism scope, missing definition, stale source | Reasons describe unresolved evidence; they are not unsupported statements that a candidate is chemically wrong. |
| Review provenance | Reviewer ID/role, decision time, evidence snapshot references, rationale and rules version | No fabricated reviewer or confidence score. Keep identity decisions separate from NatMed clinical approval. |

`candidate_unspecified` cannot be accepted without first choosing a supported relation. A rejection of one proposed relation does not automatically reject every possible relationship to that target. Preserve the decision history.

**Updates and reversals:**

- New source captures retain existing source identity where the native identifier remains applicable, but create a new captured version/evidence context.
- A previous acceptance remains historical evidence for its original snapshots; it is not automatically carried forward to changed source scope or properties. Flag affected mappings for re-review.
- Source retirements, replacements, canonical-identity splits or merges require explicit lineage/decision records. Preserve old IDs as historical references; never silently repoint old evidence to a different ingredient.
- No transitive ingredient equivalence is inferred merely because two names share a source candidate. A reviewed canonical identity can be reused only within its accepted scope.
- Assertion identity and assertion equivalence are separate: distinct source-backed assertions remain distinct until the later deduplication rules establish compatible subject, ingredient, population, dose/basis and other relevant context.

These are proposed application-level states. They do not overwrite raw source `requires_identity_review`, NatMed `review.state`, RxNorm `valid`, or source evidence roles.


### 5. Decisions for the audited examples and design checks

| Case | Identity treatment now | What remains to resolve |
|---|---|---|
| BPI labels 25731 / 43261 | Two DSLD source identities; retain their occurrences and assertions | Equivalent mass does not establish identical formulation/variant; other ingredients differ. Product grouping needs explicit review. |
| Magnesium / Magnesium Citrate | Keep source occurrences and candidates separate | Distinguish queried ingredient scope and chemical form; no parent/salt identity collapse. |
| Creatine HCl / Creatine Hydrochloride | Preserve both source names and their occurrence IDs; shared PubChem candidate is review evidence | Potential alias relation needs scope review. RxNorm creatine/monohydrate candidates do not verify the hydrochloride form. |
| Withania somnifera → Withanolide D / Withania | Keep all source entities and unresolved candidate links | Botanical/constituent/taxonomic scope must be examined before assigning a relation. |
| Lactobacillus acidophilus / NCFM-qualified name | Separate species- and strain-qualified occurrences; retain no-match outcome for the qualified query | Missing strain lookup must not erase the strain or establish species-level equivalence. |
| NatMed Fish Oil versus project Omega-3 | Monograph source identity and scoped subject remain distinct from broad Topic | Broad topic membership is not an exact ingredient mapping. |
| NatMed annotations sharing one paragraph | Preserve each claim ID and source context | Different populations/doses are not duplicates merely because the evidence span repeats. |
| Reddit repeated text / inherited topics | Keep native post/comment identities and record-scoped topic associations | Text equality or common thread topics do not establish the same event, person, use or experience. |
| BetterYou numeric/text conflict | Source record/occurrence identity is still valid | Conflicting quantity evidence does not justify a new ingredient identity or correction; calculation handling belongs to later rules. |

**Acceptance checks for future implementation (not executed here):** same native identifier with changed payload retains source identity but gets versioned evidence; two identical strings at different paths remain separate occurrences; a form-qualified name does not resolve to a parent without an explicit relation decision; missing or contradictory qualifiers prevent automatic acceptance; accepted mappings always have scope/evidence/reviewer history; changing clinical review does not implicitly change identity review; no candidate becomes approved merely by a successful lookup.

**Step 5A outcome:** identity-key conventions, scope boundaries and review-state policy are documented for review. These are rules, not a completed entity-resolution run. The schema and exact physical ID serialization remain provisional.

**Next:** review these identity choices, then define normalization and deduplication rules as the remaining parts of Step 5. No canonical IDs were allocated, no mappings approved, no entities merged, and no raw data, `read.md`, or GitHub branch was updated by this notebook-only step.


### 6. General identity decision table

**Purpose:** define reusable decisions across all six datasets. Examples illustrate rules; they do not limit their scope. This is a proposed decision specification, not an executed classification, a completed coverage review, or permission to merge data.

**Reading the table:** “permitted decision” means the maximum conclusion the evidence supports. Semantic relations still require the review provenance defined above before becoming `accepted`. A proposal awaiting review is `unreviewed`; missing, conflicting or ambiguous evidence makes it `needs_review`. Source-identity recognition is separate from mapping approval.

| Rule | General condition | Evidence required | Permitted decision | Unresolved / blocking condition |
|---|---|---|---|---|
| ID-01 | Same native identifier in the same source namespace and entity type | Valid identifier, source provenance and compatible source object type | Recognize one source identity; retain each occurrence and capture | Suspected identifier reuse, incompatible types or malformed IDs: quarantine the identity collision for review |
| ID-02 | Same source identity appears in multiple captures | Compared retained payloads and documented fingerprint scope | Same payload may share a snapshot; changed payload requires a new snapshot, retaining all capture provenance | Unspecified fingerprint scope cannot establish version equality |
| ID-03 | Identifier is absent or invalid | Source snapshot plus stable, unique locator and object role | Retain a source-local occurrence; flag missing native identity | No stable locator: retain the source and flag extraction/identification failure; do not invent a source identifier |
| ID-04 | Different native identifiers have equal names or text | Both source identifiers, versions and locations | Keep distinct source identities; propose a semantic comparison if useful | Name/text equality alone cannot establish shared identity |
| ID-05 | One identifier appears in several contextual occurrences | Snapshot, precise locator, role and source registry reference | Reuse the source registry reference while retaining separate occurrences | Shared code cannot erase serving, formulation, population or document context |
| ID-06 | An occurrence has one or more terminology candidates | Original query, lookup method, response, candidate identifiers and definitions | Store each candidate separately with its lookup evidence | Exact API match, sole result, first result or high rank alone cannot approve identity |
| ID-07 | Cross-source objects appear to denote the same scoped entity | Traceable versions, compatible definitions and relevant form/preparation/species/strain/mixture qualifiers; no unresolved contradiction | Propose `identity_match`; accept only with recorded review | Missing identity-critical qualifiers or contradictory definitions: `needs_review` |
| ID-08 | Origin and target have a supported broader/narrower scope relationship | Definitions supporting the direction of scope inclusion | Propose `origin_broader_than_target` or `origin_narrower_than_target` | A salt, extract or constituent is not automatically a narrower instance; use a supported related relation or leave unresolved |
| ID-09 | Entities are related but are not the same scoped entity | Source support for the specific connection, such as form, constituent or preparation | Propose `related_entity`, documenting the actual relationship in the rationale | Unsupported relationship or unclear direction: `needs_review`; no identity collapse |
| ID-10 | A qualifier needed to distinguish identities is missing | Preserved original fields and an explicit account of the missing qualifier | Retain the occurrence and candidate alternatives as `needs_review` | Unknown is not a wildcard; do not infer strain, salt, extract composition or mixture members |
| ID-11 | Evidence contradicts a proposed identity match | Traceable contradictory definitions/qualifiers, reviewed in context | Reject that `identity_match` when contradiction is established; evaluate other relations separately | Uncertain extraction or conflicting sources: retain the conflict as `needs_review`, not a forced rejection |
| ID-12 | Multiple candidates remain plausible | Evidence assessed independently for every candidate at the intended scope | Keep alternatives unresolved; record what evidence would distinguish them | Ranking may prioritize review but cannot replace identity evidence; do not force one winner |
| ID-13 | No candidate is returned, or a lookup fails | Saved outcome, request context, error/truncation information | Retain the source ingredient; distinguish no candidate from request failure/incomplete response | No result does not mean the ingredient does not exist, nor justify a broader substitute |
| ID-14 | Several names or sources reach the same candidate | Each original occurrence and its own mapping evidence | Review each origin-to-target relation independently | Shared target or a chain of candidate links does not establish transitive identity |
| ID-15 | Repeated labels may belong to one commercial product family | Brand/manufacturer, product identifiers, names and source label relationships assessed together | Propose a shared ProductIdentity while retaining label identities | Names or UPC/SKU alone are insufficient; family boundaries must be explicit |
| ID-16 | Labels may describe the same product variant | Supported family relationship and compatible composition, forms, serving basis, other ingredients and package context | Propose a shared ProductVariant after all relevant differences are resolved | Equal amounts or partial composition agreement are insufficient; missing/conflicting relevant fields block acceptance |
| ID-17 | Established product family has meaningful variant differences | Source-supported differences in composition, form or package | Preserve distinct variants within the family where the evidence supports that grouping | Unclear family membership or unresolved differences: remain label-scoped; do not invent variant assignments |
| ID-18 | Assertions, passages or Reddit records repeat content | Source IDs/locators and subject, population, dose, time and evidence-role context | Preserve separate assertion/record occurrences; flag for later equivalence review | Repeated text/span is not proof of the same claim, event or experience; deduplication is a later decision |
| ID-19 | Citations may refer to the same publication | Verified external identifiers and compatible bibliographic metadata with source-local citation context | Propose a shared StudyReference; retain each citation occurrence | Equal local citation numbers or incomplete/conflicting metadata cannot establish publication identity |
| ID-20 | A broad topic or inherited thread topic matches an entity name | Topic scope, association source and mention scope | Retain TopicAssociation separately from ingredient identity | Topic membership does not establish a chemical identity, direct mention, use or clinical support |
| ID-21 | Accepted mapping encounters a revised, retired or replaced source object | Old/new versions, source lifecycle evidence and original decision provenance | Preserve historical decision; flag affected current mappings for re-review and record explicit lineage | Do not silently redirect IDs or carry acceptance to changed scope |
| ID-22 | No rule covers the case, or applicable rules conflict | Source evidence and a description of the uncovered/conflicting condition | Set `needs_review`; record a rule-gap reason and propose a versioned rule change | Never default to identity, select the highest-ranked candidate, or discard the record |

#### Applying overlapping rules

1. Establish source identity and provenance first (ID-01–05). A source-identity collision blocks dependent shared-identity acceptance until resolved.
2. Evaluate scope and evidence for each proposed relation (ID-06–14). Apply all relevant conditions: an unresolved contradiction, missing critical qualifier or ambiguity blocks acceptance even when another rule finds supporting evidence.
3. Apply entity-specific requirements (ID-15–20). Product-family acceptance does not imply variant acceptance; publication identity does not establish claim support; identity review does not grant clinical approval.
4. Apply update rules to previously reviewed mappings (ID-21). Uncovered cases use ID-22. Rule order is not a first-match-wins algorithm.

#### Decision record for the later coverage review

For each case, retain: case ID; entity type; origin and target references (target may be absent); source snapshots/locators; applicable rule IDs and rule version; scope and observed/missing qualifiers; evidence references; proposed relation (nullable for source-only cases); review state and reason; rationale; reviewer/time when reviewed; and any prior decision superseded.

Track rule coverage separately as `covered`, `rule_gap` or `not_assessable`. A covered case can still be ambiguous and `needs_review`; coverage does not mean a match was accepted. Use `not_assessable` when missing source evidence prevents evaluating the rules, and explain what is missing. Aggregate counts must state their unit (record, occurrence, candidate mapping or product group) rather than mixing these denominators.

**Status:** the general table is documented; full-dataset coverage has not been evaluated. No semantic matches were accepted, data normalized or records deduplicated. The next review can measure coverage and identify rule gaps before normalization/deduplication policies are defined.


### 7. Identity-rule coverage assessment — saved audit evidence

This assessment checks **mechanical routing for all 739 terminology query groups and all 70 exact-name product groups**, plus 13 audit findings across the six datasets. It is not a fresh baseline audit, a semantic review of every candidate, or proof that the identity rules are exhaustive. Query groups include ingredients/forms found on the 400 labels, not just the ten POC topic names.

The reusable script verifies the saved audit payloads against their manifests and checks their referenced raw evidence hashes. It also verifies that raw files remain unchanged during this run. The result retains one assessment entry per query and product group, with rule IDs and pointers back to the saved review. No clinical approvals, identity approvals, normalization or merges occur.

`covered` in query routing means an existing rule explains how to retain/review the lookup outcome. It does **not** mean identity can be accepted: scope-specific acceptance criteria remain a rule gap. Product groups are marked `rule_gap` because product-family/variant boundaries remain provisional. Counts for overlapping rules must not be summed as distinct records.


In [1]:
from pathlib import Path
import sys
import pandas as pd

identity_project = Path.cwd()
if not (identity_project / "scripts").is_dir():
    identity_project = identity_project.parent
assert (identity_project / "scripts/assess_identity_rules.py").is_file()
if str(identity_project / "scripts") not in sys.path:
    sys.path.insert(0, str(identity_project / "scripts"))
from assess_identity_rules import assess

identity_output, identity_assessment = assess(identity_project)
identity_query_review = pd.DataFrame(identity_assessment["queries"])
identity_product_review = pd.DataFrame(identity_assessment["product_groups"])
identity_gap_review = pd.DataFrame(identity_assessment["gaps"])
print("Saved assessment:", identity_output.relative_to(identity_project))
print("\nTerminology query groups by source coverage:")
print(pd.Series(identity_assessment["summary"]["terminology_source_coverage"]).to_string())
print("\nApplicable rule counts (overlap; unit = query group):")
print(pd.Series(identity_assessment["summary"]["terminology_rule_counts_overlapping"]).to_string())
print("\nProduct groups:", len(identity_product_review))
print("All 70 need product boundary clarification; no product match accepted.")
print("\nGap register:")
print(identity_gap_review[["id", "kind", "issue"]].to_string(index=False))
print("\nQuery routing preview (first 10 alphabetically; not a semantic sample):")
print(identity_query_review[["query", "applicable_rules", "review_state"]].head(10).to_string(index=False))
print("\nAccepted mappings: 0; merged entities: 0")


Saved assessment: data/processed/identity/identity-coverage-v1-f635e89799c2f4da

Terminology query groups by source coverage:
partial_source_coverage      236
none_returned                356
all_three_have_candidates    147

Applicable rule counts (overlap; unit = query group):
ID-06    383
ID-07    383
ID-13    592
ID-14    136
ID-12     43
ID-04    178

Product groups: 70
All 70 need product boundary clarification; no product match accepted.

Gap register:
id           kind                                                                                 issue
G1       rule_gap                            Scope-specific acceptance evidence is not operationalized.
G2       rule_gap                             Product family and variant boundaries remain provisional.
G3       rule_gap Missing source identity and a known target outside the snapshot are different states.
G4       rule_gap   Fingerprint serialization and lifecycle acceptance propagation are not implemented.
G5 not_assessabl

### 8. Interpretation and outstanding identity decisions

| Evidence population / unit | Finding | Rule coverage and implication |
|---|---|---|
| 739 terminology query groups | 147 have candidates in all three sources, 236 in some, 356 in none | ID-06/13 route these outcomes; none establishes semantic identity. |
| 739 query groups; overlapping flags | 43 have multiple candidates in at least one source; 178 contain multiple DSLD ingredient IDs; 136 share a target with another query | ID-04/12/14 preserve alternatives and prevent name/shared-target collapse. |
| 70 repeated-name product groups, covering 206 labels and 304 pairs | 27 groups share the compared core; 43 have core differences | ID-15/16 prevent automatic merging, but operational family/variant criteria are still missing. Equal compared core excludes some identity-relevant fields. |
| DSLD baseline over 400 labels | 291 normalized ingredient-name groups have multiple source IDs; 62 compact UPC/SKU groups are shared | ID-04/05/15 cover why names/codes alone are insufficient. The 291 groups and 178 terminology groups have different audit scopes. |
| PubChem property collision groups | 8 InChIKey, 13 ConnectivitySMILES and 21 molecular-formula collision groups | ID-07/14 require additional scope evidence; counts are groups, not compounds or accepted duplicates. |
| NatMed 1,269 claims | All pending; 13 repeated claim-text groups and 12 repeated source-span/type groups | ID-18 keeps assertion identity separate from text/span equality. Structural validity does not change clinical status. |
| Reddit 4,880 records | 230 records in 24 repeated-text groups; 3,595 records use thread context | ID-18/20 retain separate records and distinguish inherited topics from direct use. |
| Uncollected or incomplete references; different units | 451 distinct external DSLD label targets; 791 MeSH edges without target descriptions; 159 Reddit comments with parent/thread gaps; 55 upstream-reported missing NatMed bibliography references | These expose G3: define explicit unresolved-reference states. Do not add these counts together. |

**Rule coverage by family:** ID-01/04/05 have baseline source/occurrence evidence; ID-06/12/13/14 have full query-group routing; ID-15/16 have full repeated-name-group assessment but lack acceptance detail; ID-18/20 have audit-level evidence. ID-03 needs the missing-versus-uncollected reference distinction. ID-07–11 need per-case semantic evidence and scope-specific criteria; no false-match rate or acceptance rate is measured here. ID-17 cannot be validated before product boundaries are settled. ID-19 requires citation-level identity review beyond the missing-reference findings. ID-02/21 are not validated longitudinally by this assessment; historical imports exist but were not compared. ID-22 records gaps rather than making them disappear.

**Recommended order before normalization:**

1. Specify ingredient acceptance evidence by scope (G1): substance/form, botanical/preparation, species/strain, mixture and category. We can then review candidates without treating vocabulary agreement as proof.
2. Confirm product-family/variant boundaries and required comparison fields (G2).
3. Define unresolved-reference states (G3), separating absent identifier, known target outside the snapshot and captured target with incomplete description.
4. Specify version/fingerprint handling (G4); retain semantic, citation and lifecycle assessments as explicitly incomplete (G5).

The general rules are useful for conservative handling, but **routing every query is not the same as proving the rules are sufficient for identity acceptance**. The assessment found four specification gaps and a separate evidence/validation limitation. No new identity policy has been silently adopted.

The full `assessment.json` contains every query and product group; `identity_query_review`, `identity_product_review` and `identity_gap_review` remain available for filtering in this notebook. Raw data and `read.md` are unchanged.


### 9. Ingredient-specific acceptance requirements — G1 specification draft

**Status:** proposed design requirements `ingredient-acceptance-draft-v1`, refining ID-07–11. This specifies the first gap from the coverage assessment; it does not rerun that assessment, approve candidates or change its historical results. G1 is **specified, pending case review and implementation validation**, not closed. G2–G5 remain outstanding.

**Decision unit:** one origin occurrence or explicitly scoped identity → one target entity → one proposed relation. Review each origin independently, even when several queries return the same target. Preserve source identity, source wording and occurrence context throughout.

#### Common acceptance gates

| Gate | Requirement for an accepted relation | If unmet |
|---|---|---|
| A1 — Traceability | Identify both endpoints, their saved versions and locators; retain the query/method if a lookup produced the candidate. Evidence must support the endpoints actually under review. | `needs_review`: `missing_or_untraceable_evidence`. A lookup response alone is not a definition. |
| A2 — Explicit scope | Assign the applicable scope row below to each endpoint. State what identity is being asserted, including whether the subject is a generic concept or a specified material/form. | `needs_review`: `scope_unclassified`. Preserve mixed/unknown scope; do not guess from a name alone. |
| A3 — Positive semantic support | Saved definitions, source-supported aliases/cross-references or explicit descriptive fields establish the proposed relation at that scope. For every relevant qualifier, record the source value and supporting locator. | `needs_review`: `insufficient_semantic_support`. Similar strings, an API “exact” result, candidate count, rank or agreement among three lookups do not satisfy this gate. |
| A4 — Qualifier compatibility | Required distinguishing qualifiers in the applicable row are compatible. Record each as `supported`, `unknown`, `conflicting` or `not_applicable`, with a reason for `not_applicable`. | Unknown identity-critical qualifiers block exact acceptance. Missing values do not count as agreement. |
| A5 — Alternatives and contradictions | Address plausible competing candidates and contradictory evidence. A reviewed explanation must show why an accepted relation is supported despite the alternatives, or why the alternatives concern different scopes. | `needs_review`: `ambiguous_candidates` / `conflicting_evidence`. Do not choose by score alone. A demonstrated mismatch can reject that specific `identity_match`. |
| A6 — Explicit relation and review | Record relation, reviewer ID/role, decision time, rationale, rule version and evidence. All applicable gates must pass for acceptance. | Leave unreviewed until examined; use `needs_review` when a blocker is identified. No invented confidence or clinical-review status. |

There is no required number of agreeing databases. One adequate, traceable source-supported definition can be sufficient for a scoped relation; several weak lookup hits cannot substitute for it. These are project review requirements, not a claim that every saved dataset already contains the required evidence.

**Generic identity versus unspecified material:** an occurrence explicitly denoting a generic concept may map to the same generic concept without specifying a particular form. This must not be used to assign a specific form to an underspecified material. For example, matching a generic magnesium concept does not establish which magnesium-containing form a product uses. Preserve the product occurrence's unresolved form independently.


### 10. Evidence requirements by ingredient scope

The qualifiers below are required **when they distinguish the identity being asserted**. Do not require properties irrelevant to that scope, but document why they are irrelevant. A product label need not contain a molecular structure: a clear source designation plus a supported alias/definition bridge can establish a scoped identity. If the bridge cannot be supported from saved evidence, request enrichment or retain the unresolved candidate.

| Scope / requirement ID | Minimum positive evidence for `identity_match` | Distinguishing qualifiers to check | What must remain separate or unresolved |
|---|---|---|---|
| Generic substance or nutrient concept — ING-01 | Origin explicitly denotes the generic substance/concept; target definition denotes the same concept. Preserve whether the occurrence is a nutrient declaration, ingredient or form. | Generic versus specific form; nutrient/active-moiety declaration versus the material supplying it; meaning of the source term. | A generic match does not identify a salt, preparation or product formulation. A nutrient amount does not identify the mass or identity of its supplying material. |
| Specified chemical / chemical form — ING-02 | An explicit source designation or supported identifier/alias bridge to a target definition describing the same specified entity. Inspect available structural metadata where needed to distinguish candidates. | Parent versus salt/complex; counterion; hydrate/solvate; stereochemical/isomeric specification; other explicitly distinguishing chemical qualifiers. | Parent, salt, hydrate and constituent are not automatically identical or broader/narrower concepts. Shared formula, InChIKey or synonym alone does not resolve the audited collisions. Missing required form detail blocks exact acceptance. |
| Botanical organism / taxonomic concept — ING-03 | Source-supported organism designation or documented taxonomic synonym matches the target taxon at the same stated rank. | Species/genus/rank and supported synonym meaning. Use source definitions, not common-name similarity alone. | A taxon, its plant material, extract and isolated constituent are different scopes. A taxon association is not evidence of preparation identity. |
| Botanical material — ING-04 | Evidence identifies the botanical source and the same material/preparation scope on both endpoints. | Taxon; plant part; stated physical/preparation form; any defining process qualifiers. | Root, leaf, whole-plant material, extract and isolated constituent must not be collapsed. Unknown part or material type blocks a part-/preparation-specific identity. |
| Extract / standardized preparation — ING-05 | Origin and target explicitly describe the same preparation scope; any named proprietary preparation has a supported definition/identifier bridge. | Source organism/material and part; extraction/preparation type; solvent, ratio, standardization markers and concentration basis when they define the preparation. | Equal botanical name or marker percentage alone does not establish the same preparation. A generic extract concept can be matched only at its generic scope; it does not establish preparation interchangeability. Unknown defining formulation details remain blockers. |
| Microorganism species — ING-06 | Explicit source species designation or supported synonym matches the same target species concept. | Taxon and rank; whether a strain designation or multi-organism context is present. | A strain-qualified occurrence cannot be collapsed into the species. A supported species relationship can be recorded separately with explicit direction and scope. |
| Microorganism strain — ING-07 | Matching explicit strain designation, or a source-supported alias/deposit-identifier bridge connecting the strain labels, plus compatible species context. | Strain code, source namespace/collection context where relevant, species and any conflicts in the synonym evidence. | Shared species is insufficient. A commercial blend name or similar strain-code text does not establish strain identity. Unknown strain remains unknown. |
| Mixture / blend / source-derived material — ING-08 | Evidence supports the same defined mixture or material scope. For a specific formulation, require a source-supported formulation identifier/version or a sufficiently complete defining composition; resolve conflicts even when an identifier matches. | Constituent identities, proportions/ranges and basis when formulation-defining; source/preparation qualifiers; formulation version for a named blend. | Shared ingredient list, total mass, broad material name or one common constituent is insufficient for specific formulation identity. A generic mixture/material concept may be matched only as that generic concept, leaving exact composition unknown. |
| Broad category — ING-09 | Origin and target explicitly denote the same defined category with compatible inclusion/exclusion boundaries. | Category scope, taxonomic/chemical/material level and whether the origin is a topic association or an ingredient declaration. | Category membership is not identity with an individual chemical, species, strain or preparation. Project topics do not automatically become IngredientIdentity nodes. |
| Unclassified / mixed / conflicting scope — ING-10 | No exact acceptance until the scope can be supported; preserve the original occurrence and source context. | Record the missing definition, compound expression, competing scopes or evidence conflict. | Use `needs_review` and ID-22 where the specification does not cover the case. Do not infer mixture components or silently split an occurrence. |

#### Choosing the relation after scope comparison

- Use `identity_match` only when the endpoints denote the same entity at the declared scope and all acceptance gates pass.
- Use `origin_broader_than_target` / `origin_narrower_than_target` only when evidence establishes directional scope inclusion. Do not infer inclusion simply because a chemical-form name contains another name.
- Use `related_entity` when a supported connection exists without identity or demonstrated scope inclusion. Record the specific connection in the rationale (for example, preparation-to-botanical-source or material-to-constituent); this does not yet finalize new physical graph relationship types.
- Preserve `candidate_unspecified` while the relationship is unclear. It cannot be accepted. Rejecting an exact identity proposal does not reject every possible relation to the same target.
- Do not propagate dose, effectiveness, safety or interaction claims across accepted identity, broader/narrower or related links automatically. Clinical applicability and claim context require their own review.


### 11. Applying the requirements in the next identity review

#### Review record additions

Extend the proposed decision record from section 6 with: `acceptance_requirements_version`; `origin_scope`; `target_scope`; `identity_level` (generic concept or specified entity/material); applicable ING requirement IDs; `qualifier_checks` (name, origin value, target value, status, evidence locators and rationale); A1–A6 gate results; and evidence gaps needed for a decision. Use explicit nulls for unavailable values. These are proposed fields, not changes to raw files or an implemented schema migration.

A future review outcome must distinguish:

| Outcome | Meaning |
|---|---|
| `unreviewed` | Candidate has not yet received the scoped assessment. |
| `needs_review` | Assessment found missing evidence, ambiguity, contradiction needing investigation or an uncovered scope. Record the blocker. |
| `accepted` + explicit relation | All applicable gates passed for that relation, with reviewer and evidence provenance. This does not merge source records or approve clinical claims. |
| `rejected` + specific proposed relation | Review establishes that the proposed relation is unsupported by a demonstrated mismatch. Do not use rejection for a lookup miss or merely missing evidence. |

#### Audit cases as future acceptance checks — not decisions made here

| Saved case | Requirements to apply | Required review behavior |
|---|---|---|
| Creatine HCl / Creatine Hydrochloride | ING-02; A1–A6 | Seek an explicit supported alias/form bridge. A shared PubChem candidate or a generic/monohydrate RxNorm result cannot approve the hydrochloride identity by itself. |
| Magnesium declaration / Magnesium Citrate form | ING-01/02 | Preserve nutrient declaration and supplying-form roles. Review each at its own scope; do not transfer quantity meanings between them. |
| Withania somnifera query / Withanolide D and Withania candidates | ING-02/03/04/05 as supported by the occurrence | Establish whether the origin denotes organism, material, preparation or constituent before selecting any relation. |
| Lactobacillus acidophilus / NCFM-qualified occurrence | ING-06/07 | Species evidence does not resolve the strain. Retain strain context even if only a species candidate was returned. |
| Fish Oil monograph / Omega-3 project topic | ING-08/09 | Review material and category definitions separately; topic association does not establish exact ingredient identity or claim applicability. |
| Probiotic Blend with incomplete composition | ING-07/08/10 as applicable | Retain the named occurrence and unknown constituents/proportions; do not claim a specific blend identity from its total amount or broad name. |
| Any of the 135 queries with one candidate in every source | Applicable scope row; A1–A6 | Candidate availability alone satisfies neither semantic support nor qualifier compatibility. No automatic acceptance. |
| Previously unseen ingredient/preparation | ING-10 and ID-22 | Record the rule gap and preserve evidence; expand/version the requirements before forcing a match. |

**Next review:** use these requirements to inspect actual candidate evidence, starting with representative scope classes, then expanding to all relevant occurrences. Query-group routing is only a starting point: one query may span several labels or roles with different qualifiers. Do not copy a query-level decision to every occurrence without checking its context. Record supported, blocked and uncovered cases separately; no acceptance percentage can be reported until that review is performed.

**Outcome of this addition:** G1 now has a proposed acceptance specification and review-record requirements. It remains unvalidated on the full dataset. Product-family/variant boundaries (G2), unresolved-reference handling (G3), fingerprint/lifecycle policy (G4), and outstanding semantic/citation/lifecycle validation (G5) remain pending. Existing assessment artifacts are preserved as the prior baseline. No mappings approved, IDs allocated, quantities normalized, records merged, raw data edited or `read.md` changed.


### 12. Occurrence-level acceptance pilot — saved evidence only

This is a **purposive pilot**, not a full-dataset validation or a representative statistical sample. It inspects ten query classes from the acceptance examples, selecting the lowest numeric DSLD label ID and then source path within each class. This deterministic selection is for inspection, not ranking quality. Other occurrences sharing a query may require different outcomes.

The executable cell verifies the combined-review and occurrence-file hashes, the three terminology collection hashes, and each selected raw label hash. It displays the exact occurrence, its containing ingredient row for forms, every returned candidate, PubChem alias support, and saved MeSH definitions. Candidate metadata remains source evidence, not approval. No external sources were fetched.

The following interpretation is an assistant-authored **recommendation**, not a persisted `accepted`/`rejected` mapping decision. A6 approval remains pending. Prior coverage artifacts and raw files are preserved. Chemical interpretation here is confined to the saved representations; it does not establish clinical equivalence or product composition beyond the source declarations.


In [1]:
from pathlib import Path
import sys, json, csv, hashlib
from collections import Counter

pilot_root = Path.cwd()
if not (pilot_root / "scripts").is_dir():
    pilot_root = pilot_root.parent
if str(pilot_root / "scripts") not in sys.path:
    sys.path.insert(0, str(pilot_root / "scripts"))
from audit_pubchem import resolve_path

def pilot_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

pilot_dir = pilot_root / "data/processed/terminology/terminology-review-v1-34294732268f2352"
pilot_manifest = json.loads((pilot_dir / "manifest.json").read_text())
for filename in ["review.json", "dsld_occurrences.csv"]:
    assert pilot_hash(pilot_dir / filename) == pilot_manifest["files"][filename]["sha256"]
pilot_collections = {}
for source, meta in pilot_manifest["inputs"].items():
    path = pilot_root / meta["file"]
    assert pilot_hash(path) == meta["sha256"]
    pilot_collections[source] = json.loads(path.read_text())
pilot_queries = {q["query_key"]: q for q in json.loads((pilot_dir / "review.json").read_text())["queries"]}
with (pilot_dir / "dsld_occurrences.csv").open() as f:
    pilot_occurrences = list(csv.DictReader(f))
pilot_keys = ["creatine hcl", "creatine monohydrate", "magnesium", "magnesium citrate",
              "withania somnifera", "ashwagandha root extract", "lactobacillus acidophilus (ncfm)",
              "fish oil", "omega-3", "probiotic blend"]
pilot_evidence = []
for key in pilot_keys:
    occurrences = sorted([o for o in pilot_occurrences if o["query_key"] == key],
                         key=lambda o: (int(o["dsld_id"]), o["json_path"]))
    o = occurrences[0]
    path = pilot_root / o["source_file"]
    assert pilot_hash(path) == o["source_sha256"]
    label = json.loads(path.read_text())
    occurrence = resolve_path(label, o["json_path"])
    assert occurrence["name"] == o["name"]
    assert str(occurrence["ingredientId"]) == o["ingredient_id"]
    context_path = o["json_path"].rsplit(".forms[", 1)[0] if o["kind"] == "form" else o["json_path"]
    context = resolve_path(label, context_path)
    candidates = []
    for source, values in pilot_queries[key]["candidates"].items():
        for candidate in values:
            evidence = {"source": source, **candidate}
            if source == "pubchem":
                record = pilot_collections[source]["records"][str(candidate["source_id"]) ]
                aliases = record["synonyms"].get("Synonym", [])
                matches = [{"index": i, "value": a} for i, a in enumerate(aliases)
                           if a.casefold() in {o["name"].casefold(), "creatine hydrochloride"}]
                evidence["matching_saved_aliases"] = matches
                evidence["alias_pointer"] = "/records/" + str(candidate["source_id"]) + "/synonyms/Synonym"
            candidates.append(evidence)
    case = {"query_key": key, "origin": o, "context_path": context_path,
            "occurrence": occurrence, "context": context, "candidates": candidates,
            "lookup_status": pilot_queries[key]["source_status"],
            "query_occurrences_not_reviewed": len(occurrences) - 1}
    pilot_evidence.append(case)
    print("\nCASE:", key, "| DSLD", o["dsld_id"], "|", o["json_path"])
    print("Origin file/hash:", o["source_file"], o["source_sha256"])
    print("Occurrence:", json.dumps(occurrence, ensure_ascii=False))
    if context_path != o["json_path"]:
        print("Containing ingredient row:", json.dumps(context, ensure_ascii=False))
    print("Candidates:", json.dumps(candidates, ensure_ascii=False))
    print("Lookup statuses:", case["lookup_status"])
print("\nPilot totals:", len(pilot_evidence), "occurrences;",
      sum(len(c["candidates"]) for c in pilot_evidence), "candidate links;",
      sum(not c["candidates"] for c in pilot_evidence), "occurrences without candidates.")
assert len(pilot_evidence) == 10
assert sum(len(c["candidates"]) for c in pilot_evidence) == 18
print("No mapping decisions applied. Full evidence remains in pilot_evidence.")



CASE: creatine hcl | DSLD 25731 | ingredientRows[0].forms[0]
Origin file/hash: data/raw/dsld/labels/label_25731.json 1383effee1ff97ce9d704f32727a8679bdceade2976f8e2e894eb641e6bc4b00
Occurrence: {"order": 1, "ingredientId": 6225, "prefix": "and", "percent": null, "name": "Creatine HCl", "category": "non-nutrient/non-botanical", "ingredientGroup": "Creatine", "uniiCode": null}
Containing ingredient row: {"order": 1, "ingredientId": 87125, "description": null, "notes": "Creatine (Form: as pH-Sorb(TM) Buffering Technology Creatine Monohydrate, and Creatine HCl) Note: patent-pending ", "quantity": [{"servingSizeOrder": 1, "servingSizeQuantity": 2, "operator": "=", "quantity": 1500, "unit": "mg", "dailyValueTargetGroup": [{"name": "Adults and children 4 or more years of age", "operator": null, "percent": null, "footnote": "Daily Value not established"}], "servingSizeUnit": "Capsule(s)"}], "nestedRows": [], "name": "Creatine", "category": "non-nutrient/non-botanical", "ingredientGroup": "Cre

### 13. Pilot findings — recommendations per target, not approved mappings

The table accounts for all **18 candidate links and two no-candidate occurrences** above. “Support for scoped match” means the inspected evidence supports moving that particular proposal to review; A6 remains pending and no source-wide decision is implied. A scope mismatch is not an instruction to delete the source candidate.

| Origin occurrence | Target(s) inspected | Evidence and qualifier check | Recommended outcome / remaining gate |
|---|---|---|---|
| Creatine HCl, label 25731, `ingredientRows[0].forms[0]` | PubChem CID 134732 | Source explicitly names HCl. Saved aliases include Creatine HCl and Creatine hydrochloride; target IUPAC name, formula and structure explicitly represent hydrochloride. The containing row lists HCl and monohydrate separately. ING-02 has a designation/alias bridge plus compatible form evidence. | **Support for scoped chemical-form match**; A6 pending. Does not establish HCl quantity: 1,500 mg belongs to the parent row containing both forms. |
| Same HCl occurrence | RxNorm 2907 and 1310467 (two links) | Targets are creatine and creatine monohydrate, returned by normalized lookup. Neither describes the explicitly designated hydrochloride form. | **Recommend rejecting exact form identity for both targets**; A4 fails for the specified-form proposal. Parent/form relationships require a separately supported relation; these are not alternate names for HCl. |
| Creatine Monohydrate, label 19397, `ingredientRows[3].forms[0]` | PubChem CID 80116 | Source form explicitly states monohydrate; saved target title and hydrate-containing representation support that designation. | **Support for scoped chemical-form match**, A6 pending. No inferred allocation of the parent quantity. |
| Same monohydrate occurrence | RxNorm 1310467 | Source's explicit form and target's explicit monohydrate designation and PIN type are compatible; PubChem inspection independently supports the form reading, not equivalence of the namespaces. | **Support for scoped terminology match**, A6 pending. Preserve the RxNorm concept and PubChem compound as separate source entities. |
| Magnesium, label 1452, `ingredientRows[0]` | PubChem CID 5462224 | Source is a mineral declaration with a separate citrate form; target structure is `[Mg]`. The literal name match does not establish that the product contains this represented material. | **Needs scope review**: A2/A4 block material identity. A generic nutrient-concept proposal must be recorded separately and must not assert elemental material in the product. |
| Same magnesium occurrence | RxNorm 6574; MeSH D008274 (two links) | Source declaration and vocabulary names support a generic magnesium reading; MeSH definition describes the element. Citrate remains separately explicit in the source. | **Generic-concept proposals only; needs review** of representation level (A2). Do not promote a nutrient declaration to a specified-material identity. |
| Magnesium Citrate, label 1452, `ingredientRows[0].forms[0]` | PubChem CID 6099959 | Source says magnesium citrate; target specifies trimagnesium dicitrate with a particular stoichiometric representation. The inspected source does not specify that ratio or hydrate state. | **Needs review / additional evidence** (A4), not automatic rejection. Matching alias alone cannot fill the missing chemical detail. |
| Same citrate occurrence | RxNorm 52356; MeSH C110422 (two links) | Both retain the magnesium-citrate designation. Saved MeSH candidate has no scope note; RxNorm name/type does not settle detailed material scope. | **Needs definition/scope evidence** (A2–A4) before exact material acceptance. A generic terminology match remains a proposal, not a stoichiometric assertion. |
| Withania somnifera, label 323200, `ingredientRows[0].forms[0]` | PubChem CID 161671 (Withanolide D) | Source form is under botanical Ashwagandha; target is a specified constituent compound. The saved genus scope note mentions withanolides generally, not proof of this particular compound in this label. | **Recommend rejecting exact identity** (A2/A4 mismatch). Do not invent a specific constituent relationship from this candidate hit. |
| Same Withania occurrence | MeSH D032341 (Withania) | Saved scope note describes a genus and names W. somnifera. The label occurrence is a botanical form, not an explicit extracted taxon node. | **Needs origin-role review** (A2). Species-to-genus `origin_narrower_than_target` is plausible only for an explicitly taxonomic subject; botanical material may instead have a source-organism association. |
| Ashwagandha root extract, label 44575, `ingredientRows[0]` | RxNorm 1309279 | Label notes explicitly give root, genus Withania and species somnifera; target names Withania somnifera root extract. This supports the alias, organism, part and generic extraction scope. | **Support for generic root-extract concept match**, A6 pending (ING-05). Specific solvent/ratio/standardized preparation identity remains unknown and must not be inferred. |
| L. acidophilus (NCFM), label 249674, `ingredientRows[0].nestedRows[0].nestedRows[0]` | None in the three saved lookups | Source explicitly preserves the strain qualifier; no strain target was collected. Quantity has NP/0 and provides no identity support. | **Needs enrichment**, not rejection of the ingredient. No species substitute or strain identity approval (ING-07; A1/A3 lack a target). |
| Fish Oil, label 70486, `ingredientRows[3].forms[0]` | MeSH D005395 | Source identifies fish oil as one source form alongside krill oil; MeSH definition describes oils from fish/fish parts. | **Support for generic fish-oil concept match**, A6 pending (ING-08). Exact oil composition unknown; parent 1,530 mg omega-3 amount cannot be assigned to fish oil alone. |
| Same fish-oil occurrence | RxNorm 4419 | Target names fish oils, compatible with source designation; inspected target metadata gives no specific formulation definition. | **Generic-material proposal needs scope review** (A2/A3); do not assert formulation equivalence. |
| Omega-3, label 268232, `ingredientRows[3].nestedRows[0].nestedRows[0]` | PubChem CID 56842239 | Source includes Alpha-Linolenic Acid as its form. Target saved IUPAC/structure describes a three-component representation. Source does not support all target components. | **Do not accept exact material identity**; A4 blocks it. Recommend rejecting the exact represented-material proposal after review; retain any broader-category proposal separately. |
| Same omega-3 occurrence | RxNorm 4301 | Target is omega-3 fatty acids; origin's form is ALA. | **Needs category-versus-material review** (ING-09; A2). Preserve ALA and consider a category relationship only with appropriate subject scope. |
| Probiotic Blend, label 11613, `ingredientRows[0]` | None in the three saved lookups | Four nested species are listed. Notes state 10 billion cfu/g; nested quantities are NP/0. No strain identities or complete defining proportions are supplied. | **Needs target/composition evidence** for a specific blend identity (ING-08; A1/A3/A4). The listed constituents can be preserved without claiming the blend is equivalent to another blend. |

**Gate interpretation:** A1 traceability passed for the 18 returned links after saved hash/locator checks. No-target cases cannot pass target traceability. The table records where A2–A4 have positive support or blockers. A5 was considered against the returned candidates and containing row for each sampled occurrence; no exhaustive search for uncollected alternatives was performed. All A6 approvals remain pending. This pilot records five supported scoped-match recommendations, three recommended exact-form/entity rejections plus one blocked/reject-recommended represented-material proposal, nine candidate links needing scope/definition review, and two no-candidate cases. These categories describe recommendations, not persisted review states or accuracy metrics.


### 14. What the pilot establishes and what remains open

The requirements distinguish several useful outcomes: evidence-supported scoped matches, explicit entity/form mismatches, insufficiently specified materials, and no collected target. They also prevent moving a parent quantity onto a constituent/form. These are reusable behaviors, not special exceptions for the chosen names.

**A remaining specification issue is now visible:** we need an explicit modeling distinction between “this occurrence denotes a generic concept,” “this material belongs to a category,” and “this is the same specified material.” `identity_match` must not silently carry all three meanings. Existing `related_entity` can preserve a reviewed connection provisionally, but typed category/source-organism/material relationships require schema review before implementation. This is a refinement within G1, not a new accepted graph relation.

The pilot does not close G1: generic species without a strain, plant-part-only materials, fully specified mixtures and unknown-scope cases still need dedicated review examples; the other occurrences and query groups have not been assessed semantically. G2–G5 remain pending. The next concrete design decision is to separate generic concept mapping from material identity, then review additional occurrences within each scope. No automatic merging or full-dataset semantic validation is justified by these ten cases.

Raw files, `read.md`, previous assessment artifacts and existing mapping decisions remain unchanged. The notebook code can be rerun to verify and display the saved evidence; the narrative recommendations are explicitly assistant-authored and are not an automated clinical or identity approval system.


### 15. Mapping semantics refinement — concept, category and material

**Specification:** `mapping-semantics-draft-v2`. This is a design refinement prompted by the occurrence-level pilot, not an executed migration or acceptance decision. For new proposals, the explicit relations below replace the overloaded `identity_match` convention in earlier sections. Earlier audits, recommendations and outputs remain historical evidence under their original rule versions.

#### Three different questions

| Proposed relation | Question answered | Allowed subject and target scope | Positive evidence required | What acceptance does not establish |
|---|---|---|---|---|
| `denotes_concept` | What generic concept does this source expression denote? | A source occurrence or scoped identity, interpreted in an explicit semantic role, → a compatible generic terminology concept or reviewed generic IngredientIdentity. | Source wording **and context** support the generic meaning; target definition/representation supports that same concept and level. Apply A1–A6 and relevant ING requirements. | It does not identify the exact material in a product, unite source records, establish composition, or create `sameAs`. |
| `member_of_category` | Does this specified subject belong to this broader class? | A supported material/entity referent → a defined category concept. An occurrence can be the origin only when its referent is sufficiently supported for this classification. | Definitions or source-backed classification support membership, with explicit subject and category boundaries. | It does not make the member identical to the category or every other member. It does not imply a product contains all members of the category. |
| `exact_material_match` | Does this occurrence identify the same specified chemical form, preparation, organism ingredient or mixture type represented by the target? | A source material/form occurrence or reviewed material identity → a compatible specified ChemicalCompound, TerminologyConcept or IngredientIdentity. | All identity-critical qualifiers at that scope agree, with a supported designation/definition bridge and no unresolved contradictions. Apply A1–A6 and the applicable ING row. | It does not identify the same physical batch, prove purity or interchangeability, merge products, allocate quantities, or approve clinical claims. |

**Material type, not physical specimen:** `exact_material_match` concerns the defined ingredient type. A species/strain designation may identify an organism ingredient at that declared level; it does not identify a culture batch, viable dose or manufacturing preparation. Do not extend a species-level decision to a strain-specific identity.

**Generic scope is permitted, specificity is never invented:** a generic root-extract or fish-oil expression can have a supported `denotes_concept` link while exact preparation/composition is unknown. Missing solvent, ratio or strain does not prevent every useful concept link; it prevents conclusions that require that missing detail. Record which qualifiers remain unknown rather than marking them supported.

**Source node type alone is insufficient:** PubChem compounds are not generic categories merely because their titles sound broad. A MeSH or RxNorm entity can represent a generic or specific subject; inspect its definition, role and saved representation. A declared nutrient amount is not proof that the product contains an elemental material.

#### Preserve other relations with narrower meanings

- `origin_broader_than_target` and `origin_narrower_than_target` describe **concept-to-concept scope inclusion**. They are not aliases of member-to-class membership. A botanical-material occurrence is not automatically a taxonomic concept.
- `related_entity` remains a provisional, evidence-backed connection where identity or class inclusion is unsupported. Its rationale must name the connection, such as source organism or constituent; no unspecified connection is accepted. Detailed source-organism/constituent graph predicates remain pending.
- `candidate_unspecified` remains unresolved and cannot be accepted.
- `TopicAssociation` remains independent. A Reddit thread topic or a DSLD search category does not prove `member_of_category`, direct use, or ingredient identity.


### 16. Mapping assertion fields, constraints and transition from v1

#### Required semantics on each proposed mapping

Retain the existing origin, target, evidence, lookup, review and version fields. Add/refine the following design fields:

| Field | Meaning / proposed constraint |
|---|---|
| `relation` | One explicit v2 relation. Every alternative relation to the same target is a separate proposal with its own review history. |
| `origin_role` | Evidence-backed interpretation such as nutrient declaration, ingredient material, form, generic concept expression, taxonomic concept or category expression. Unknown role blocks acceptance where the relation depends on it. |
| `origin_scope`, `target_scope` | ING scope plus relevant qualifiers; preserve source values and unknowns. A node label alone does not determine these values. |
| `interpretation_basis` | Source locators and rationale establishing the role and scope; no automatic projection from a lookup query. |
| `mapping_semantics_version` | `mapping-semantics-draft-v2` for proposals following this refinement. Retain the original version on historical recommendations. |
| `qualifier_checks`, `gate_results` | A1–A6 checks and qualifier evidence, including what the chosen relation does and does not require. |
| `supersedes_decision` | Optional historical decision reference when a reviewed replacement is made. A design revision alone does not rewrite earlier decisions. |

These are additions to **MappingAssertion**, not a requirement to create artificial nodes for every interpretation. `IngredientOccurrence` retains the original path, source role and wording. A new generic canonical node is created only if later reviewed schema/identity rules require it; no IDs are allocated here.

#### Validation and inference boundaries

1. An assertion has exactly one origin, one target, one relation and one explicit origin interpretation. Unresolved alternatives stay separate proposals.
2. Zero or more accepted concept/category links may coexist if each has evidence and compatible scope. Multiple classification links do not imply multiple materials in the product.
3. The existing zero-or-one `RESOLVES_TO` rule applies **per occurrence and declared material scope**, not across all concept/category links. Only an accepted `exact_material_match`, plus a separately established canonical identity, can justify material resolution. An external CID/RxCUI match does not itself create a canonical node. Composite rows retain separate components/forms and are not forced into one material.
4. Generic concept links do not produce material `RESOLVES_TO` edges. If generic canonical concepts are introduced later, their resolution contract must be explicit rather than reusing material resolution silently.
5. `member_of_category` is not symmetric; two members of one category are not identical. `denotes_concept` is not symmetric or transitive. None of these links generates cross-source `sameAs` automatically.
6. Accepted `exact_material_match` links are evidence-backed mapping assertions, not unrestricted equivalence closure. Reusing an identity requires compatible scope, source versions and provenance. Conflicting exact targets block material resolution until reviewed.
7. No relation transfers clinical claims, efficacy ratings, dose, safety, source quality or clinical approval. Retrieval may follow a relation only while retaining its type, subject scope and evidence role.
8. All parent amounts remain attached to their original quantity assertions. Concept/class/form links never allocate a parent amount to a child or form.

#### Transition policy

The earlier `identity_match` value is **legacy and ambiguous**, not invalid historical data. Do not bulk rename it to `exact_material_match`. Re-examine the origin role, target scope and evidence; propose `denotes_concept`, `member_of_category`, `exact_material_match`, a supported scope/related relation, or keep it unresolved. Record a new versioned decision only after review. No existing approved mapping is present in the current project; this policy also supports future imports that may carry such decisions.

The review states remain `unreviewed`, `needs_review`, `accepted` and `rejected`. Acceptance/rejection is relation-specific. A rejected exact-material proposal can coexist with a supported generic concept proposal without erasing either decision. Lookup outcome and clinical review remain separate.

**Schema impact:** this refinement specializes MappingAssertion relation semantics and clarifies `RESOLVES_TO`; it does not finalize the physical graph schema, change raw data or implement migration code. The previous Step 3/5A tables must be read with this v2 refinement for future work.


### 17. Pilot cases expressed using the refined relations

This table translates the earlier recommendations into **proposed relation types**, not new approvals. A6 remains pending for every supported recommendation; no earlier counts or outputs are rewritten.

| Pilot origin → target | Refined proposal | Evidence boundary / remaining work |
|---|---|---|
| Label 25731 Creatine HCl form → PubChem 134732 | `exact_material_match` | Saved HCl/hydrochloride aliases and hydrochloride-specific representation support the named form; review approval pending. Parent total remains unallocated. |
| Same HCl form → RxNorm creatine / creatine monohydrate | Exact-material proposals remain mismatch recommendations | Neither candidate establishes the HCl form. A generic relationship needs its own supported proposal; do not automatically rescue a rejected identity with category membership. |
| Label 19397 Creatine Monohydrate form → PubChem 80116 / RxNorm 1310467 | Separate `exact_material_match` proposals | Explicit monohydrate scope supports each proposal; their shared origin does not merge the two source entities. |
| Label 1452 Magnesium nutrient declaration → compatible generic magnesium concept | `denotes_concept`, subject to target scope review | Identifies the declared nutrient concept, not elemental magnesium material in the product. PubChem `[Mg]` is not automatically substituted for a generic nutrient concept. |
| Same label's Magnesium Citrate form → PubChem 6099959 | `exact_material_match` remains blocked | Source does not settle the target's specific stoichiometric/hydration representation. Generic concept mapping to compatible terminology can be reviewed separately. |
| Same citrate form → RxNorm 52356 / MeSH C110422 | Potential `denotes_concept`; still needs definition/scope evidence | Similar designations do not settle material specification or automatically pass A3. |
| Label 323200 Withania somnifera botanical form → Withanolide D | Exact-material mismatch recommendation | Do not invent a constituent relation from a lookup hit. |
| Same botanical form → MeSH Withania genus | Keep proposed connection unresolved | A taxonomic concept interpretation could support concept inclusion; a botanical material instead needs a source-organism connection. Evidence must establish the origin role first. |
| Label 44575 Ashwagandha root extract → RxNorm 1309279 | `denotes_concept` | Source notes support organism, root and generic extract scope. No exact standardized preparation identity is established. |
| Label 249674 NCFM-qualified occurrence → no target | No mapping assertion with an invented target | Preserve LookupAttempt and strain-qualified occurrence. Enrichment is needed; no automatic species-level replacement. |
| Label 70486 Fish Oil source form → MeSH D005395 | `denotes_concept` | Supported generic fish-oil concept recommendation, not exact composition or allocation of the parent omega-3 total. RxNorm fish oils remains a separate scope-review proposal. |
| Label 268232 Omega-3 row with ALA form → PubChem 56842239 | Exact-material proposal remains blocked/mismatch recommendation | The target's three-component representation cannot be inferred from the ALA-qualified source. |
| Same Omega-3 row → RxNorm omega-3 fatty acids | Potential `denotes_concept` for the generic declaration | A separate **ALA-form → omega-3 category** proposal would use `member_of_category` and require its own evidence and endpoints. Do not reuse the row-to-target assertion as proof of that membership. |
| Label 11613 Probiotic Blend → no target | No exact-material proposal accepted | Preserve the component declarations and unknown strain/proportion details. Shared blend names do not establish formulation identity. |

**Design checks for later implementation (not executed tests):** a generic root-extract mapping must not create material resolution; two category members must not merge; material matching must not allocate parent quantities; a known form with no candidate must remain retrievable with its qualifiers; legacy `identity_match` must not auto-migrate; a category-membership proposal must retain the specific member origin rather than the parent query row.

**Status:** G1 now has explicit relation semantics and a pilot reinterpretation. It remains pending broader occurrence review and implementation validation. The next review should include a well-supported category-membership case and the scope classes not covered by the pilot, while retaining G2–G5 as separate pending work. No graph nodes/edges, accepted mappings or normalization outputs were created.


### 18. Broader occurrence review — testing v2 against saved evidence

This extension examines **20 occurrences across 12 query classes**, covering category membership, species/strain, botanical materials/preparations and mixtures. Nineteen occurrences are new to the prior pilot; Probiotic Blend on label 11613 is deliberately reused to compare with another label. Selection is purposive, generally the two lowest label IDs per selected query or its only occurrence, not statistically representative.

The script produces a **reviewed recommendation table**, not an approved mapping table. It retains all 17 returned candidate links for the selected occurrences, adds two explicitly labeled manual category proposals, and preserves nine no-target occurrences. Category targets are existing saved MeSH records reached through hierarchy evidence, not results fabricated for the original lookup.

`review.json` retains raw occurrence excerpts, target properties/definitions, source hashes, alias locators and the category's RDF evidence. `mapping_review.csv` is a flat table with origin paths, proposed relations, requirements, rationales and evidence-case pointers. The manifest pins input hashes. Recommendations are assistant-authored interpretations; code validates and reproduces them but does not perform general semantic reasoning. All rows retain `needs_review`, A6 pending, and no approval applied.

**Evidence checks:** verify the earlier review/occurrence hashes, three saved terminology collections and every selected label; verify occurrence names/IDs at their paths; require the saved ALA-to-omega-3 hierarchy statement and turmeric-powder synonym; reconcile all table counts; verify the shared-ID/different-components blend comparison; check raw files remain unchanged.


In [1]:
from pathlib import Path
import sys, json
import pandas as pd

scope_project = Path.cwd()
if not (scope_project / "scripts").is_dir():
    scope_project = scope_project.parent
if str(scope_project / "scripts") not in sys.path:
    sys.path.insert(0, str(scope_project / "scripts"))
from review_ingredient_scopes import review as review_ingredient_scopes

scope_output, scope_report = review_ingredient_scopes(scope_project)
scope_mapping_review = pd.DataFrame(scope_report["review_rows"])
scope_case_evidence = {c["case_id"]: c for c in scope_report["cases"]}
print("Saved report:", scope_output.relative_to(scope_project))
print(json.dumps(scope_report["summary"], indent=2))
print("\nComplete recommendation table (not approvals):")
print(scope_mapping_review[["case_id", "label_id", "query", "target", "relation", "recommendation"]].to_string(index=False))
print("\nSource context for all 20 occurrences:")
for case_id, case in scope_case_evidence.items():
    o = case["occurrence"]
    print(case_id, "|", case["origin"]["json_path"], "| notes:", o.get("notes"))
    print("  components:", [x["name"] for x in o.get("nestedRows", [])],
          "forms:", [{"name": x["name"], "percent": x.get("percent")} for x in o.get("forms", [])])
print("\nInspect a full evidence case with scope_case_evidence['case-01']; filter scope_mapping_review by relation or recommendation.")


Saved report: data/processed/identity/ingredient-scope-review-v2-36f4c0d499c594f7
{
  "query_classes": 12,
  "occurrences": 20,
  "saved_candidate_links": 17,
  "manual_category_proposals": 2,
  "no_target_occurrences": 9,
  "review_rows": 28,
  "recommendation_counts": {
    "needs_evidence": 8,
    "supported_for_review": 11,
    "no_target": 9
  },
  "accepted_mappings": 0,
  "merged_entities": 0
}

Complete recommendation table (not approvals):
case_id label_id                                                    query                             target             relation       recommendation
case-01    65671                                     alpha-linolenic acid                pubchem:CID:5280934 exact_material_match       needs_evidence
case-01    65671                                     alpha-linolenic acid                       rxnorm:52071      denotes_concept       needs_evidence
case-01    65671                                     alpha-linolenic acid http://id.nlm.nih.go

### 19. Findings from the broader review

| Cases / source occurrences | Proposed relation and result | Evidence and practical implication |
|---|---|---|
| 01–02: Alpha-Linolenic Acid, labels 65671 / 80083 | `denotes_concept` to MeSH D017962 and separate `member_of_category` proposals to D015525 are supported for review | Both occurrences explicitly name ALA. Saved MeSH states D017962 `broaderDescriptor` D015525. This vocabulary edge stays intact; a separate, reviewed occurrence-to-category proposal uses it as evidence. The second label also explicitly says Omega-3 in its notes. This supplies the category-membership case missing from the first pilot. |
| Same two ALA occurrences → PubChem 5280934 / RxNorm 52071 | Needs evidence for exact stereochemical material scope / target definition | PubChem representation specifies all-Z stereochemistry; source alias evidence supports candidacy but must be assessed at that specificity. RxNorm's shorter linolenic-acid name does not by itself settle the alpha qualifier. Do not strip qualifiers to manufacture agreement. |
| 03–04: Lactobacillus acidophilus, labels 11613 / 215449 | Species-level `denotes_concept` proposals to RxNorm 6205 and MeSH D007779 supported for review | Source full species name and bacterial context agree; MeSH explicitly describes a species. No strain is specified. A useful generic species link is possible without inventing strain, live-cell quantity or batch identity. |
| 05: La-14 Lactobacillus acidophilus, label 177509 | Strain-designation-level `exact_material_match` to RxNorm 2737252 supported for review | Species and explicit LA-14 code match the target designation; word order/case differ. No viability, batch or potency equivalence is inferred. Approval remains pending. |
| 06: Lactobacillus acidophilus LA14, label 54155 | No saved lookup target; possible alias investigation | A similarly named LA-14 record exists elsewhere, but no authoritative LA14↔LA-14 alias/deposit bridge was established in this review. Keep it unresolved; do not silently copy the other query's result. This is a punctuation/alias enrichment candidate, not an approved normalization rule. |
| 07–08: L. acidophilus DDS-1, labels 259959 / 260008 | No saved target; retain both qualified occurrences | Repetition and shared source ingredient ID do not resolve the abbreviated genus or identify a target strain. Do not substitute LA-14, NCFM or the generic species. |
| 09–10: Turmeric root powder, labels 214296 / 219999 | `denotes_concept` to RxNorm 2705361 supported for review | Saved RxNorm **Synonym** explicitly states turmeric root powder; preferred name is Curcuma longa root powder. This is a source-supported alias bridge with compatible part/preparation words, not simply an exact query hit. Preserve source wording; do not rewrite root to rhizome or powder to extract. |
| 11–12: Ashwagandha, labels 13653 / 69019 | Generic concept proposal to MeSH C030693 still needs definition | Source notes specify root, species and 1.5% withanolides. The broad target name and absent projected scope note do not establish a matching preparation. Repeated standardization values are not proof of formulation identity. |
| 13–14: Ashwagandha extract, labels 39970 / 54522 | Exact-material proposal to RxNorm 1876359 remains blocked | Both source notes say root; standardization differs (2.5% versus 4%). Target says whole extract. Its meaning is not established by the saved name properties. Do not infer that whole means whole plant, declare a definite mismatch, or assume the preparations are interchangeable. |
| 15: Organic ashwagandha root powder, label 25775 | No saved target | Preserve powder, root and 0.2% marker information. A root-extract concept from another query does not supply material identity. |
| 16: Creatine Blend, label 3714 | No target; no exact mixture identity | Six nested ingredient declarations, a 5,000 mg parent amount and NP child quantities do not establish proportions. The source lists a botanical component as well; the word Creatine must not collapse the whole blend into a single creatine compound. |
| 17–18: Ashwagandha extract/powder blend, labels 74939 / 74940 | No target; same compared declarations are insufficient for exact formulation identity | Same registry ID, 350 mg total and marker values (0.5% withanolides, 0.8% alkaloids) do not specify extract-to-powder proportions. Source `forms` contain marker compounds; do not automatically reinterpret them as separate bulk formulation components. |
| 19–20: Probiotic Blend, labels 11613 / 32494 | No target; shared registry ID cannot identify a formulation | Both use DSLD ingredient ID **288938**, but list **4 versus 8 immediate components**. Preserve different nested declarations. Notes also differ (10 billion cfu/g versus 500 million CFU); no potency comparison or unit conversion was performed. |

**Counts and limits:** 28 review rows = 17 original candidate links + 2 manual category proposals + 9 no-target occurrences. Eleven proposals are supported for further review, eight require additional evidence, and nine have no target. All remain pending approval. These are recommendation counts, not 28 distinct ingredients, an acceptance rate or an accuracy measurement.

For supported proposals, source/target tracing and the stated scope evidence are available; A6 is still pending. For blocked proposals the specific evidence gap remains recorded. This extension does not exhaustively review all uncollected alternatives, resolve every A1–A6 question across the dataset, or create a clinical approval.


### 20. Consistency check and remaining work

The refined relations now work on positive and incomplete cases: generic species and botanical powder mappings need not invent precise material identity; a supported category-membership proposal preserves its member origin; strain qualifiers survive lookup failure; mixture occurrences retain their component context despite repeated source identifiers.

Three focused issues remain:

1. **Alias evidence:** common case/word-order differences with an explicit matching strain code can support a proposal, while code variants such as LA14/LA-14 need a documented bridge. An enrichment/review policy is still needed before any automatic name normalization.
2. **Scope definitions:** broad or unclear target labels (especially whole extract and broad botanical records) need definition evidence. Missing detail should block only the relation that needs it; it must not erase usable source information.
3. **Complete mixture validation:** these sampled blends do not provide enough defining proportions/identifiers for an exact formulation match. No fully specified positive mixture case was established. That acceptance path remains unvalidated rather than assumed correct.

The shared-ID/different-components finding strengthens the existing rule to separate source ingredient registry references from occurrence and formulation identity; no new entity merge rule is needed. Parent/child amounts and CFU expressions still await the later normalization work.

**Recommended next bounded step:** define product-family versus product-variant boundaries (G2), using the already audited label comparisons. Keep ingredient alias/scope/mixture gaps explicit for later enrichment and review. We have enough tested distinctions to continue the design, but not to approve all ingredient mappings or start automatic deduplication. G3–G5 remain pending.

No raw files, `read.md`, prior audit outputs or mapping approval states were changed. The new artifacts are a versioned review report and recommendation CSV only; this notebook preserves its earlier cells and outputs.


### 21. Product family and variant boundaries — G2 design specification

**Version:** `product-boundaries-draft-v1`. This refines ProductIdentity/ProductVariant in Step 3 and ID-15–17. It defines project identity boundaries, not a claim that the source labels have already been grouped correctly. No labels are merged and no product IDs are allocated.

#### Three distinct levels

| Level | What it identifies | Boundary |
|---|---|---|
| Label source record / snapshot | A DSLD label and a particular captured payload | Always retain native ID, source version and label-scoped assertions. Two labels may describe the same variant without becoming one source record. |
| ProductIdentity (product family) | One evidenced, branded commercial product line | Can group different strengths, formulations, flavors or package sizes **only when commercial-line evidence connects them**. This is not an entire brand, supplement category or set of interchangeable products. |
| ProductVariant | A distinguishable, label-described offering within a family, including formulation/presentation **and package** | A meaningful supported difference in declared formulation, dosage form/delivery, flavor or package creates a distinct variant proposal. Matching variants means agreement at the disclosed-label scope, not proof of identical batches or undisclosed manufacturing details. |

**POC choice:** package size is part of ProductVariant. A 120-capsule and 700-capsule offering can belong to one family but are separate package variants. Keep `difference_dimensions` such as formulation, strength, delivery, flavor and package as attributes; no additional Formulation or Package node is required now. A future schema can separate those dimensions without discarding their evidence.

#### Family acceptance requirements

All of the following must be reviewed before assigning one family:

1. Source labels and their brand/product names are traceable. Brand and commercial-line identity are compatible. Do not infer a manufacturer from brand name; preserve source contact roles separately.
2. There is **positive commercial continuity evidence** beyond name similarity: for example a saved DSLD same-product relationship with compatible brand/line context, or a corroborated publisher/manufacturer product-line identifier. UPC/SKU may support the review but is not a universal family key.
3. Potential conflicts in commercial identity are resolved. A documented rename or ownership change needs its own continuity evidence; equal ingredient lists, a common indication, common brand or similar names alone are insufficient.
4. The reviewer records the family scope, evidence, rationale, time and rule version. Source relationship annotations support grouping review; they do not override conflicting fields or prove variant equivalence.

Differences in strength, flavor, package or formulation do not by themselves disprove family membership. Conversely, no source relationship or shared code forces unrelated product lines into a family. A bundle relationship does not make a bundle and its components one family or variant. Unknown family membership remains label-scoped.

#### Variant acceptance requirements

To propose the **same** variant, establish family membership first and compare all applicable identity-relevant fields:

- Declared dietary ingredients, chemical forms, strains/preparations, nested membership and defining standardization/mixture qualifiers.
- Labeled strengths/amounts **with operators, units and explicit comparable basis**; do not compare numeric values without serving context.
- Disclosed other ingredients and relevant capsule/coating/flavor composition. Missing lists do not prove equivalence; raw spelling/list-order differences do not prove reformulation.
- Dosage form and delivery attributes where stated, including release/coating details that define the offering.
- Flavor and package presentation, including net count/mass/volume and bundle composition when applicable.

Required fields must be available or documented as not applicable. Each comparison records `equivalent`, `different`, `unknown` or `conflicting`, with supporting evidence. `equivalent` cannot be assigned merely because both values are null. Optional metadata absence does not automatically block the whole decision, but unknown identity-critical fields do.

A shared variant requires every applicable dimension to be equivalent, with no unresolved conflict. Equivalence through units or aliases requires the later approved normalization/mapping rules; no transformations run in this step. Identical explicit source declarations may support equivalence at their stated scope without resolving every ingredient to PubChem. This does not prove undisclosed proportions or manufacturing identity.


### 22. Product decision table and conflict handling

| Rule | Observed difference / agreement | Family implication | Variant / label treatment |
|---|---|---|---|
| PV-01 | Same brand/name, no corroborating continuity evidence | Family candidate only | Retain labels; do not merge a repeated-name group. |
| PV-02 | Compatible commercial identity plus reviewed same-product evidence | May support one family | Still compare every applicable variant dimension independently. |
| PV-03 | Different ingredient, form, strain, defined preparation, or disclosed excipient composition | Can remain one evidenced family | If the source difference is supported and not an unresolved extraction/representation conflict, propose distinct formulation variants. If ambiguous, retain unresolved variant assignment. |
| PV-04 | Different strength on the same explicit basis | Can remain one evidenced family | Propose distinct strength/formulation variants, retaining both source values. This is not a medical determination or proof of historical reformulation. |
| PV-05 | Different units or aliases with potentially equivalent meaning | No family decision | Mark comparison pending approved normalization/mapping. Even demonstrated amount equivalence does not settle all other dimensions. |
| PV-06 | Different serving sizes or instructions | No family decision by itself | Compare amount basis separately. A directions-only change may remain a label revision; unequal strength on a comparable basis can indicate a variant difference. If basis cannot be established, mark unknown. |
| PV-07 | Different net package count/mass/volume, flavor or dosage/delivery form | May remain one family with continuity evidence | Distinct package/flavor/presentation variants when the difference is supported. Two presentations of the same net content are not automatically different packages. |
| PV-08 | Different artwork, entry date, market status or administrative version code only | Does not by itself split family | May describe the same variant if all defining dimensions are equivalent. Keep separate labels/snapshots. Do not assume version code or entry date establishes release chronology. |
| PV-09 | Different warnings, target groups, directions, claims or daily-value percentages | Do not infer a different product identity from wording alone | Keep these label-scoped. Assess whether a stated difference reveals a real variant dimension; a warning revision alone need not create a variant, but its content must not be discarded. |
| PV-10 | Same UPC/SKU or source says image difference, but composition/package disagrees | Supporting commercial evidence can remain | Cannot override supported variant differences. Keep conflicting source relationship evidence and actual fields side by side. |
| PV-11 | Required comparison field missing, NP amount, uncertain alias, or contradictory extraction | Family may still be reviewable independently | Variant comparison stays unknown/conflicting. Do not force a match, automatic split, correction or preferred value. |
| PV-12 | Bundle/set relationship | Do not treat relationship as same-family evidence by itself | Preserve bundle context and components. Do not equate bundle amounts/package with a component product. |
| PV-13 | All compared core fields match | Candidate evidence, not family proof | Compare fields outside that core before any variant acceptance: other ingredients, package, delivery, notes and source context. |

**Conflict precedence:** a supported variant-defining difference blocks a same-variant proposal even when names, codes and source relationship annotations agree. If the difference itself is uncertain, use `needs_review`; do not declare a confirmed variant split. Do not infer that one label is erroneous or obsolete because another differs.

**Package comparison:** preserve every `netContents` entry and its meaning. For example, “120 capsules” and “60 servings” can be two descriptions of one package, not quantities to add. Missing an alternate display does not establish a package-size change. Servings-per-container alone is not a safe package key when serving directions differ.

**Grouping constraints:** one accepted variant belongs to one reviewed family in this POC; family membership can be accepted while variant assignment remains unresolved. Ordinary single-offering labels may describe at most one accepted variant at their captured scope. Bundle or multi-offering labels require explicit decomposition and remain unresolved under that constraint until designed. Do not apply pairwise transitive closure: A/B and B/C candidate similarity does not establish A/C equivalence. Check each proposed group against its defining dimensions and contradictions.

**Review record:** retain compared label IDs/snapshots, commercial-continuity evidence, family proposal, per-dimension values/status/evidence, applicable PV rule IDs, difference dimensions, unresolved questions, and separate family/variant review states. A relation to a previously accepted family/variant must retain provenance and decision history. Keep source-native IDs and do not use mutable name/composition hashes as canonical IDs.

**Retrieval consequence:** the family can organize search results, but an answer about dose/composition must select a label or variant context. Never union ingredient amounts, warnings or instructions from all family members into an imaginary product. Deduplicating later for display must preserve label-specific provenance and uncertainty; no retrieval ranking or deduplication is implemented here.


In [1]:
from pathlib import Path
import json, hashlib

product_root = Path.cwd()
if not (product_root / "data/raw/dsld").is_dir():
    product_root = product_root.parent
product_review_dir = product_root / "data/processed/dsld/dsld-ingredient-review-v1-0baa34b90a1fe69e"
product_manifest = json.loads((product_review_dir / "manifest.json").read_text())
product_review_bytes = (product_review_dir / "review.json").read_bytes()
assert hashlib.sha256(product_review_bytes).hexdigest() == product_manifest["files"]["review.json"]["sha256"]
product_review = json.loads(product_review_bytes)
product_boundary_examples = {}
for product_label_id in [25731, 43261, 74939, 74940, 248198, 276213, 7841, 212304]:
    product_meta = product_manifest["sources"][str(product_label_id)]
    product_bytes = (product_root / product_meta["file"]).read_bytes()
    assert hashlib.sha256(product_bytes).hexdigest() == product_meta["sha256"]
    product_label = json.loads(product_bytes)
    product_boundary_examples[str(product_label_id)] = product_label
    print("\nDSLD", product_label_id, product_label["brandName"], "|", product_label["fullName"])
    for field in ["physicalState", "netContents", "upcSku", "productVersionCode", "labelRelationships"]:
        print(field + ":", json.dumps(product_label.get(field), ensure_ascii=False))
print("\nSaved ingredient/serving audit summary (not variant decisions):")
print(json.dumps(product_review["summary"], indent=2))
print("\n8 raw label hashes and the comparison report hash verified. No grouping, unit conversion or normalization applied.")



DSLD 25731 BPI Sports | Creatine Alkaline
physicalState: {"langualCode": "E0159", "langualCodeDescription": "Capsule"}
netContents: [{"order": 1, "quantity": 120, "unit": "Capsule(s)", "display": "120 Capsule(s)"}, {"order": 2, "quantity": 60, "unit": "Serving(s)", "display": "60 Serving(s)"}]
upcSku: "8 51780 00591 0"
productVersionCode: "Rev. 01-001-CPS001 03/13"
labelRelationships: [{"type": "Package size difference, same product", "labelId": 43261}]

DSLD 43261 BPI Sports | Creatine Alkaline
physicalState: {"langualCode": "E0159", "langualCodeDescription": "Capsule"}
netContents: [{"order": 1, "quantity": 120, "unit": "Capsule(s)", "display": "120 Capsule(s)"}]
upcSku: "8 51780 00591 0"
productVersionCode: "Rev. 01-004-CPS001 01/14"
labelRelationships: [{"type": "Package size difference, same product", "labelId": 25731}]

DSLD 74939 Herbal Hills | Ashwagandha
physicalState: {"langualCode": "E0159", "langualCodeDescription": "Capsule"}
netContents: [{"order": 1, "quantity": 120, "u

### 23. Worked DSLD boundary examples — proposed treatments

The executed evidence cell above reads eight saved labels and the earlier ingredient comparison. Full label payloads remain inspectable in `product_boundary_examples`. These examples apply the proposed boundary policy; they are not accepted family/variant assignments.

| Pair | Source evidence | Family proposal | Variant proposal / blocker |
|---|---|---|---|
| Herbal Hills Ashwagandha **74939 / 74940** | Same brand/name; source package-size relationship; ingredient/serving core and disclosed other ingredients match in the saved comparison. Net contents are **120 vs 700 Vegi Capsules**, with different UPCs. | Strong candidate for one family, pending recorded review. | **Separate package variants** under the proposed policy (PV-07). The matching compared core does not make these labels duplicates. No claim of identical undisclosed manufacturing composition. |
| BPI Creatine Alkaline **25731 / 43261** | Same name/brand and UPC; source says package-size difference. Both list **120 capsules**, with the first also listing 60 servings. Parent amount is **1,500 mg vs 1.5 g per two capsules**. Other ingredients include a Malic Acid versus Sodium Citrate difference in the listed buffering blend; other textual differences also exist. | Strong family candidate, pending review. | Unit representations describe equivalent mass, but no conversion is applied here. **Do not accept same variant:** the disclosed other-ingredient difference needs review. The package-size annotation is not corroborated by the explicit capsule counts; preserve the discrepancy instead of declaring a package split. |
| InnovixLabs Triple Strength Omega-3 **248198 / 276213** | Same UPC and **200-capsule** package; source says image difference. Saved quantities use the same one-capsule basis: DHA **288 vs 360 mg**, EPA **552 vs 480 mg**. | Strong family candidate, pending review. | **Separate declared formulation/strength variant proposals** (PV-04/10), subject to source conflict review. The image-difference annotation cannot establish same formulation. No inferred reformulation chronology. |
| Jarrow Magnesium Optimizer **7841 / 212304** | Same commercial name/UPC; image-difference source relationship. Magnesium **100 mg per one vs two tablets**; citrate vs malate form; added Vitamin B6 row in the latter; **100 vs 200 tablets** per package. | Strong family candidate, pending review. | **Separate form/package variant proposals**; serving-basis differences independently block a same-strength conclusion. No per-tablet normalization or claim of clinical equivalence is performed. |

**What this means for all repeated-name groups:** the saved audit has 70 groups covering 206 labels, with 304 pair comparisons. Of those groups, 27 have the same *compared core* and 43 have at least one core difference. These counts are not same-variant/different-variant counts. PV-13 requires reviewing package and other dimensions even for the 27 groups; PV-05/11 prevent treating every raw difference in the other 43 as a genuine formulation difference. The remaining 194 labels are not proven unique products simply because they lack an exact repeated-name partner.

**Status and handoff:** G2 now has explicit family/variant boundaries, acceptance requirements, conflict rules and four evidence-backed worked pairs. Full 400-label product resolution has not run. G1 ingredient evidence gaps remain open; G3 unresolved-reference handling and G4 fingerprint/lifecycle details remain pending, as does G5 validation. Next, define G3's missing-versus-uncollected reference states before implementing normalization or deduplication.

No family/variant IDs were minted, label relationships rewritten, records merged or raw values corrected. `read.md` and previous audit artifacts remain unchanged.


### 24. Unresolved references — G3 design specification

**Version:** `reference-resolution-draft-v1`. This refines ID-03/19/22 and the Step 3 external-reference convention. It concerns links asserted by a saved source (label relationships, vocabulary edges, conversation parents, citations). It does not approve ingredient mappings, validate clinical claims, fetch missing targets or create graph nodes.

**Core distinction:** knowing that a source references something is different from possessing that target's record. An identifier is not its content. A collected record may still lack information needed for a particular task.

#### Separate reference identity, target availability and retrieval attempts

| Dimension | Proposed values | Meaning |
|---|---|---|
| `identifier_state` | `missing`, `invalid`, `ambiguous`, `usable` | Whether the source provides a usable, appropriately scoped target reference. `usable` means resolvable within a known namespace/scope, not that the remote target is known to exist. |
| `target_availability` | `not_assessable`, `not_collected`, `collected` | Availability in a named local dataset snapshot. Missing/invalid/ambiguous identity is `not_assessable`; do not call it a known uncollected target. |
| `resolution_state` | `unresolved`, `candidate`, `resolved`, `conflicting` | Whether the specific reference has been linked to a captured target of the correct identity/type/scope. “Resolved” is referential, not semantic or clinical approval. |
| `content_status` | `not_assessed`, `incomplete_for_task`, `sufficient_for_task` | Assessment of a **collected** target against a named requirement set. Retain required/missing fields and requirement version. This is not universal completeness or a quality rating. |
| Retrieval attempt outcome | `not_attempted`, `retrieved`, `not_found`, `access_denied`, `request_failed`, `incomplete_response` | Record each attempt separately with time, endpoint and available source status. A request failure or access denial does not mean the target does not exist. No attempt outcome is inferred merely from absence in the sample. |

#### General decision table

| Case / rule | Required representation | Permitted use | What must not be inferred |
|---|---|---|---|
| REF-01 — Source reference field is missing | `identifier_state=missing`, `target_availability=not_assessable`, target key null; retain source context and expected role | Keep the source record and the gap; investigate source extraction later | No invented target, no guessed parent from thread order, no assumed citation identity |
| REF-02 — Source identifier is malformed or has uncertain namespace/type | Preserve original value; mark `invalid` or `ambiguous`, availability `not_assessable`; record reason and candidates if available | Candidate investigation only | No silent repair, namespace guess or match to the nearest-looking identifier |
| REF-03 — Usable reference but no matching captured target | `identifier_state=usable`, `target_availability=not_collected`, `resolution_state=unresolved`; retain namespaced/scoped key | Report the source relationship or queue targeted enrichment | No target title, content, composition, author, source quality or clinical support inferred from the key |
| REF-04 — Correct target captured but required details missing | `target_availability=collected`; resolve the reference if identity/type checks pass; `content_status=incomplete_for_task` with named missing fields | Use available fields with the gap retained | Do not call the entire reference unresolved merely because content is incomplete; do not manufacture the missing fields |
| REF-05 — Correct target captured with sufficient required details | `resolution_state=resolved`, `content_status=sufficient_for_task` for the explicit task | Follow the reference with provenance and scope intact | Does not approve clinical claims, ingredient equivalence, endorsement, family/variant grouping or factual accuracy |
| REF-06 — More than one plausible target, or incompatible target type/scope | Retain alternatives and conflict evidence; `candidate` or `conflicting`; no single resolved target | Review evidence or obtain a better identifier | No ranking-only resolution, arbitrary first target or deletion of alternatives |
| REF-07 — Link predicate is inappropriate for the target kind | Preserve source assertion and flag role/type mismatch | Investigate source or extraction issue | No conversion of a hierarchy target, term, tree number, citation label or bundle into an ingredient/product identity |

**State invariants:** a missing/invalid/ambiguous target key cannot yield `resolved`; `resolved` requires a matching captured target and valid role/type checks. Collected content may remain incomplete while the reference is resolved. A known ID mentioned only as an RDF object is not a captured description of that target. An intentionally optional/absent field is not automatically incomplete—for example, missing MeSH tree numbers are not uniformly an error across entity types. Unknowns remain null with a reason, not empty strings that imply no issue.

**No-reference versus missing-reference:** absence of an optional relationship need not create a missing-reference issue. Record a gap only when the source structure or expected role establishes that a reference is expected. A failed ingredient lookup is a LookupAttempt with no candidate, not evidence of a missing external citation or parent record.


### 25. Source-specific reference scope and evidence

| Source / relationship | Target key when supplied | Unresolved handling | Saved audit evidence and scope limits |
|---|---|---|---|
| DSLD `labelRelationships` | (`dsld`, `label`, source `labelId`) | Keep source relationship type and source label snapshot. A known ID outside the collection is `not_collected`; resolve only to a label record, not directly to ProductIdentity/Variant. | The baseline audit reports **916 external relationship occurrences targeting 451 distinct IDs**. These are references outside the 400-label snapshot, not 451 proven nonexistent products. Source “same product” remains evidence, not an accepted grouping. |
| MeSH saved RDF relationships | Exact target URI and original predicate | Preserve the RDF statement and snapshot. Object-only targets remain known references without described content. Match target roles to the predicate (descriptor, concept, term, tree number, etc.). | **791 selected edges lack target descriptions**. This is an edge count, not a count of distinct missing concepts. Do not automatically classify every such target as IngredientIdentity or require every target to have the same fields. |
| Reddit immediate parent | Source full ID (`t1_` comment / `t3_` post) with Reddit namespace | Absent parent ID → REF-01; valid known ID absent locally → REF-03. A known thread post is not automatically the immediate parent. Preserve the source wrapper/ID representation and a documented parsing rule. | **99 comments have no parent ID; 53 reference parents outside the sample**, totaling 152 without saved parents. The first group cannot be enriched by a fabricated parent key. |
| Reddit thread | Source thread/post identifier in its own role | Resolve independently of the parent. Keep `IN_THREAD` and `REPLY_TO` separate. Missing context does not erase the comment or imply deleted content. | **60 comments lack a saved thread; 159 have either parent or thread gaps**. Counts overlap; do not add 152 and 60 as distinct comments. Inherited topic scope still does not establish direct use or endorsement. |
| NatMed local bibliography reference | (monograph snapshot, reference-document context when known, original citation label) | A local citation token can be usable even without a PMID/DOI. Match only within its declared bibliography scope. If that scope is ambiguous, keep a source-local citation occurrence and candidates rather than selecting a publication. | The audit relays **55 upstream-reported missing bibliography citation references**. This is not a fresh count of unique missing studies. Missing bibliography records do not prove the cited work does not exist. |
| NatMed external publication identifier | Namespaced PMID/DOI only when actually supplied or independently established | Preserve the local citation occurrence and any reviewed external mapping separately. A capture with title/author but missing required metadata can be resolved yet incomplete. | A bibliography entry or PubMed metadata match establishes neither excerpt support nor clinical approval; claim support and pending review remain separate. |

**Citation safeguards:** identical citation numbers in different monographs are not shared study identities. A missing external DOI is not a missing identifier when the local citation is traceable. Do not manufacture a StudyReference from title similarity, attach all bibliography entries to every claim, or present unresolved citation tokens as retrieved studies.

**Illustrative state tuples, not executed classifications:**

- Reddit comment with no parent ID: `missing / not_assessable / unresolved / not_assessed`.
- DSLD related label ID known but outside the sample: `usable / not_collected / unresolved / not_assessed`.
- Captured publication with verified ID but missing fields required for a bibliography display: `usable / collected / resolved / incomplete_for_task`.
- MeSH URI appearing only as the object of a relationship: `usable / not_collected / unresolved / not_assessed` for its target description, while the original relationship statement itself remains captured evidence.

These examples express identifier / availability / resolution / content status. The audit counts motivate the policy; this addition does not assign these fields to all raw records or claim that a particular incompleteness task profile has been executed.


### 26. Reference representation, later resolution and acceptance checks

#### Proposed reference record

Keep one provenance-bearing **SourceReference** assertion per source snapshot, reference locator and role/predicate. This is a conceptual addition; it can be a table or an assertion node rather than a new domain entity for every missing object.

Required design fields: source snapshot and exact locator; source relation/predicate; original reference value; identifier namespace/type or local scope; nullable target key; identifier state/reason; availability and dataset-snapshot context; resolution state; nullable matched target record/snapshot; content requirement/task and version; missing required fields; evidence; retrieval attempt references; rule version; decision method, reviewer where applicable, timestamp and decision history. Missing values remain explicit nulls. A source-provided name can be retained as an attributed reference label, never presented as a verified target property.

- A missing-ID reference gets a **source-local reference-occurrence key**, not a fabricated target identity. Different missing references are not merged because their targets are null.
- Several references can share one known target key while retaining their own source evidence and relation roles.
- `SourceReference` differs from ingredient MappingAssertion: it records a source's explicit reference, not our proposed cross-source semantic equivalence. Reuse provenance/review structures where helpful, but keep their meanings distinct.
- Existing source-native identifiers and citation labels are preserved. Parsing/normalization of identifiers must be specified and tested later; this step does not alter their strings.

#### Graph-export policy

For the POC design, retain unresolved references in a reference table/assertion record and leave the resolved domain endpoint null. Do not emit a dangling `REPLY_TO`, clinical citation, or product relationship edge to an invented domain record. A graph implementation may represent the reference assertion itself as a node with the known external key, but it must be visibly typed as a reference—not a fetched Reddit post, study, product or ingredient.

Once the target is captured and identity/type checks pass, the source relationship can link to that target with provenance. The original source reference remains available. Resolution of a DSLD label reference does not approve family membership; resolution of a MeSH edge does not approve ingredient identity; resolution of a citation does not approve claim support.

#### Resolution on later data collection

1. Capture the new target payload with source/version provenance. Match usable keys within the correct namespace/local scope and expected target type; never use bare local citation numbers across documents.
2. Compare the candidate target with the existing reference. Exact native-ID/type resolution may be mechanical when validated; ambiguous/cross-source identity needs review. Record the method and evidence rather than a fictional reviewer.
3. Append a resolution event with old/new state, target snapshot, timestamp and rule version. Preserve earlier “not collected in snapshot X” history; do not rewrite what was available at collection time.
4. Assess task-specific completeness separately. Resolving an ID does not fill missing content. Keep failed/access-denied retrieval attempts separate from local availability.
5. Reassess dependent references when keys, type, scope or captured content change. Source replacements, splits and redirects require explicit lineage; do not silently retarget old evidence. Detailed version/fingerprint rules remain G4 work.
6. Recheck only affected downstream views. Enrichment does not automatically normalize ingredients, merge products, approve claims or change Reddit mention scope.

#### Checks required before implementation

| Check | Expected behavior |
|---|---|
| Two absent Reddit parent IDs | Two source-bound gaps; no shared invented parent. |
| Parent ID absent but thread ID present | Thread can resolve; immediate parent remains unknown. |
| Known external label ID later collected | Append resolution with captured target evidence; preserve prior missing-capture history and grouping uncertainty. |
| Same citation number in two monographs | Separate locally scoped references, no automatic shared publication. |
| Captured target lacks a task-required field | Reference can resolve while content remains incomplete. |
| Retrieval fails or is denied | Keep attempt failure; do not mark target nonexistent. |
| Object-only MeSH URI or target with incompatible role | Preserve RDF evidence; no fabricated description or coerced entity type. |
| Two plausible targets or an ID collision | Preserve conflict/candidates; no arbitrary resolved endpoint. |
| Optional relationship genuinely absent | Do not create a fictitious missing-reference issue. |

These are **design checks, not executed tests**. G3 is now specified, pending implementation and record-level validation. Saved audit findings remain unchanged. G4 fingerprint/lifecycle rules are the next bounded design step; normalization and deduplication are still pending. No target content was downloaded, no reference-resolution decisions were applied, and raw files, `read.md` and prior assessment artifacts remain unchanged.


### 27. Fingerprints and version history — G4 design specification

**Version:** `version-history-draft-v1`. This specifies how future imports and reruns preserve evidence without duplicating logical source records. It refines SourceRecord/SourceSnapshot, Step 5A identity-key conventions, ID-02/21 and G3 reference resolution. It is a design specification: no historical files are rewritten, versions migrated, hashes substituted, nodes created or training jobs run here.

#### Identity, content version and capture are separate

| Object | Proposed key / fingerprint | Meaning and boundary |
|---|---|---|
| SourceRecord | Source namespace + native entity type + native ID, using the earlier identity rules | Stable source identity across captures. Matching content under different native IDs does not merge records. Missing identifiers retain source-local reference/occurrence handling. |
| RawArtifact | SHA-256 of the **exact saved bytes**; record byte length, media type and artifact path | Integrity of the retained file/response. Whitespace, line endings or reordering can change this hash. Keep the original bytes; never normalize the raw file to make hashes agree. |
| SourceSnapshot | Typed tuple of source-record key, representation kind, fingerprint-profile version and payload digest | Immutable version of that source record's retained representation. Hash equality is meaningful only within the same source identity, representation kind and profile. It is not semantic ingredient/product equivalence. |
| CaptureEvent | Persisted event ID associated with collection/import run and request/member locator | When/how we observed the record. A real new download/import observation is a new event even if its content repeats; rerunning processing on the same saved run does not invent a new capture. |
| DatasetSnapshot | Digest of a deterministically ordered manifest of selected artifact/snapshot references plus scope/version metadata | Records which evidence a particular audit/build used. Preserve counts, query/selection policy, collection completeness and import commit. Absence from a limited snapshot is not deletion at the source. |
| DerivedArtifact / build | Fingerprint of declared inputs plus transformation/configuration/schema/rule/code versions | Distinguishes preprocessing, mapping, chunking, export or embedding outputs without pretending the source changed. Keep output file hashes and dependency manifests. |
| ReviewDecision / resolution event | Persisted event ID linked to exact evidence versions, relation/scope and rule version | Append decisions and supersession/lineage events; never overwrite historical approval or silently point it at different evidence. |

Keys are stored as unambiguous typed fields/serialized tuples, not delimiter-concatenated strings. Hash collisions or key reuse with incompatible content/type are integrity conflicts, not permission to merge. Preserve source-native IDs even when local canonical identities are later introduced.

#### Fingerprint profiles

**Profile `raw-bytes-v1`:** SHA-256 of exact retained bytes. Use for HTML, PDF, text, opaque responses and any representation without a validated structured profile. RDF response bytes may change from statement ordering alone; do not claim graph-semantic equivalence from this profile. Treat the new representation conservatively and compare its content separately.

**Proposed profile `json-lossless-tree-v1`:** for a single JSON record, hash a deterministic, typed tree that preserves all fields. Parse without losing number tokens; reject duplicate object keys or invalid JSON rather than silently choosing a value. Encode the tree in UTF-8 JSON with no formatting whitespace:

- object → `["object", [[key, encoded_value], ...]]`, sorting keys by decoded Unicode code-point order;
- array → `["array", [encoded_item, ...]]`, preserving order and multiplicity;
- string → `["string", decoded_value]`, with no trimming, case folding or Unicode normalization;
- number → `["number", original_number_token]`, preserving distinctions such as `1`, `1.0` and exponent spelling;
- boolean → `["boolean", true_or_false]`; null → `["null"]`.

Use deterministic JSON string escaping: escape quote/backslash and control characters; emit other Unicode characters directly; reject invalid Unicode scalar sequences. Hash the resulting bytes. Object-key order/insignificant JSON whitespace and equivalent JSON string escaping may then compare equal, but arrays, strings, numeric tokens, absent fields and nulls remain distinguishable. This is representation comparison, not unit/ingredient normalization. The implementation must pass the fixtures in section 30 before this profile is used to allocate snapshots.

**Default scope is the full retained record.** Include identifiers, text, notes, amounts, forms, qualifiers, references, status and source-provided update metadata. No clinical field or review flag is silently excluded. Separate collector run time, local path and request timing into CaptureEvent when they are already an external envelope. If capture metadata is embedded in a retained record, include it conservatively until an explicit, versioned source-specific extraction profile is approved; do not infer which fields are safe to discard.

For records inside JSONL/collections, preserve the containing raw-artifact hash and exact source locator as well as the record fingerprint. Reordering records can change the file hash while record content stays equal; offsets must still refer to the correct capture artifact. A text-only hash remains a useful subordinate check but cannot identify a snapshot containing references, qualifiers or other metadata.

**Source representations stay distinct:** NatMed HTML, imported record and upstream-derived claim are separate provenance-bearing representations. An unchanged monograph with a changed upstream claim creates a new claim artifact/version, not a claim that the publisher changed its monograph. DSLD labels, terminology responses and raw RDF predicates retain their own scopes. Keep the source snapshot provenance even when several representations appear equivalent.


### 28. Capture, update and lifecycle rules

| Rule | Situation | Required behavior |
|---|---|---|
| VER-01 | First observation of a source ID | Retain the artifact; create source identity, snapshot and capture references after integrity/type checks. Do not infer a canonical ingredient/product. |
| VER-02 | Same ID, same representation/profile and payload digest on a new observation | Reuse the logical SourceSnapshot; append CaptureEvent with source/run/version provenance. Physical raw downloads may remain separately archived. |
| VER-03 | Rerun of processing on the same saved inputs/configuration | Reuse/verify the deterministic derived build key and output hashes. Do not create a new source capture or duplicate graph IDs. Unexpected different output is a reproducibility conflict; preserve/flag it rather than overwrite. |
| VER-04 | Same ID but retained payload changes | Create a new immutable SourceSnapshot with the changed fields recorded. Keep old snapshot and assertions. Determine downstream impact from evidence dependencies, not from hash difference alone. |
| VER-05 | Byte formatting changes but validated structured record fingerprint is unchanged | Keep distinct raw-artifact/capture provenance while reusing logical structured content. Do not repoint an old byte/text locator to the new artifact without checking it. |
| VER-06 | Hash-profile, extraction, schema or transformation version changes | Produce a new profile/build identity and explicit migration/comparison provenance. Do not compare digests across profiles as if they establish equality. Historical IDs/decisions remain valid for their original profile. |
| VER-07 | A new snapshot omits an earlier record | Record dataset membership difference. Do not label the missing record deleted, retired or off-market without source evidence. Partial/failed collections cannot establish source-wide absence. |
| VER-08 | Source explicitly retires/replaces/redirects an ID | Store the source lifecycle assertion, dates and evidence. New target ID remains distinct until reviewed; do not redirect old ingredient, product or clinical assertions silently. |
| VER-09 | Local canonical identity is split or merged after review | Retain original IDs and explicit lineage/supersession decisions. Reassess affected mappings/exports. Do not rewrite raw source IDs or erase historical decisions. |
| VER-10 | A missing-reference target is newly captured | Apply G3 checks; append reference-resolution event tied to the target snapshot. Preserve prior not-collected history and independent semantic/clinical review state. |
| VER-11 | A repeated ID now has incompatible source entity type or unexplained identity | Record an identifier collision/conflict; block automatic continuity and investigate. New content does not prove ordinary version succession. |

#### Time and current-view policy

Preserve separately: source-issued revision/update time if supplied; source release/version; observed capture time; import time; processing time; and decision time. Unknown source times remain null. DSLD entry dates and increasing IDs do not prove product release or reformulation chronology. Import order does not prove event order.

A “current” local view means the **latest eligible observed snapshot within the configured source/representation scope**, not guaranteed current truth. Eligibility requires integrity/completeness checks for the intended use; a failed or partial capture must not silently replace a usable one. When source version order or simultaneous competing captures is ambiguous, expose the conflict instead of choosing by hash or filename. Record why a snapshot was selected. Future full refreshes and partial enrichments must declare their collection scope separately.

Graph construction should reconcile stable IDs against a declared build snapshot. Repeated execution of the same build must not append duplicate nodes/edges. Source occurrence/claim IDs remain version-bound; a current view can select active versions while historical snapshots remain queryable. Physical graph import/update mechanics are a later implementation choice.


### 29. Change-impact and re-review policy

Each derived assertion/mapping must declare its source snapshots, evidence locators, relevant field dependencies and rule/model/configuration versions. If dependencies are not recorded precisely, conservatively mark the affected record's dependent decisions for review/rebuild rather than assuming they are unaffected. Only the latest-view use is blocked; original decisions remain historical records for their original evidence.

| Changed information | Data/graph action | Review implications |
|---|---|---|
| Only capture/import observation metadata; retained content unchanged | Add observation provenance; reuse content-derived artifacts where dependencies match | No semantic or clinical re-review solely because the capture date changed. Preserve new release metadata as context. |
| Raw formatting or ordering only | Verify structured equivalence where supported; refresh/rebind affected locators and raw provenance | No automatic clinical re-review for formatting alone. Reuse a conclusion only via recorded equivalence/rebinding checks, not a silent evidence substitution. |
| Ingredient name, chemical form, strain, botanical preparation or defining mixture detail | New source snapshot; regenerate affected occurrences and mapping candidates/views | Re-review affected concept/material/category proposals. A past acceptance does not automatically apply to changed scope. |
| Labeled quantity, serving basis, dietary/other ingredient composition, dosage form or package | Preserve new label assertions; update affected product comparisons | Re-review relevant formulation/variant decisions and quantity transformations. Family membership is re-reviewed only when its own evidence changes. |
| Directions, precautions, warnings, population, clinical excerpt, rating, dose context or interaction partner | Preserve both versions; regenerate affected passages/assertions and retrieval content | Revalidate source support and affected clinical applicability/review decisions. Do not equate source updates with clinical approval. |
| Parent/thread ID, RDF relationship, citation/reference label or target metadata | Rebuild affected SourceReference/relationship assertions; apply G3 | Recheck references and claims depending on changed citation/support context; unrelated ingredient identities need not be rerun merely because a citation changed. |
| Target definition, suppression/retirement status, source scope or source identity | Preserve target history; mark dependent mappings/relationships affected | Re-review mapping applicability. A status change alone does not authorize a replacement identity. |
| Reviewer changes a decision without source change | Append review event; update permitted downstream views | No new source snapshot is required. Keep clinical review, identity review and reference resolution distinct. |
| Code/config/schema, normalization rules or chunking changes | New derived build and affected outputs; source snapshots unchanged | Validate transformation behavior; recheck decisions that depend on changed extraction/locators or semantics. |
| Unrelated field changes outside declared dependencies | New snapshot if retained payload changed; reuse unaffected results only after a documented dependency/equivalence check | Avoid blanket reapproval or retraining; retain the evidence binding/lineage record for the reuse. |

**Review validity:** the decision itself remains immutable with its original accepted/rejected/pending state. Track applicability to the new snapshot separately (for example `not_assessed`, `revalidated`, `needs_review`, `not_applicable`). A source change must not rewrite a formerly accepted historical decision as if it had never been approved. New-snapshot acceptance requires a new event, including the validation method and evidence; do not fabricate a human reviewer for mechanical checks.

#### Embeddings and training are separate dependencies

- **KG/Neo4j:** add or update the affected graph view/build. This is data maintenance, not model training.
- **Text embeddings:** regenerate only when the actual embedding input or model/configuration changes; preserve model/version, input hash and chunk provenance. Unchanged vectors may be reused with validated new metadata bindings. Refresh indexes/filter metadata and remove superseded entries from the *current* retrieval view while preserving their lineage.
- **R-GCN/GNN, if later adopted:** record training/inference graph snapshot, feature schema, weights/checkpoint and node-ID mapping. Changed topology/features may require recomputation or retraining depending on the architecture and evaluation. New nodes must never receive invented old embedding rows; do not promise incremental inference before it is supported and tested.
- **Conversational LLM using retrieval:** newly indexed evidence does not by itself require LLM retraining. Fine-tuning, if ever introduced, is a separate evaluated workflow.

A hash change is an invalidation signal to inspect dependencies, **not a command to retrain every model**. No embedding refresh or training runs are triggered by this design step.


### 30. G4 validation fixtures and handoff

These are requirements for the future implementation, **not tests executed in this notebook**.

| Fixture | Expected result |
|---|---|
| Same source payload in two new captures | One logical content snapshot under one profile; two capture events. |
| Same archived run processed twice | Same build identity and verified outputs; no new capture or duplicate graph entities. |
| Same payload under different source IDs | Distinct SourceRecords; no semantic merge from shared hash. |
| JSON object key order or insignificant whitespace changes | Different byte hashes may occur; `json-lossless-tree-v1` content fingerprint remains equal. |
| Array order/multiplicity, text whitespace/case, absent-versus-null, or `1` versus `1.0` changes | Structured fingerprint changes; later semantic normalization is a separate decision. |
| Duplicate JSON keys, invalid Unicode scalars or incompatible ID/type | Integrity failure/conflict; do not silently repair or emit a trusted snapshot. |
| Text unchanged, references/qualifiers/review metadata changed | Full retained-record fingerprint changes; text-only hash cannot mask it. |
| Embedded capture metadata changes | Conservative full-record version change unless a validated source-specific envelope profile explicitly separates it. |
| Old target missing from a partial refresh | Dataset membership change only; no inferred retirement/deletion. |
| New quantity/form but same commercial name | New snapshot and targeted mapping/variant re-review; old assertions remain historical. |
| Formatting change moves evidence offsets | Validate/rebind the new locator; old source span remains attached to old artifact. |
| New model, chunking or fingerprint-profile version | New derived/profile identity; no false equality based on comparing incompatible digests. |
| Same inputs/config but different derived output | Reproducibility conflict, not silent replacement of previous results. |
| Previously unknown target is captured | Versioned reference-resolution event; no automatic ingredient or clinical approval. |

**G4 status:** fingerprint scopes, capture/version distinctions, lifecycle rules and downstream impact are now specified as a draft. No migrations, fingerprint-profile implementation, lifecycle tests, model updates or source refreshes have been performed. Existing raw hashes and audit manifests retain their current meanings; the structured fingerprint proposal does not retroactively change them.

**Next:** define normalization rules, followed by deduplication rules, using the established identity and provenance boundaries. Keep unresolved G1 evidence gaps and G2/G3 implementation validation explicit. G5 end-to-end validation remains pending and should gate acceptance/merging during implementation; completing design documentation alone is not a successful preprocessing run.

Raw files, `read.md`, prior audit/review artifacts and existing notebook outputs are preserved. The update is limited to this design notebook.


## Step 5B — Normalization rules (design specification only)

**Version:** `normalization-draft-v1`. This defines future derived representations for all six datasets. No normalization code is executed, processed data rewritten, identity approved or duplicate removed in this step. Illustrative results below are design examples, not generated normalized records.

**Purpose:** make values comparable and searchable without changing their meaning or making them more specific than the evidence. Identity review determines what an expression refers to; normalization determines how supported information is represented. Neither operation establishes clinical correctness.

### B1. Common transformation contract

Every derived value retains: source system, source record/snapshot, exact locator and original value/type; destination field; normalized value/type or explicit null; rule ID/version; transformation kind; parameters/conversion factor where applicable; execution/build provenance; dependencies on other fields or reviewed mappings; outcome/reason; and precision/qualifier information when applicable.

| Transformation kind | Permitted purpose | Boundary |
|---|---|---|
| `representation` | Approved field-specific formatting, type parsing or alias representation | Does not approve entity equivalence or overwrite original text. |
| `unit_conversion` | Explicit, same-dimension conversion under B3 | Preserve operator, amount meaning, serving basis and original precision. |
| `context_derivation` | Optional calculation supported by explicit compatible source fields | Store separately from the stated value with every dependency and assumption; blocked when context is incomplete. |
| `reviewed_mapping` | Apply a separately reviewed identifier/alias/status mapping at its approved scope | Reference its decision/version; do not implement semantic inference as string cleaning. |

Use separate dimensions: `normalization_outcome` = `unchanged`, `normalized`, `blocked`, `not_applicable`; `value_state` = `present`, `missing`, `explicit_unknown`, `invalid`, `conflicting` as supported; and `review_required` with reason codes. Zero is a present value unless a field-specific source convention establishes otherwise. Empty, absent, null, placeholder and contradictory values are not interchangeable.

If two fields disagree, retain both representations and the conflict; no field wins merely because it is numeric or structured. Do not manufacture a single reconciled value. A supported independent transformation can remain available while dependent comparisons/calculations are blocked.

**Implementation invariants:** deterministic output for the same source snapshot and rule/config versions; no silent rounding; no dropping rows/duplicates; no automatic identity acceptance; no schema/rule version changes hidden under an existing artifact key. Preserve evidence offsets against original text. G4 governs new builds and re-review after rule changes.


### B2. Names, identifiers, text, dates and source statuses

| Rule | Derived representation | Preserve / block conditions |
|---|---|---|
| N-T01 — Original text and display | Preserve original names, excerpts, notes, warnings and Reddit text verbatim. A display-only cleaned value may trim surrounding whitespace. | Never rewrite the source evidence or recompute its offsets against cleaned text. Text cleanup does not alter quotations or claim support. |
| N-T02 — Search name key | For explicitly designated name-search fields only: Unicode NFC, trim, collapse whitespace runs to one space, then Unicode casefold. Store a separate versioned search key. | Do not use as an entity key, deduplication key or preferred name. Retain punctuation, symbols, digits, Greek letters, hyphens and qualifiers; no blanket NFKC, stemming or stopword removal. Whole claim/Reddit text is not automatically processed by this name rule. |
| N-T03 — Qualifiers / aliases | Preserve chemical forms, hydrate/salt terms, botanical parts, preparation type, strain codes, brand/trademark terms and percentages. A separately supported alias may populate a candidate-search expansion or reviewed mapping. | Do not turn LA14 into LA-14, expand a genus abbreviation, replace HCl with hydrochloride, or drop organic/strain/form terms as an identity-changing cleanup rule. Record the evidence and intended use of an alias. |
| N-I01 — Identifier representation | Retain original type/string and a separate namespaced key. Convert an actual source integer to a decimal string only under the field contract. Use documented parsing for Reddit wrappers/full IDs. | Do not strip leading zeros from string IDs, coerce arbitrary numeric text, remove `t1_`/`t3_`, or derive chronology from numbers. Preserve UPC/SKU as strings; punctuation-compacted comparison keys remain candidates, not universal identifiers. |
| N-I02 — URI and citation scope | Preserve full MeSH URI and entity role; preserve local bibliography tokens with monograph/document scope. | No global HTTP→HTTPS rewriting, URI casefolding, DOI/PMID guessing or cross-monograph citation-number merging. Field-specific identifier alias rules require evidence and validation. |
| N-I03 — Placeholder codes | Interpret only established source/field-specific placeholders; keep original value and reason. | DSLD UNII values such as `0`/`1` cannot be promoted to valid UNII keys. This does not make every numerical zero or `1` in other fields missing. Unknown placeholder meaning remains unresolved. |
| N-D01 — Dates and timestamps | Parse only supported, unambiguous formats into typed date or offset-aware timestamp fields. Convert an offset-aware timestamp to UTC for comparison while preserving the original value/offset and date role. | Preserve date-only/month/year precision; do not invent midnight or a timezone. Ambiguous dates stay unresolved. Source update, entry, capture, import and review times remain different fields; never substitute capture time for a missing source date. |
| N-S01 — Source status mapping | Keep source-native status, source namespace and a versioned interpretation. Preserve actual booleans/nulls without truthiness coercion. | DSLD off-market is source label status; RxNorm validity/suppression is vocabulary status; neither is an identity or clinical approval. Unknown is not false. |
| N-S02 — Lookup outcomes | Distinguish returned candidates, explicit no-result, failed request and incomplete response using the recorded response/method. | A missing result array alone cannot establish no-match. No-match is not rejected identity; exact/normalized lookup method is not accepted identity. |
| N-S03 — Review and evidence roles | Preserve NatMed pending clinical review, local identity review, reference resolution and Reddit `user_report`/mention scope separately. | Do not map all statuses into one approved flag. Reddit thread context remains inherited context, not a direct mention or report of use. |
| N-T04 — Clinical and user-report context | Preserve original indication, population, formulation, interaction partner, citation and rating wording; extract only under a separately validated field rule. | No blanket synonym replacement, spelling repair or common rating scale across source-specific concepts. Preserve negation, uncertainty and source qualification. An unquantified dose statement stays text with unknown structured quantity. |

Search keys can collapse different originals; store the full original-to-key mapping and collisions. Do not erase variants from queries or reuse a query-level identity decision on every occurrence. Normalized terminology labels remain separate from source CID/RxCUI/MeSH identities.


### B3. Units, quantities and serving context

#### Unit vocabulary and conversion policy

| Rule | Permitted representation / illustrative conversion | Conditions and exclusions |
|---|---|---|
| N-U01 — Plain mass | Project symbols `g`, `mg`, `mcg`. Explicit mass aliases: `Gram(s)` / `grams` → `g`; `ug`, `µg`, `μg` → `mcg`. Preserve original spelling. | Apply only to a validated mass field. Unit tokens are case-sensitive; no universal casefolding of units. Alias table is finite/versioned; unknown tokens stay unresolved. |
| N-U02 — Mass comparison value | Use exact decimal factors: g→mg ×1000; mcg→mg ×0.001. Store an optional comparison value in `mg` separately from the normalized source-unit value. | Same amount meaning/ingredient scope and retained basis are required for equivalence comparison. Use decimal/rational arithmetic, not binary-float equality or ad hoc tolerances. Do not infer extra measured precision from added zeros. |
| N-U03 — Volume | Preserve `mL`; explicit `L`→`mL` ×1000 may be used if later observed in a validated volume field. | No mass↔volume conversion without supported density/context. No tsp/drop/scoop→mL conversion from a generic assumption; device, locale and source measure can matter. |
| N-U04 — Qualified mass / activity | Preserve `mcg DFE`, `mcg RAE`, `mg NE`, IU and enzyme-activity units as distinct qualified measures. | No generic conversion to plain mass, no cross-activity-unit equivalence and no vitamin/form-specific factor guessed from a supplement name. Such conversions require separately sourced and reviewed rules beyond this POC default. |
| N-U05 — CFU and concentration | Preserve CFU/count expressions, per-g basis, notation and stated context; parse only explicit unambiguous expressions under a future validated parser. | CFU, CFU/g and CFU/serving are not interchangeable. Do not infer viability, duration/stability, concentration basis or component counts. Parent count never populates every strain. |
| N-U06 — Energy and unfamiliar units | Retain original `Kcal`, `Calorie(s)`, `{Calories}`, enzyme-unit tokens and other unfamiliar tokens with their source context. | No global energy/unit alias conversion until that source field's meaning is verified. An unsupported unit blocks arithmetic, not retention or retrieval. |
| N-U07 — Serving unit display | Finite basic display aliases may produce `capsule`, `tablet`, `softgel`, `caplet` from corresponding explicit plural source tokens; retain the full source descriptor in a separate field. | Do not collapse softgel into capsule, erase vegetarian/coating/trademark qualifiers, or treat gummies, scoops, drops and tablets as interchangeable count units. Broad display categories cannot determine product identity or serving equivalence. |

#### Quantity and basis rules

| Rule | Required behavior |
|---|---|
| N-Q01 — Numeric parsing | Preserve source numeric representation; parse only finite, unambiguous numbers with decimal precision retained. Blank is not zero. Unsupported separators, free text, locale or scientific notation must be handled by a documented parser or remain blocked. |
| N-Q02 — Zero and NP | For audited DSLD `quantity=0, unit=NP`, retain 0/NP as source evidence, set usable normalized amount null with `unsupported_source_amount`, and block arithmetic. Do not interpret it as absence of ingredient. Named-unit zeros remain explicit zeros with a review flag; do not automatically turn them into missing values or quantities above zero. |
| N-Q03 — Operators and ranges | Preserve `=`, `<`, `<=`, `>`, `>=`; keep interval endpoints, inclusivity and unknown bounds. A positive unit-conversion factor scales the bound and preserves the operator. Never replace `<1` with `1`, a range with its midpoint, or a missing bound with the known bound. |
| N-Q04 — Source amount basis | Retain quantity's own `servingSizeOrder`, `servingSizeQuantity`, unit and context separately from label serving definitions and daily directions. Resolve links without replacing their values. Missing/zero/nonpositive/ambiguous denominator blocks basis calculations. |
| N-Q05 — Amount equivalence | Compare only compatible ingredient/amount role, unit dimension, operator/range semantics and basis. Equal normalized numbers alone are insufficient. Named-unit zero review flags, conflicting fields or unknown critical context block an accepted equivalence. |
| N-Q06 — Optional per-unit derivation | Only when the source explicitly supplies a positive exact count basis and applicability to identical dosage units is supported, a future derived comparison may divide the amount by that count. Store as a separate contextual calculation with dependencies; do not overwrite the serving amount or imply that a tablet/capsule can be split. Mixed colored-tablet regimens, ranges and unidentified units stay blocked. |
| N-Q07 — Daily totals and serving ranges | Do not reinterpret a quantity basis as daily intake merely because it fits a computed daily range. Daily totals require explicit linked regimen context; incomplete bounds remain incomplete. Record calculations separately if later authorized and validated. |
| N-Q08 — Parent, form and component amounts | Preserve the hierarchical source location. No allocation of parent totals to each form/component and no sum of parent+children unless a validated non-overlapping composition relationship supports it. Keep nutrient declaration, supplying-material mass and blend total distinct. |
| N-Q09 — Percentage and standardization | Keep percentage value plus its subject, qualifier and basis when stated. Do not equate a marker percentage with a mixing proportion, mass amount or percent daily value. Unknown basis stays unknown; no inferred conversions. |
| N-Q10 — Numeric/text contradictions | Retain numeric field and text separately with evidence and conflict reason; do not choose a correction or silently alter decimal placement. Block calculations/comparisons that depend on resolving the contradiction. |
| N-Q11 — Net contents | Preserve count, mass, volume and servings entries separately by role. Do not add alternative descriptions of one package or derive package identity from servings alone. Unit normalization cannot override the G2 package rules. |

No clinical dose recommendation is calculated by these rules. Computed comparison quantities describe source assertions, not suitability or safety for a person.


### B4. Applying the rules to audit findings — expected treatment

| Saved finding / example | Proposed normalization treatment | What remains unresolved |
|---|---|---|
| BPI 25731 / 43261: 1,500 mg vs 1.5 Gram(s), both per two capsules | N-U01/02 would yield **1,500 mg per two capsules** for both comparison representations, retaining originals. | Equivalent mass representation does not resolve other-ingredient differences or permit product/variant merging. No conversion is executed here. |
| Jarrow 7841 / 212304: 100 mg per one vs two tablets | Retain each amount and distinct count basis (N-Q04/05). Optional per-unit comparison needs N-Q06 validation. | Do not call the strengths equal because both source numbers are 100; form/package differences remain. |
| 705 NP/zero entries on 74 DSLD labels | Keep raw 0/NP; normalized usable quantity null and reason recorded (N-Q02). | No zero-dose, no absent ingredient, no arithmetic. |
| 32 zero entries with named units | Keep zero and source unit with a review flag (N-Q02). | Do not assume these share NP semantics or allow automatic equivalence before review. |
| 24 inequality quantities | Preserve operator and bound (N-Q03); convert bound only if the unit rule applies. | Never treat a bound as an exact amount. |
| 8 missing quantity-serving bases; 35 incomplete daily-serving bounds | Retain available values and missing-state reasons (N-Q04/07). | No imputed serving size, missing upper/lower bound or derived daily total. |
| 67 quantity bases outside serving range; 52 compatible with possible daily ranges | Keep quantity basis, serving range and audit flag separately. | Compatibility with a daily range does not prove the quantity is a daily dose. |
| BetterYou 257287: structured 52 mL versus note 0.52 mL | Preserve both values and conflict (N-Q10). | No decimal correction; dependent calculation stays blocked. |
| Creatine HCl / Creatine hydrochloride | Preserve separate original strings; ordinary search-key normalization does not equate them. Use the saved PubChem alias evidence only in a separate versioned candidate/mapping record. | No automatic canonical ingredient assignment from text cleaning. |
| LA14 / LA-14 | Preserve punctuation and code distinctions in search keys. | Alias bridge remains unestablished; no auto-normalization to the other strain code. |
| Probiotic notes: 10 billion CFU/g vs 500 million CFU | Preserve magnitude and denominator wording; keep source scope. | Not comparable without compatible amount/serving context; do not infer CFU/serving. |
| Botanical marker percentages in a `forms` array | Preserve source path, marker and percent; record its standardization role when supported. | Not automatically separate bulk ingredients or extract-to-powder mixing ratios. |
| Reddit inherited topic / NatMed pending claims | Keep normalized role/status fields separate from source-native fields and from identity states. | No upgrade to direct use, clinical evidence or clinical approval. |

**Dataset coverage:** DSLD uses quantity/basis, hierarchy, identifier, text/date/status rules. PubChem, RxNorm and MeSH use source-scoped identifier/search/status rules while retaining chemical properties, vocabulary types and RDF predicates. NatMed retains claim context, support spans, local citations and review state; missing dose fields remain unknown. Reddit retains post/comment/thread identity, text, timestamps and mention scope. No cross-source shared rating scale or universal ingredient identity is created.

**Precision and unknowns:** retain source lexical precision even when arithmetic produces a shorter exact decimal representation. Display rounding, if later introduced, must not change comparison values. Do not substitute zero, empty strings or a generic default for a blocked/missing value.


### B5. Validation fixtures and implementation handoff

These are future test requirements, **not executed normalization tests**:

| Fixture | Expected result |
|---|---|
| Repeated transformation of the same source and rule version | Same derived result and provenance; no duplicate source rows or graph identities. |
| Name-key collision after whitespace/case normalization | Both original names/occurrences retained; no identity merge. |
| Strain code, salt, botanical part or trademark in a name | Qualifier preserved; no unreviewed alias substitution. |
| `1.5 g` and `1500 mg` with the same explicit compatible basis | Equal mass comparison values, originals and precision retained. |
| `100 mg per 1 tablet` versus `100 mg per 2 tablets` | Distinct bases; not equal amount assertions. |
| `0 NP`, `0 mg`, null and absent amount | Distinct source states and rule outcomes; no global zero/missing coercion. |
| `<1 mg` versus `1 mg` | Different assertion meanings; operator retained after allowed conversion. |
| `mcg DFE`, IU, enzyme units or CFU/g | No unreviewed plain-mass/count conversion. |
| Same parent amount with several forms/components | No amount copied to every child; no double counting. |
| Missing/ambiguous serving denominator or numeric/text conflict | Dependent derivation blocked with reason; source fields preserved. |
| Date-only or timezone-unknown timestamp | Original precision/unknown timezone retained, not invented midnight/UTC. |
| Cleaned display text with source support spans | Original excerpt and locator still valid against the original source snapshot. |
| Source status pending/valid/off-market | Correct source-specific meaning; no automatic clinical or identity approval. |

**Step 5B outcome:** normalization policy and audited examples are documented. It remains a draft pending review, implementation and fixtures. Raw data and earlier reports are unchanged; no normalized outputs have been created by this addition.

**Next: Step 5C — deduplication rules.** Define when records/assertions can share a comparison group or display representative, when exact duplicate ingestion can be suppressed, and when source occurrences must remain separate. Only after that design review should a small implementation pilot run, validate results and expand to the datasets. G5 end-to-end validation remains pending; G1 unresolved mappings and G2/G3/G4 implementation work are not silently marked complete.


## Step 5C — Deduplication rules (design specification only)

**Version:** `deduplication-draft-v1`. These rules define which repeated representations can be reused and which assertions can be grouped for a stated purpose. No deduplication, source deletion, canonical merge, representative selection or graph import is performed here. Apply Step 5A identity/provenance rules and Step 5B normalization prerequisites first.

### C1. Four different operations

| Operation | What can be shared or suppressed later | What remains distinct |
|---|---|---|
| Idempotent ingestion | Reprocessing the same declared input/run and identity key must not append a second copy of the same logical record, occurrence or assertion. | A genuinely new capture keeps its capture event; different source IDs or locators are not duplicates merely because content matches. |
| Content reuse | Identical bytes/chunk content may share storage or a reusable embedding under matching model/input configuration. | Every source snapshot, occurrence, passage locator, context and citation binding remains separately attributable. |
| Reviewed assertion grouping | Assertions may join a scoped equivalence group only after required subject/context/value checks pass. | Original assertions and evidence remain accessible; grouping is not clinical validation, universal equality or permission to erase provenance. |
| Retrieval/display grouping | Equivalent or overlapping results may be organized under a representative with links to members. | Distinct variants, conflicting values, source roles and unresolved context must remain visible when relevant. Display grouping does not change KG identity. |

**Default:** uncertain duplicate candidates stay separate. Exact replay suppression can be mechanical; semantic grouping requires evidence and review at the declared scope. Never run `drop_duplicates(name)` or a whole-row equality shortcut across these source roles.

### C2. General decision table

| Rule | Condition | Permitted action | Blocking condition / safeguard |
|---|---|---|---|
| D-01 — Same ingestion identity | Same source snapshot, locator, occurrence role and extraction/rule version replayed from the same archived input | Reuse the logical assertion/occurrence; verify payload and output hashes rather than append it | Same key with different payload is a conflict, not a duplicate to discard. New extraction semantics create a new derived version. |
| D-02 — Repeated capture | Same native source ID and verified retained content in a new capture | Reuse logical content snapshot; preserve new observation under G4 | Do not count repeated captures as independent evidence or fabricate captures on rerun. |
| D-03 — Equal payload under different source IDs | Identical or nearly identical content | Flag possible relationship; optionally share content storage | Do not collapse source records, publication identities, users, conversations or products. |
| D-04 — Repeated names, codes, candidates or fingerprints | Same name key, UPC/SKU, source ingredient code, CID candidate, synonym, formula or fingerprint | Generate a comparison candidate with its reason | Not proof of common product/ingredient/assertion identity. Scope and source context still required. |
| D-05 — Equal normalized amounts | Approved same-dimension conversion yields equal numbers | Mark amount representation comparable at the explicit basis | Subject identity, formulation, operator, amount role and basis must also agree before assertion equivalence. Unit equality alone cannot merge labels. |
| D-06 — Same subject, different context | Different population, indication, preparation, serving basis, regimen, time, variant, interaction partner, claim type or evidence role | Keep distinct assertions and optionally related comparison groups | Do not average values, select one by popularity, or label context differences duplicates. |
| D-07 — Conflicting values or qualifications | Comparable subject/context but disagreement in amount, polarity, rating, warning, scope or support | Preserve a conflict set with source/version provenance | Ranking, recency or source count alone does not resolve the conflict or authorize correction. |
| D-08 — Unknown critical context | Required identity, basis, qualifier or support field is missing/ambiguous | Keep an unresolved comparison candidate | Two nulls do not prove equivalence; unknown is not a wildcard. Optional irrelevant missing fields need not block comparison when explicitly justified. |
| D-09 — Same span/text, separate assertion IDs | Repeated NatMed claim text/span or differently scoped extracted assertions | Retain occurrences; compare semantic fields and extraction lineage before any grouping | Same paragraph can support several different claims. Identical extraction replays are handled by D-01; shared text alone is insufficient. |
| D-10 — Same Reddit text, different post/comment IDs | Copied/repeated text across records or threads | Optional repeated-content display group with every member retained | Do not infer same author/event, independent experiences, direct use or medical truth. |
| D-11 — Repeated RDF statement | Same subject/predicate/object including RDF term type, datatype/language and graph scope if supplied | Represent one statement per source-snapshot graph scope with all capture/query provenance | Do not deduplicate by display label or drop language/datatype. Equal triples from different source versions retain their provenance. |
| D-12 — Overlapping retrieval chunks | Passages overlap or repeat source text | Suppress redundant display context when the answer retains necessary evidence and source spans | Do not treat overlapping chunks as corroborating sources or remove non-overlapping qualifications/negation. |
| D-13 — Accepted same product variant | Labels have reviewed common family/variant assignment and compatible comparison scope | Organize under one variant; group only the assertions that separately pass equivalence checks | Preserve each label's directions, warnings, status and dates. Same variant is not automatic equality of every label field. |
| D-14 — Dataset/rule changes | New evidence, revised mappings, normalization rules or review decisions affect a group | Revalidate affected membership; preserve prior group version and decision history | No silent carry-forward of old equivalence to changed scope. Reversible grouping, not destructive deletion. |

Pairwise similarity is not transitive proof. Do not form semantic groups by connected components of candidate edges. Every group must have a declared common scope and satisfy all required comparisons without an unresolved contradictory member.


### C3. Assertion-equivalence requirements by source role

**Comparison scope is mandatory:** a group must say whether it concerns content reuse, amount representation, a source assertion, a product variant, or retrieval display. Never reuse an amount-only group as a product identity group. An `equivalent_for_scope` outcome is a reviewed project decision about that scope—not an external source approval or universal `sameAs`.

| Assertion / record type | Required agreement before semantic grouping | Separate even when text or amounts match |
|---|---|---|
| DSLD composition assertion | Reviewed compatible product/variant subject; ingredient identity at applicable scope; nutrient/material/component role; parent context; form/preparation; normalized value/operator/unit and explicit serving basis; relevant qualifications | Different package/formulation/variant, ingredient forms, source conflicts, unresolved basis or different parent/component roles. A family match alone is insufficient. |
| DSLD label statement | Compatible label/variant applicability, statement type, meaning, target context and relevant temporal scope | Directions, precautions, warnings and marketing claims are not interchangeable. Different warning versions remain source-visible; word overlap is not semantic equivalence. |
| NatMed clinical assertion | Compatible scoped subject, claim type, indication/population, formulation/strain, polarity and qualifiers; rating meaning; dose/route/frequency/duration/basis when relevant; interaction partner/role when relevant; source-support and temporal context | Different populations, outcomes, qualifications, ratings, doses or review/support status must not be flattened into a single approved claim. Shared excerpts/citations are not sufficient. |
| Terminology mapping | Same origin occurrence/scoped identity, target, v2 relation, source/evidence version and review/rule context | Different CID/RxCUI/MeSH entities remain distinct. `denotes_concept`, `member_of_category` and `exact_material_match` cannot share one undifferentiated equivalence group. |
| SourceReference | Same source snapshot/reference locator, predicate/role, target key/local scope and resolution context | Distinct citation occurrences, parent versus thread, external versus missing-ID references, or differing source versions retain separate assertions even if the target key is shared. |
| Study/publication identity | Reviewed matching external identity or corroborated bibliographic evidence under citation rules | Identical local citation numbers, title similarity or repeated bibliography paragraphs do not establish shared publication identity. Publication identity still does not prove claim support. |
| Reddit reports | Only exact ingestion replays qualify for logical record reuse; repeated content can be grouped for display | Different native IDs remain distinct source records. No semantic merge of reported experiences is authorized by this policy. |

**Missingness versus exact replay:** D-01 can suppress an exact replay with null fields because it is the same source occurrence, not a claim that two independently incomplete assertions are equivalent. D-08 blocks semantic equivalence when unknown fields matter to the comparison. Field relevance is declared per claim/type/task; do not require an interaction partner on an ordinary dose assertion, but do not ignore it on an interaction assertion.

**Semantic text comparison:** no general paraphrase-equivalence algorithm is authorized yet. Similarity/LLM suggestions can later generate review candidates, with evidence and method retained; they cannot delete assertions or replace polarity/context checks. A future matcher needs evaluation before broader automated decisions.

**Source support and independence:** group members retain their own evidence spans, citations, extraction lineage and review state. Repeated captures, overlapping passages and duplicate extraction are not independent corroboration. Different source names or URLs do not prove independent evidence when they repeat the same underlying study or report. Track known common provenance and otherwise leave independence unknown; do not inflate confidence from raw record counts.


### C4. Group records, representatives and audit examples

#### Proposed derived comparison record

Store group/case ID and version; grouping purpose/scope; member assertion IDs/snapshots/locators; candidate-generation method; relevant identity/mapping decisions; normalized comparison fields and rule versions; compared dimensions and results; outcome (`candidate`, `equivalent_for_scope`, `distinct`, `unresolved`, `conflicting`); rationale/evidence; reviewer/method and time when assessed; and supersession/lineage references.

Candidate groups are suggestions, not equivalence classes. IDs should persist with explicit group versions/lineage when reviewed membership changes; do not key canonical groups only by a mutable display name. No group IDs are allocated in this design step.

If a representative is later needed, store `representative_id`, purpose, selection-policy version and reason. The representative must be a real member at the relevant scope. Preserve access to all members and source-specific metadata. Do not construct a synthetic record by taking convenient fields from mutually incompatible labels or claims.

For current-view retrieval, use G4's eligible snapshot policy and query relevance; explain source date/version and uncertainty where material. Do not automatically pick the highest source ID, most frequent duplicate, latest capture of stale content, longest excerpt or highest clinical rating. A representative of one result group is not the uniquely authoritative truth. Conflicts and clinically material qualifications must remain visible. Ranking mechanics remain later retrieval work.

#### Expected treatment of audited examples

| Audited case | Deduplication outcome proposed by these rules |
|---|---|
| BPI 25731 / 43261: 1,500 mg vs 1.5 g per two capsules | Supports an amount-representation comparison after normalization validation. Different disclosed other ingredients and unresolved variant identity block product/assertion consolidation. Preserve both labels. |
| Herbal Hills 74939 / 74940: matching compared core, 120 vs 700 capsules | Separate package variant proposals. Shared text/content may be reusable; package-specific source records and assertions remain distinct. |
| Innovix 248198 / 276213: same UPC and image-difference annotation, different DHA/EPA quantities | Separate declared formulation/strength proposals; no averaging or selection of one amount as a duplicate correction. |
| 27 repeated-name groups with the same compared ingredient/serving core | Comparison candidates only. Core comparison omits defining fields; not 27 approved merges. |
| Probiotic blends 11613 / 32494 share ingredient ID 288938 but list 4 vs 8 immediate components | Retain occurrence-specific formulations; registry ID equality cannot deduplicate the blends. |
| NatMed: 13 repeated claim-text groups and 12 repeated span/type groups | Preserve claim IDs and full contexts; inspect populations/doses/qualifications before a scoped group. No clinical approval inherited from group membership. |
| Reddit: 24 repeated-text groups involving 230 records | Optional repeated-content display grouping only. Keep posts/comments and thread context; do not count copies as independent endorsement. |
| MeSH: 400 repeated triple occurrences | Future statement-set deduplication may reuse identical RDF statements within the defined graph scope while retaining query/response provenance. These are not 400 redundant concept identities. |
| Same vocabulary candidate reached by several queries | Preserve each query/occurrence mapping and its evidence. Shared target is not transitive ingredient identity or duplicate source occurrences. |

The audit counts describe observed candidate patterns, not approved equivalence decisions. No dataset-wide duplicate count is claimed by this specification.


### C5. Validation fixtures and design-review handoff

The following are required future tests, **not executed deduplication tests**:

| Fixture | Expected behavior |
|---|---|
| Reprocess same snapshot/locator/role with same extraction version | Reuse same logical occurrence; verify identical payload. |
| Same replay key with changed payload | Flag integrity/version conflict rather than dropping one record. |
| New capture with unchanged payload | Preserve capture history; no duplicate logical content or extra independent evidence. |
| Equal converted amount with different serving basis or product variant | No assertion-equivalence acceptance. |
| Two null critical qualifiers | Unresolved semantic comparison, not equality. |
| Exact same incomplete source occurrence replayed | D-01 reuse permitted without inventing its missing values. |
| Same paragraph supports different populations or interaction partners | Keep distinct clinical assertions. |
| Same text in different Reddit IDs | Keep separate records and source contexts. |
| Same RDF labels but different URI/datatype/language | Do not collapse distinct statements or entities. |
| Overlapping chunks hide a negation or qualifier if truncated | Do not suppress the required evidence context. |
| A/B and B/C are similar but A/C conflict | No transitive semantic group formation. |
| Representative changes after new evidence | Preserve group/selection history and all source members. |
| Old approval becomes inapplicable after source/rule change | Preserve historical decision; revalidate current group membership. |

**Step 5C outcome:** separate policies now exist for replay suppression, content reuse, reviewed assertion grouping and retrieval/display grouping. There is no global rule that “same name” or “same normalized value” means duplicate. Raw records and source assertions remain traceable through any future grouping.

**Next: formal Step 6 — review the combined design against representative end-to-end cases.** Trace a small set from raw fields through proposed identity, normalization, reference/version handling and deduplication outcomes. Check that the design preserves answers, qualifiers and provenance, including negative/ambiguous cases. Earlier pilots inform this review but do not replace an integrated check.

After Step 6 resolves material design conflicts, Step 7 can consolidate the schema/mapping/rule versions and define a bounded implementation pilot in a separate notebook. Do not start full preprocessing, automatic entity merging or KG construction merely because these design sections are written. G1 evidence gaps and G2–G5 implementation/validation work remain explicitly pending. `read.md` will be updated only after the schema is finalized, as requested.

No deduplication code or normalized/deduplicated output files were produced by this notebook-only addition. Raw files, earlier artifacts and existing outputs remain unchanged.


## Step 6 — Integrated design review on representative source evidence

**Scope:** trace selected source evidence through schema fields, identity semantics, normalization policy, reference/version handling and deduplication policy. This is an integrated **design review**, not an executed preprocessing/KG pipeline. The code below verifies source evidence and the preconditions for the walkthroughs; it does not implement the proposed transformations or demonstrate production readiness.

Cases are purposively selected for equivalence, ambiguity, conflict, missing context and source-role differences across DSLD, PubChem, RxNorm, MeSH, NatMed and Reddit. Earlier pilots supply additional occurrence-level evidence. The expected outcomes are assistant-authored design judgments, not approved identities or clinical claims. No model training or new collection is needed for this review.


In [1]:
from pathlib import Path
import json, hashlib, sys

integrated_root = Path.cwd()
if not (integrated_root / "scripts").is_dir():
    integrated_root = integrated_root.parent
if str(integrated_root / "scripts") not in sys.path:
    sys.path.insert(0, str(integrated_root / "scripts"))
from audit_mesh import bindings

integrated_inputs = {}
integrated_checks = []
def integrated_read(relative_path, expected_hash=None):
    b = (integrated_root / relative_path).read_bytes()
    digest = hashlib.sha256(b).hexdigest()
    if expected_hash is not None:
        assert digest == expected_hash, relative_path
    integrated_inputs[relative_path] = digest
    return b

def integrated_check(name, condition):
    assert condition, name
    integrated_checks.append(name)

label_manifest = json.loads(integrated_read("data/processed/dsld/dsld-ingredient-review-v1-0baa34b90a1fe69e/manifest.json"))
integrated_labels = {}
for lid in [25731, 43261, 74939, 74940, 1452, 248198, 276213]:
    meta = label_manifest["sources"][str(lid)]
    integrated_labels[lid] = json.loads(integrated_read(meta["file"], meta["sha256"]))
left = integrated_labels[25731]["ingredientRows"][0]["quantity"][0]
right = integrated_labels[43261]["ingredientRows"][0]["quantity"][0]
integrated_check("BPI: 1500 mg vs 1.5 Gram(s), both per two capsules", (left["quantity"], left["unit"], right["quantity"], right["unit"]) == (1500,"mg",1.5,"Gram(s)") and left["servingSizeQuantity"] == right["servingSizeQuantity"] == 2)
integrated_check("BPI other-ingredient payloads differ", integrated_labels[25731]["otheringredients"] != integrated_labels[43261]["otheringredients"])
integrated_check("Herbal Hills package counts differ 120 vs 700", [integrated_labels[l]["netContents"][0]["quantity"] for l in [74939,74940]] == [120,700])
integrated_check("Magnesium label preserves separate citrate form", integrated_labels[1452]["ingredientRows"][0]["name"] == "Magnesium" and integrated_labels[1452]["ingredientRows"][0]["forms"][0]["name"] == "Magnesium Citrate")
integrated_check("Innovix equal UPC does not imply identical ingredient payload", integrated_labels[248198]["upcSku"] == integrated_labels[276213]["upcSku"] and integrated_labels[248198]["ingredientRows"] != integrated_labels[276213]["ingredientRows"])

term_manifest = json.loads(integrated_read("data/processed/terminology/terminology-review-v1-34294732268f2352/manifest.json"))
integrated_terminology = {s:json.loads(integrated_read(m["file"],m["sha256"])) for s,m in term_manifest["inputs"].items()}
pc = integrated_terminology["pubchem"]["records"]["134732"]
integrated_check("PubChem retains explicit HCl/hydrochloride aliases", {"creatine hcl","creatine hydrochloride"}.issubset({s.lower() for s in pc["synonyms"]["Synonym"]}))
rx = integrated_terminology["rxnorm"]["search_results"]["creatine hcl"]
integrated_check("RxNorm HCl lookup retains competing parent/monohydrate targets", set(rx["rxcuis"]) == {"2907","1310467"})
integrated_mesh_hierarchy = [b for b in bindings(integrated_terminology["mesh"]["record_subgraph_responses"]) if b["subject"]["value"].endswith("/D017962") and b["predicate"]["value"].endswith("#broaderDescriptor") and b["object"]["value"].endswith("/D015525")]
integrated_check("Saved MeSH ALA-to-omega-3 hierarchy exists", bool(integrated_mesh_hierarchy))

for source, directory in [("natmedpro","natmed-baseline-audit-v1-b548c344b8c0a9b7"),("reddit","reddit-baseline-audit-v1-058aea355a5bb848")]:
    m = json.loads(integrated_read(f"data/processed/{source}/{directory}/manifest.json"))
    files = [f"data/raw/{source}/records.jsonl"]
    if source == "natmedpro":
        files.append("data/raw/natmedpro/upstream_derived/pending_clinical/clinical_claims.jsonl")
    for f in files:
        data = [json.loads(line) for line in integrated_read(f,m["inputs"][f]).decode().splitlines() if line.strip()]
        if source == "reddit": integrated_reddit = data
        elif f.endswith("clinical_claims.jsonl"): integrated_claims = data
        else: integrated_monographs = {r["id"]:r for r in data}

claim_id = "claim:3a1e7c59e8144dd11cfe90e000be0afa605130adb7d5baf9a94e590a529ea45f"
integrated_claim = next(c for c in integrated_claims if c["id"] == claim_id)
cs = integrated_claim["source"]
section = integrated_monographs[cs["record_id"]]["sections"][cs["section_index"]]
integrated_check("NatMed selected excerpt offsets resolve to original section", section["text"][cs["start"]:cs["end"]] == cs["excerpt"])
integrated_check("NatMed mixed-route excerpt retains null dose and pending review", "orally" in cs["excerpt"] and "topically" in cs["excerpt"] and integrated_claim["dose"]["value"] is None and integrated_claim["review"]["state"] == "pending")
anchor = integrated_claim["field_support"]["safety_rating"]
integrated_check("NatMed safety-rating support span remains traceable", cs["excerpt"][anchor["start"]:anchor["end"]] == integrated_claim["safety_rating"])

reddit_by_id = {r["id"]:r for r in integrated_reddit}
native_ids = {r["native_id"] for r in integrated_reddit}
integrated_reddit_topic = reddit_by_id["reddit:t1_19rg1ztatzi8z"]
integrated_reddit_missing = reddit_by_id["reddit:t1_19rl50ku815cw"]
integrated_reddit_external = reddit_by_id["reddit:t1_19rgc7uzlnucg"]
integrated_check("Reddit inherited topic remains a user report", integrated_reddit_topic["mention_scope"] == "thread_context" and integrated_reddit_topic["evidence_role"] == "user_report")
integrated_check("Missing parent identifier remains null", integrated_reddit_missing["parent_id"] is None)
integrated_check("Known external parent remains distinguishable from missing ID", bool(integrated_reddit_external["parent_id"]) and integrated_reddit_external["parent_id"] not in native_ids)

print("Evidence/precondition checks passed:",len(integrated_checks))
for name in integrated_checks: print("-",name)
print("\nSelected NatMed claim:",claim_id)
print("Source section:",cs["record_id"],cs["section_index"],"offsets",cs["start"],cs["end"])
print("Clinical state:",integrated_claim["review"]["state"],"structured dose:",integrated_claim["dose"])
print("\nReddit inherited-topic example:",integrated_reddit_topic["id"])
print("Text:",integrated_reddit_topic["text"])
print("Mention scope:",integrated_reddit_topic["mention_scope"],"topics:",[s["supplement"] for s in integrated_reddit_topic["supplements"]])
print("\nPinned input hashes:",len(integrated_inputs),"(available in integrated_inputs)")
print("These checks verify saved evidence, not an implemented normalization/deduplication pipeline.")


Evidence/precondition checks passed: 14
- BPI: 1500 mg vs 1.5 Gram(s), both per two capsules
- BPI other-ingredient payloads differ
- Herbal Hills package counts differ 120 vs 700
- Magnesium label preserves separate citrate form
- Innovix equal UPC does not imply identical ingredient payload
- PubChem retains explicit HCl/hydrochloride aliases
- RxNorm HCl lookup retains competing parent/monohydrate targets
- Saved MeSH ALA-to-omega-3 hierarchy exists
- NatMed selected excerpt offsets resolve to original section
- NatMed mixed-route excerpt retains null dose and pending review
- NatMed safety-rating support span remains traceable
- Reddit inherited topic remains a user report
- Missing parent identifier remains null
- Known external parent remains distinguishable from missing ID

Selected NatMed claim: claim:3a1e7c59e8144dd11cfe90e000be0afa605130adb7d5baf9a94e590a529ea45f
Source section: natmed:0d296c0daa1cccb456da568c 4 offsets 0 464
Clinical state: pending structured dose: {'duratio

### 6A. End-to-end walkthrough and expected output boundaries

Each row follows **source → schema/identity → normalization → reference/version → deduplication/retrieval**. Expected outputs are specifications; no transformed records or graph edges were emitted.

| Case and user-facing question | Trace through the design | Outcome / answer boundary |
|---|---|---|
| E2E-01 — BPI 25731/43261: “Are these amounts the same?” | Two SourceRecords/snapshots → separate label CompositionAssertions with parent/serving role → N-U02 would compare 1.5 g as 1,500 mg on the same two-capsule basis → retain each source version and family/variant review → D-05 supports amount-only comparison, not variant equality. | The stated mass amounts are equivalent representations. Other ingredients differ; no duplicate-label removal or same-variant conclusion. No amount copied onto both HCl and monohydrate forms. |
| E2E-02 — Herbal Hills 74939/74940: “Which package is this?” | Separate labels → proposed common ProductIdentity, separate package ProductVariants under PV-07 → preserve 120/700 counts and full unit descriptors → source relationships remain attributed evidence → D-13 cannot flatten distinct package variants. | Two package offerings remain distinguishable despite matching compared ingredient/serving core. No invented merged package or ingredient total. |
| E2E-03 — Innovix 248198/276213: “How much EPA/DHA?” | Source-specific quantity assertions and explicit one-capsule bases → candidate family plus distinct formulation proposals → preserve differing quantities and context → image-difference/UPC evidence cannot override them → retain distinct assertions, not an averaged or newest-ID-selected value. | Answer must identify the label/version or ask which offering is intended. If unresolved, present the alternatives with provenance; do not state one universal family dose. |
| E2E-04 — Magnesium declaration and citrate form on 1452 | Parent nutrient declaration and child IngredientOccurrence → generic concept proposal separate from exact material proposal → normalize only supported quantity/unit representation → evidence stays at the parent/form paths → no dedup between nutrient and supplying material. | A nutrient amount does not become citrate mass or proof of elemental material. The target's extra stoichiometric detail remains unresolved. |
| E2E-05 — Creatine HCl candidate disagreement | HCl occurrence → separate MappingAssertions to saved PubChem/RxNorm candidates → alias evidence is separate from N-T02 search formatting → decisions depend on exact target snapshots/properties → D-04 prevents shared-query/target collapse. | PubChem hydrochloride proposal has scope support; RxNorm parent/monohydrate results cannot identify HCl. All approvals remain pending; cross-source candidates are not three votes for one identity. |
| E2E-06 — ALA membership in omega-3 | Explicit ALA occurrence from the broader pilot → denotes-concept proposal and independent member-of-category proposal → no qualifier stripping → MeSH D017962→D015525 source hierarchy retained separately as evidence → category and member remain distinct. | Category navigation can retrieve ALA evidence while retaining its scope; it cannot imply all omega-3 components occur in that product or transfer every category claim. |
| E2E-07 — NatMed mixed-route safety excerpt | Claim ID → pending ClinicalAssertion with original excerpt/field-support spans and monograph reference → N-T04 retains null structured dose/route; no extraction of one convenient number → version-bound section offsets and locally scoped citations remain independent → D-09 does not equate paragraph identity with a single uniform claim. | Preserve the source safety wording with its contexts; no single dose statement or clinically approved recommendation is created. Separate oral/topical assertions would require an additional validated extraction step. |
| E2E-08 — Reddit inherited topics | Native comment → Reddit SourceRecord plus TopicAssociation with `thread_context`, `user_report` → preserve original text/unknown score/date fields → resolve thread/parent separately → D-10/12 may reduce repeated display without changing source role. | The sampled comment about CBN must not become a direct magnesium/omega-3/creatine experience. Topic relevance is not direct mention, endorsement or clinical evidence. |
| E2E-09 — Reddit missing versus external parent | Null parent on `reddit:t1_19rl50ku815cw` versus known absent parent on `reddit:t1_19rgc7uzlnucg` → distinct SourceReference states → no ID guessing or null coercion → G3 permits later resolution only when a key/target is supported → missing references never merge through null target equality. | Both comments stay retrievable with their context gaps; no invented parent record. Known external key can guide later collection; null ID cannot. |
| E2E-10 — Future replay/change | A future capture/rerun, explicitly hypothetical → G4 distinguishes source identity, content and capture → new normalization rules produce derived builds, not rewritten raw records → reference/identity applicability rechecked only where dependencies change → D-01/02/14 prevent duplicate ingestion and silent reuse of stale decisions. | Design scenario only; no second capture or lifecycle transition was executed. No automatic model retraining follows from adding data. |

**Source-support trace:** the NatMed claim inspected above has an exact monograph section match and a valid saved safety-rating span. That verifies extraction location, not clinical truth. Similarly, source hash verification ensures consistency with the pinned local collection, not that the source itself is correct or up to date.


### 6B. Integrated findings and implementation blockers

| Finding | Design disposition | Implementation action / gate |
|---|---|---|
| A5 “all alternatives reviewed” can be overread as an exhaustive search of every source | Interpret all pilot recommendations as evaluation of the **saved candidate set**, not proof that no uncollected alternative exists. | Record candidate-set scope/version and reviewed versus unreviewed alternatives. A6 approval remains pending; do not assign an accuracy/coverage rate from purposive pilots. |
| One clinical excerpt can contain several routes or contexts while structured fields are null | Current schema can retain the original pending assertion and excerpt. Normalization must not invent missing structured claims. | Preserve nulls; any future claim splitting requires separate extraction rules, child provenance and validation before use. It is not part of unit normalization. |
| Generic concept/category/material meanings can be confused in graph exports | v2 MappingAssertion relations resolve the conceptual ambiguity; older `identity_match` is historical. | Export must use explicit relation/scope versions; block automatic legacy-to-exact-material conversion and unqualified `sameAs`. |
| “Same product” annotations and identical UPCs can disagree with composition/package | Product boundary rules keep source assertions separate from local family/variant review. | Store per-dimension comparison and unresolved state. Do not treat a source relationship label as an accepted family/variant assignment. |
| Replay identity, source identity and assertion equivalence require different keys | G4 and D-01–14 define the distinction, but no physical key serializer/importer has been implemented. | Test serialization, collision detection, update behavior and deterministic replay before loading graph data. |
| Positional evidence can break after text cleanup, rechunking or file reordering | Raw snapshots/locators stay immutable; derived text/chunks carry separate provenance. | Validate paths/offsets before reusing an assertion; never bind old offsets to normalized text. |
| Missing references cannot become ordinary fetched domain nodes | G3 SourceReference can preserve known and unknown targets independently of graph endpoints. | Implement unresolved-reference storage before emitting resolved parent/citation/product edges. |
| Rule versions and pending decisions must be visible across stages | Design permits stage-specific rules and review states; no single global approval flag is sufficient. | A pilot manifest must pin schema, mapping, normalization, reference, versioning and deduplication rules and report stage-specific blockers. |

**Review conclusion:** the selected cases can be represented without collapsing ingredient forms, package variants, clinical contexts or evidence roles. No additional entity merge is justified. The review demonstrates design consistency for these cases and verifies their saved evidence; it does not validate a working pipeline, every field mapping, all 400 labels or every clinical/terminology candidate.

**G5 remains open:** normalization/deduplication functions, physical serialization, graph constraints, lifecycle behavior and retrieval behavior still need implementation tests. Full mixture identity, unresolved aliases/definitions and uncollected targets remain explicit limitations from G1/G3. A design check passing is not an accepted mapping or clinical approval.


### 6C. Step 6 outcome and handoff to Step 7

**Completed here:** integrated walkthroughs for nine evidence-backed cases plus one hypothetical lifecycle scenario; executed evidence/precondition checks spanning all six sources; documented interpretation boundaries and implementation blockers. Existing source data, review decisions and earlier artifacts remain unchanged.

**Next — Step 7:** consolidate the current draft into one implementation contract: definitive proposed entities/relations, selected source-field mappings, current rule versions, supported transformations, output table schemas, manifests and acceptance fixtures. Resolve superseded conventions explicitly rather than asking implementation to infer the newest rule from notebook history. Keep unapproved identities and clinical claims pending.

Then prepare a small, separate preprocessing implementation notebook with raw-preservation and provenance checks. Start with bounded representation changes and the audited positive/negative fixtures; inspect outputs before expanding. Do not treat this Step 6 completion as permission for automatic semantic merges or full graph construction. `read.md` remains unchanged until the design is finalized.


## Step 7 — Consolidated implementation contract and bounded pilot handoff

**Contract:** `supplementrx-poc-preprocessing` **0.1.0**, saved at [`../schemas/poc_preprocessing_contract_v0.1.json`](../schemas/poc_preprocessing_contract_v0.1.json). This is the current implementation baseline for a bounded pilot, not a final production schema or a declaration that unresolved mappings are approved. The JSON is a table/pipeline specification, **not a JSON Schema validator**. Implementation must enforce both its structural and semantic gates.

This section consolidates the final conventions from Steps 3–6. Use it and the linked contract for implementation rather than guessing precedence from earlier notebook history. Earlier source audits and evidence reviews remain provenance; no review result or raw record is rewritten. If the contract changes, create a new version and record the reason rather than silently modifying a build's meaning.

### 7A. Definitive choices for contract 0.1.0

| Area | Contract choice | Superseded or excluded behavior |
|---|---|---|
| Source versus shared identity | Native source IDs plus captured versions; source occurrences/assertions are usable before shared identity is resolved | No canonical ingredient/product identity from names, candidate counts or shared codes |
| Mapping semantics | `candidate_unspecified`, `denotes_concept`, `member_of_category`, `exact_material_match`, directional concept inclusion, or supported `related_entity` | Legacy `identity_match` cannot appear in new pilot outputs; no automatic migration or sameAs |
| Product representation | Family = evidenced commercial line; variant includes disclosed formulation/presentation and package | No family-wide pooled dose or automatic repeated-label merge; family/variant assignments remain empty in pilot |
| Reference handling | SourceReference keeps missing, ambiguous, uncollected and collected/incomplete states distinct | No invented target record or dangling resolved-domain edge |
| Processed format | UTF-8 JSONL tables with explicit nulls, nested selected fields and source evidence; derived decimals as strings | No lossy CSV flattening or graph import yet |
| Physical roles | ChemicalCompound/TerminologyConcept are source-record roles with versioned values, not duplicate copies of the same payload | Canonical identities remain separate and optional; no duplicate node merely to satisfy two conceptual labels |
| Normalization | Original fields remain immutable; derived values are separate transformation records with rule/evidence/precision/basis | No automatic alias identity, dose inference, clinical claim splitting or correction of contradictions |
| Deduplication | Exact archived-input replay handling only during first pilot | No semantic merging, representative selection or raw-file deletion |
| Review state | Clinical, identity, reference, transformation and validation status remain separate | No global approved flag; existing NatMed claims stay pending and Reddit stays user_report |
| Graph/GNN work | Deferred until validated processed data and reviewed export rules exist | No Neo4j import, vector generation or model training in this handoff |

**Snapshot implementation decision:** the pilot uses a conservative `raw-bytes-v1` **container-bound** snapshot key: source-record identity + representation/profile + containing artifact digest + exact record locator. Its `payload_scope` explicitly says `container_bound_record`; the digest never masquerades as a per-record hash. A JSONL/container change may therefore version all contained records. This is an intentional conservative cost, avoiding an unvalidated lossless serializer. The proposed `json-lossless-tree-v1` profile is deferred until its fixtures pass; later adoption requires a new profile/migration. This refines G4's implementation choice without changing existing raw hashes or old IDs.

**Scope of schema freeze:** table/field/type conventions, relation names and pilot boundaries are fixed for contract 0.1.0. Semantic identity decisions, generic-concept canonicalization, complete mixture acceptance, later graph layouts and unimplemented validators are not declared final. `read.md` remains unchanged until the broader schema is finalized, as requested.


In [1]:
from pathlib import Path
import json, hashlib
import pandas as pd

contract_root = Path.cwd()
if not (contract_root / "schemas").is_dir():
    contract_root = contract_root.parent
contract_file = contract_root / "schemas/poc_preprocessing_contract_v0.1.json"
contract_bytes = contract_file.read_bytes()
poc_contract = json.loads(contract_bytes)
assert poc_contract["version"] == "0.1.0"
assert "identity_match" not in poc_contract["enums"]["mapping_relation"]
contract_field_types = {"string","integer","boolean","object","json","string?","object?","string[]","object[]"}
for table_name, table in poc_contract["tables"].items():
    assert table["primary_key"] in table["fields"]
    assert set(table["fields"].values()) <= contract_field_types
    assert table["file"] == table_name + ".jsonl"
    for field, target_table in table["foreign_keys"].items():
        assert field in table["fields"] and target_table in poc_contract["tables"]
assert len({t["file"] for t in poc_contract["tables"].values()}) == len(poc_contract["tables"])
print("Contract:",poc_contract["contract_id"],poc_contract["version"])
print("SHA-256:",hashlib.sha256(contract_bytes).hexdigest())
print("\nLogical output tables:")
print(pd.DataFrame([{"table":name,"columns":len(t["fields"]),"pilot":t["pilot"]} for name,t in poc_contract["tables"].items()]).to_string(index=False))
print("\nPinned rule versions:")
print(pd.Series(poc_contract["rule_versions"]).to_string())
print("\nValidation gates:")
print(pd.DataFrame(poc_contract["validation_gates"])[["id","gate","failure_action"]].to_string(index=False))
print("\nContract consistency checks passed. No data rows, FKs or transformation functions were validated by these specification-only checks.")


Contract: supplementrx-poc-preprocessing 0.1.0
SHA-256: 46b3798bca370e601b55cee9a1db389fc6799006d150e7f8647337c16dd5c3e5

Logical output tables:
                table  columns                            pilot
            artifacts        6                    as_applicable
       source_records        7                    as_applicable
            snapshots       11                    as_applicable
       capture_events        9                    as_applicable
             evidence        9                    as_applicable
          occurrences       10                    as_applicable
             servings        7                    as_applicable
           assertions       13                    as_applicable
          term_values        6                    as_applicable
vocabulary_statements       10                    as_applicable
              lookups       12                    as_applicable
       lookup_origins        5                    as_applicable
   mapping_assertions  

### 7B. Output layout, selected fields and constraints

The machine-readable contract lists **21 logical tables**, their exact required top-level columns, permitted field types, primary/foreign keys and pilot status. They are not 21 new graph node types. Some are join/provenance tables; canonical identities, semantic groups and approval events stay empty, and passages are deferred until window validation. Empty/deferred status must be explicit in the manifest; a missing file is not silently treated as an empty table.

Outputs go under `data/processed/poc/<build_id>/`, with `manifest.json`, `validation_report.json`, `review_queue.jsonl` and applicable JSONL tables. A build ID binds contract/input/code/configuration/rule/dependency versions. Output hashes, row counts, unresolved counts with their counting unit, and raw-preservation checks are mandatory. Execution timestamps/logs are separate from deterministic payloads so a rerun does not manufacture a different data build.

| Table family | Consolidated projection |
|---|---|
| artifacts / source_records / snapshots / capture_events | Exact saved file identity, source-native records, container-bound versions, original roles/dates and source/collection metadata. New processing is not a new source capture. |
| evidence | Typed locator, snapshot/artifact, original value/presence and support role. Preserve JSON pointers, character spans, word windows and RDF statements as distinct coordinate systems. |
| occurrences / servings / assertions | DSLD recursive rows/forms/other ingredients and every quantity entry; source serving definitions/bases; selected label statements; NatMed native claim IDs with complete selected clinical context and pending review. |
| term_values / vocabulary_statements | PubChem properties/aliases; selected RxNorm names/codes/source metadata; MeSH actual subject properties and typed predicates. Preserve external URI references without inventing target records. |
| lookups / lookup_origins / mapping_assertions | Full selected lookup outcomes/methods and all candidates tied to exact DSLD occurrences. No-match keeps a lookup without target mappings. Candidates remain unreviewed/needs_review, not accepted v2 relations. |
| source_references / topic_associations | Separate DSLD/RDF/citation/Reddit reference roles; preserve local citation scope. Topic context keeps source record-level limitations and cannot create a clinical assertion. |
| transformations / review_queue | Derived normalization values and outcomes with parameters, dependencies, source evidence and explicit blockers. Do not overwrite assertion.original_fields. |
| passages / review_events / canonical_identities / comparison_groups | Defined extension tables. Passages need source-window validation first; the other three contain no approval/identity/semantic-group rows in the pilot. |

`source_mapping` in the contract consolidates the selected-field projection for all six sources; Step 4 retains detailed native path examples. Object fields such as `original_fields` preserve the selected original hierarchy/types and have kind-specific obligations: composition retains amount/operator/unit/basis/role, clinical retains ingredient/population/indication/ratings/dose/interaction/source support. Their detailed executable JSON validators must be implemented before output generation; an unrestricted object is not permission to omit required source context.

**Additional referential checks:** mapped target snapshots must belong to mapped target records; clinical source bindings must resolve to the correct monograph version; servings and ingredient parents must belong to the appropriate label snapshot; polymorphic owner/member references must resolve to emitted IDs. Capture events can refer to multiple snapshots. Claims with several field-support spans retain all spans. Unresolved external keys remain values on SourceReference rather than nonexistent foreign keys.

**Source support caveat:** current NatMed annotations have no dedicated structured claim-citation list. Do not create StudyReference links by copying all bibliography entries; citation extraction/resolution is deferred. Imported source references may be retained without inventing claim-to-study support. RDF properties remain typed and source-scoped; terminology metadata never becomes clinical evidence.


### 7C. First implementation pilot — scope and order

**Planned notebook:** `notebooks/Preprocessing_Pilot.ipynb` (not created or executed by this step).

**Bounded inputs:** DSLD labels 1452, 25731, 43261, 74939, 74940, 257287 and 249674; the explicitly pinned NatMed mixed-route safety claim and its monograph; the three Reddit examples from Step 6; saved terminology candidates/lookup provenance associated with selected occurrences, plus the saved MeSH ALA/omega-3 hierarchy fixture. Declare all selected IDs/locators and file hashes in the pilot input manifest. The examples exercise ordinary amounts, equivalent units, different packages, a numeric/text conflict, NP entries, strain context, inherited topics and missing/uncollected references. No full-dataset run or network collection is part of this scope.

**Implementation order:**

1. Implement contract row validators, key serialization and fixtures; verify selected raw artifacts and locators. Stop on integrity/type/key contradictions.
2. Project source-scoped provenance, occurrences, servings, assertions, selected vocabulary evidence, lookups, topic associations and references. Preserve every selected multiplicity and unknown. Canonical identities remain absent.
3. Apply only the contract's pilot allowlist: display trim, separate NFC/whitespace/casefold name-search key, source-contract ID representation, explicit plain-mass aliases/exact decimal comparison, NP/zero distinction, operator/bound retention and explicit serving-basis preservation.
4. Record unsupported aliases, activity/qualified units, conflicts and missing context as blockers; preserve raw values. Do not perform per-unit/daily-dose derivation, CFU conversions, claim splitting, clinical approval or semantic deduplication. Dates are copied with roles, not automatically converted in this first pilot.
5. Verify replay idempotence, row counts, provenance and all V1–V7 gates. Known source uncertainty can remain in the review queue; broken extraction, invented values, unresolved required FKs or unintended approvals fail the build.
6. Display positive, negative and ambiguous outputs for V8 inspection before expanding. A completed pilot is not authorization to auto-approve matches or merge the remaining 400 labels.

**Acceptance fixtures:** unit equivalence without product merging; distinct serving bases; 0/NP versus 0/mg versus absent/null; inequality and range preservation; parent/form amount separation; numeric/text conflict; pending multi-context NatMed claim; inherited Reddit topic; null parent versus known external parent; RDF datatype/language/URI distinctions; identical rerun versus genuinely new capture. Where a selected source does not contain a needed fixture, use a clearly labeled synthetic test input—never present it as a collected record.

Evidence/semantic gates are independent: a schema-valid clinical claim can remain pending, a structurally valid mapping can remain unresolved, and a known uncollected reference is not an extraction failure. Report these outcomes separately rather than chasing a misleading zero-unknowns target.


### 7D. Handoff outcome and unresolved decisions

**Completed:** a consolidated versioned contract with source roles/projections, relation and status enums, output fields/types/keys, build provenance, bounded transformation allowlist, eight validation gates and an explicit pilot scope. The contract consistency cell ran successfully; it checks the specification's field types and foreign-key table names, not an implemented dataset pipeline.

**Still pending:** executable nested-field validators, normalization functions, snapshot/key fixtures, reference/lifecycle implementation and semantic review of unresolved identities. No complete mixture-identity validation, full 400-label grouping, graph-export design or retrieval/GNN evaluation is claimed. These limitations do not block faithful source-scoped extraction but must block unsupported acceptance/merging.

The design is ready to guide a small implementation pilot, not a final production deployment. Updating `read.md`, preprocessing full collections, creating a graph, computing embeddings, training a model and pushing to GitHub are not performed by this handoff. Raw files and previous audit/review artifacts remain unchanged.

**Next action:** create the separate pilot notebook and implement the bounded workflow against this contract, starting with its validators and fixtures. Review its actual outputs before widening scope or finalizing the broader schema documentation.
